# Repairing the iteration-8 research record (iteration 9, Step 0)

**Artifact:** `eval.py` of `art_iYh9zVP7myjJ`, an evaluation that runs no model (CPU only, USD 0).

**The research.** This AI Inventor run asks how "creativity" shows up inside open LLMs. Here creativity means which
algorithm a code model chooses to write: asked to "implement a path search", does it always write BFS? The run also
asks whether prompts, temperature or activation edits can move that choice. Iteration 8 ran six experiments on
Qwen3-8B, Qwen3-32B and gemma-3-4b-it:
- temperature dosed at the token where the algorithm is decided (the *fork*);
- DAS subspace edits at the fork;
- a circuit for the fork;
- a Gemma "dial";
- a temperature law;
- a top-five prohibition, whose survivors were placed on a manifold of prompt states.

**This artifact.** A blocking review found seven problems in the iteration-8 write-up (items 0a–0g). Step 0 repairs the
record and changes no experiment:
- **A value ledger.** Every number that a correction quotes is recomputed from iteration 8's result files. Each value
  carries the number the report printed, the recomputed number, a status (MATCH, CORRECTED, NEW or CARRIED) and a
  source `<artifact_id>:<path>#<key>`. The full run has 298 values: 147 MATCH, 151 NEW, 0 CORRECTED.
- **49 report edits.** Each maps a verbatim `old_text`, found exactly once in the report, to a `new_text` tagged
  *(iteration-9 correction)*. An **apply test** and a **whole-report lint** for the residual claims check them; the lint
  finds 28 live hits before the edits and 0 after.
- **Item 0e, the one new computation.** Under the top-five ban, the model's surviving programs landed on the side of the
  manifold *opposite* the banned algorithms. That manifold is built from 300 forced prompts, each naming its algorithm
  by one of three aliases. Is the geometry following the *names*? The test correlates each alias's geodesic distance
  to the banned algorithms with its lexical overlap with their names. It uses exact permutation nulls, a template
  bootstrap, a within-algorithm test that holds the algorithm fixed, placebos and a planted control. The committed
  verdict is **CONFOUNDED**: across algorithms names and mechanisms move together, and within an algorithm the name
  effect is not detected.
- **Two audits.** Standard-library scripts re-derive the numbers by separate code paths.

**What this notebook runs.** The whole `eval.py` pipeline, on real inputs shipped in `mini_demo_data.json`:
- the report under repair (1.4 MB, byte-identical, sha256-checked);
- the iteration-8 result files, pruned to the parts the code reads;
- the artifact's own pre-commitment and its live web checks.

The 7 `src/` modules and the 2 audit scripts are copied unchanged into `%%writefile` cells. `eval.py` is split into
cells at its function boundaries, with 6 small changes, each marked `# demo:`.

Two kinds of input are not shipped:
- the 181 MB states file, so gate **E4** (an independent recomputation of the geodesics) is skipped, which `eval.py`
  already does for any copy without that file;
- the images of 9 iteration-8 figures (18 png and pdf files), which the run only hashes for figure captions.

On a 2-core CPU the whole notebook ran in about 45 s, package installs included, and `main()` itself takes about 30 s.
On Colab only `loguru` is installed, and the 11 MB tokenizer is downloaded once.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# loguru: NOT pre-installed on Colab, always install (the version eval.py pinned)
_pip('loguru==0.7.3')

# numpy, scipy, matplotlib, tokenizers, huggingface_hub: pre-installed on Colab; install locally only, at Colab's versions
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'scipy==1.16.3', 'matplotlib==3.10.0', 'tokenizers==0.22.2', 'huggingface_hub==1.4.0')


In [ ]:
#!/usr/bin/env python3
"""Iteration-9 Step 0: repair of the iteration-8 research record (items 0a-0g of the blocking review).

uv run eval.py --stages all
  setup    vendor-check the report, hash every input (inputs_manifest.json)
  0e       alias-overlap geometry on iteration 8's stored geodesics (needs predictions/commit_s0e.json first)
  ledger   recompute every quoted number (0a circuit, 0b DAS, 0c entropy/law, 0f Gemma arms)
  edits    positioning (0d), figure markers (0g), 0e text, build edits, apply test, whole-report lint, outputs
  eval_out eval_out.json (exp_eval_sol_out) + inputs-unchanged assert
"""
from __future__ import annotations

import argparse
import csv
import datetime as dt
import hashlib
import json
import os
import resource
import sys
import time
from collections import Counter
from pathlib import Path
from typing import Any

from loguru import logger

# --- not in eval.py: for writing the shipped inputs back to files, and for the summary at the end ---
import base64
import difflib
import re

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display


In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/ai-inventor-papers/ai-invention-295f8e-prompt-bans-substitute-not-just-subtract/fork/run_jSassd-YWLVC/round-9/evaluation-8/demo/mini_demo_data.json"
import http.client
import json
import urllib.request
from pathlib import Path

def load_data():
    try:
        with urllib.request.urlopen(GITHUB_DATA_URL, timeout=15) as response:
            return json.loads(response.read().decode())
    # Network errors, timeouts, a placeholder URL, bad JSON, a cut-off response.
    except (OSError, ValueError, http.client.HTTPException) as e:
        print("GitHub data unavailable, using local mini_demo_data.json:", e)
    local = Path("mini_demo_data.json")
    if local.exists(): return json.loads(local.read_text())
    raise FileNotFoundError("Could not load mini_demo_data.json")


In [ ]:
data = load_data()
print(data["metadata"]["description"])
EXAMPLES = data["datasets"][0]["examples"]
print(f'dataset {data["datasets"][0]["dataset"]!r}: {len(EXAMPLES)} input files')


### What is in `mini_demo_data.json`

There is one dataset, `step0_inputs`, with one example per input file that `eval.py`, its modules or its audits read.
Each example has these fields:
- `path`: where the file sits in the layout of the published repository, which has one folder per artifact;
- `source`: the original `<artifact_id>:<path>`;
- `format`: `text`, `json`, `jsonl` or `npz`;
- `role`: what the file is for;
- `pruned`: what was kept;
- `sha256_original`: the hash of the original file;
- `content`: the file itself.

- **`art_iYh9zVP7myjJ/`, this artifact.** It holds the report under repair, the vendored prompt index of iteration 8's
  states file, the item-0e pre-commitment (`predictions/commit_s0e.json`, `commitments.log`) and the raw outputs of the
  live neighbour checks (`audit/web/*.txt`). All of them are byte-identical copies.
- **The six iteration-8 artifacts.** These are the circuit (`art_18XvfcJJZi1O`), DAS (`art_e-_q5mSJGrSb`), fork-dosing
  (`art_vnyi0Fm5SyNH`), Gemma dial (`art_FJMIE9T9_PMv`), temperature-law (`art_lzuosbrb34wB`) and top-five prohibition
  (`art_R-HSmXq2t97f`) artifacts. Their result files, raw rows and the source lines the corrections quote are pruned
  to the keys, rows and fields the code reads. The largest piece is the prohibition artifact's stored kNN-geodesic
  matrix between its 300 forced prompts at block 31 of Qwen3-32B. It is stored as base64 little-endian float64, so it
  rebuilds bit for bit.

`metadata.full_run` holds the original run's numbers (metrics, the 298-value ledger, hashes of the 49 edits, the audits),
and the comparison at the end of the notebook uses them.

In [ ]:
for e in EXAMPLES:
    print(f"{e['format']:5s} {len(json.dumps(e['content'])) / 1e3:8.1f} kB  {e['path']}")
print("\nexample:", json.dumps({k: v for k, v in EXAMPLES[-1].items() if k != "content"}, indent=1))


### Configuration

- `STAGES` replaces `eval.py`'s `--stages` argument. `"all"` (the default) runs every stage; `"0e"` stops after item 0e.
- `N_BOOT`, `N_PLACEBO` and `N_MC_SECONDARY` are item 0e's resampling sizes. They are constants in
  `src/alias_geometry.py`, and the `eval.py` header cell overrides them:
  - `N_BOOT`: template-bootstrap resamples for the 95% intervals;
  - `N_PLACEBO`: draws per placebo;
  - `N_MC_SECONDARY`: Monte Carlo draws for the content-word null. It is sampled because it has 5.4 million distinct
    arrangements; the other nulls are exact.

  The original values (2000, 1000 and 200,000) reproduce the full run's numbers. Smaller values change only the
  intervals, the placebo results and that one Monte Carlo p-value, so they also change the report sentences that quote
  them. The exact statistics and the verdict do not change.
- `FETCH_TOKENIZER` downloads the Qwen3-32B tokenizer (one 11 MB file) for the optional BPE-overlap sensitivity. With
  `False`, that sensitivity is skipped, as the pre-commitment allows.

In [ ]:
# ---- tunable parameters -------------------------------------------------------------------------------------------
STAGES = "all"            # eval.py --stages: "all" (original default) or "0e"
N_BOOT = 2000            # template-bootstrap resamples per 95% interval (original: 2000)
N_PLACEBO = 1000         # draws per placebo P1-P3 (original: 1000)
N_MC_SECONDARY = 200000  # Monte Carlo draws for the content-word null (original: 200_000)
FETCH_TOKENIZER = True    # fetch the Qwen3-32B tokenizer for the optional BPE sensitivity (original: read from the HF cache)


### Write the shipped inputs back to files

`eval.py` reads files, not a data object. This cell writes every example to its path under `eval_demo_run/`. That
recreates the layout of the published repository: this artifact's folder (`eval.py`'s `ROOT`), with the six iteration-8
artifacts as sibling folders. `S0_DEPS_DIR` is the original code's single setting for where those six live, and it
points at that folder.

`text` files are written byte for byte, `json` and `jsonl` files are written from their pruned content, and the `.npz`
of stored geodesics is rebuilt from its arrays. Its k = 10 matrix holds only the non-banned × banned block, which is the
only part any code reads at k = 10; every other entry is NaN.

In [ ]:
DEMO = Path("eval_demo_run")   # one folder per artifact, side by side (the published repository layout)
SELF = "art_iYh9zVP7myjJ"      # this artifact: eval.py's ROOT is DEMO / SELF


def write_example(ex: dict) -> None:
    p = DEMO / ex["path"]
    p.parent.mkdir(parents=True, exist_ok=True)
    c = ex["content"]
    if ex["format"] == "text":            # byte-identical copy
        p.write_bytes(c.encode("utf-8"))
    elif ex["format"] == "json":          # pruned JSON
        p.write_text(json.dumps(c, indent=1))
    elif ex["format"] == "jsonl":         # pruned rows, one JSON object per line
        p.write_text("".join(json.dumps(r) + "\n" for r in c))
    elif ex["format"] == "npz":           # stored geodesics: little-endian float64 bytes, rebuilt bit for bit
        arrays = {"labels": np.array(c["labels"]), "L": np.array(c["L"])}
        for k in ("geo_k15", "geo_k10"):
            a = c[k]
            v = np.frombuffer(base64.b64decode(a["b64"]), dtype=a["dtype"])
            if "block_rows" in a:         # only the non-banned x banned block is shipped; NaN elsewhere
                full = np.full(a["shape"], np.nan)
                full[np.ix_(a["block_rows"], a["block_cols"])] = v.reshape(len(a["block_rows"]), len(a["block_cols"]))
                v = full
            arrays[k] = v.reshape(a["shape"])
        np.savez(p, **arrays)
    else:
        raise ValueError(f"unknown format {ex['format']}")


for ex in EXAMPLES:
    write_example(ex)
for d in ("src", "audit"):
    (DEMO / SELF / d).mkdir(parents=True, exist_ok=True)
os.environ["S0_DEPS_DIR"] = str(DEMO.resolve())   # where src/recompute.py and the audits look for the six artifacts
for art in sorted({e["path"].split("/")[0] for e in EXAMPLES}):
    fs = [DEMO / e["path"] for e in EXAMPLES if e["path"].startswith(art + "/")]
    print(f"{art}/: {len(fs):2d} files, {sum(f.stat().st_size for f in fs) / 1e6:.2f} MB")


## Item 0e, part 1: `src/alias_geometry.py` (unchanged)

Item 0e asks whether iteration 8's "opposite" landing is a property of the alias-named manifold. The definitions below
were committed in `predictions/commit_s0e.json` before any alias-level number was computed.
- **g(alg, alias).** This uses iteration 8's stored kNN-geodesic matrix between the 300 forced prompts: 10 algorithms ×
  10 templates × 3 aliases, using states averaged over each prompt's task-specification tokens, at block 31 of
  Qwen3-32B. For each banned algorithm (BFS, DFS, IDDFS, A* and greedy best-first), take the mean geodesic between the
  10 prompts of one (algorithm, alias) set and that algorithm's 30 prompts. g is the minimum over the five. A small g
  means close to the banned five (the *edge*); a large g means far from them (the *opposite* side).
- **Primary overlap.** The number of the 11 committed regex detectors that match the alias string: D1 `bfs`,
  D2 `breadth-first`, …, D11 `heuristic`.
- **Secondary overlap.** The largest content-word Jaccard between the alias and any of the 15 banned aliases.

The module also holds the statistics helpers:
- Spearman correlation;
- exact permutation nulls over every distinct arrangement of a tied vector;
- the within-algorithm (stratified) test over all (3!)^5 relabelings;
- the template bootstrap, in which a resampled geodesic mean is a quadratic form in template weights;
- classical MDS;
- the committed verdict wording rule.

`eval.py` checks this file's sha256 against the committed hash.

In [ ]:
%%writefile eval_demo_run/art_iYh9zVP7myjJ/src/alias_geometry.py
#!/usr/bin/env python3
"""Item 0e: is iteration 8's 'opposite' landing a property of the alias-named manifold?

Works only on iteration 8's own stored state of art_R-HSmXq2t97f, read in place:
  cache/states/geo_C6_nf4.npz   geo_k10, geo_k15 (300 x 300 kNN geodesics at block 31), labels, L
  cache/states/smean_C6_nf4.npz kind, algorithm, group (= template * 3 + alias), template, smean, blocks
The k-NN graph used for inference is never refitted; gate E4 recomputes it once, with separate code, as a cross-check.

Definitions (committed in predictions/commit_s0e.json before any alias-level number was computed):
  g(alg, alias) = min over banned b of the mean geodesic between the 10 points of (alg, alias) and the 30 points of b
  primary overlap   = number of distinct committed detectors D1-D11 matching the alias string
  secondary overlap = max content-token Jaccard with any of the 15 banned aliases
"""
from __future__ import annotations

import ast
import itertools
import math
import re
from pathlib import Path
from typing import Any

import numpy as np
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components, shortest_path
from scipy.spatial import ConvexHull, cKDTree
from scipy.spatial.distance import cdist
from scipy.stats import rankdata

BANNED = ["bfs", "dfs", "iddfs", "astar", "gbfs"]
NONBANNED = ["bibfs", "dijkstra_unit", "idastar", "jps", "biastar"]   # CANON_ORDER restricted to the non-banned five
DISPLAY = {"bibfs": "bidirectional BFS", "dijkstra_unit": "unit-weight Dijkstra", "idastar": "IDA*",
           "jps": "jump point search", "biastar": "bidirectional A*"}
DETECTORS: list[tuple[str, str]] = [
    ("D1_bfs", r"\bbfs\b"), ("D2_breadth_first", r"breadth[-_ ]?first"), ("D3_dfs", r"\bdfs\b"),
    ("D4_depth_first", r"depth[-_ ]?first"), ("D5_iddfs", r"\biddfs\b"),
    ("D6_iterative_deepening", r"iterative[-_ ]?deepening"), ("D7_astar", r"a\*|a[-_ ]?star|astar"),
    ("D8_greedy", r"greedy"), ("D9_best_first", r"best[-_ ]?first"), ("D10_manhattan", r"manhattan"),
    ("D11_heuristic", r"heuristic")]
EXPECTED_PRIMARY = {"bibfs": [1, 1, 1], "dijkstra_unit": [0, 0, 0], "idastar": [1, 2, 3], "jps": [0, 0, 0],
                    "biastar": [1, 1, 1]}
STOPWORDS = {"a", "an", "the", "with", "from", "both", "at", "once", "on", "run", "and", "of", "to", "s", "search",
             "algorithm"}
N_TEMPLATES, N_ALIAS = 10, 3
N_BOOT, N_PLACEBO, N_MC_SECONDARY = 2000, 1000, 200_000
SEED = 20261023


# ------------------------------------------------------------------------------------------- inputs from source code
def read_literal(py: Path, name: str) -> Any:
    """Read a module-level literal assignment (e.g. ALIAS_TABLE) from a source file without importing it."""
    tree = ast.parse(py.read_text())
    for node in tree.body:
        tgt = None
        if isinstance(node, ast.Assign) and len(node.targets) == 1 and isinstance(node.targets[0], ast.Name):
            tgt, val = node.targets[0].id, node.value
        elif isinstance(node, ast.AnnAssign) and isinstance(node.target, ast.Name) and node.value is not None:
            tgt, val = node.target.id, node.value
        if tgt == name:
            return ast.literal_eval(val)
    raise KeyError(f"{name} not found in {py}")


# ------------------------------------------------------------------------------------------- lexical overlap scores
def primary_overlap(alias: str) -> tuple[int, list[str]]:
    hits = [n for n, pat in DETECTORS if re.search(pat, alias, flags=re.IGNORECASE)]
    return len(hits), hits


def content_tokens(s: str) -> set[str]:
    return {t for t in re.split(r"[^a-z0-9*]+", s.lower()) if t and t not in STOPWORDS}


def secondary_overlap(alias: str, banned_aliases: list[str]) -> tuple[float, str]:
    a = content_tokens(alias)
    best, arg = 0.0, ""
    for b in banned_aliases:
        bt = content_tokens(b)
        j = len(a & bt) / len(a | bt) if (a | bt) else 0.0
        if j > best + 1e-15:
            best, arg = j, b
    return best, arg


def overlap_table(alias_table: dict[str, list[str]]) -> list[dict[str, Any]]:
    banned_aliases = [s for b in BANNED for s in alias_table[b]]
    rows = []
    for a in NONBANNED:
        for j, s in enumerate(alias_table[a]):
            p, hits = primary_overlap(s)
            sec, arg = secondary_overlap(s, banned_aliases)
            rows.append({"algorithm": a, "alias_index": j, "alias": s, "primary": p, "detectors": hits,
                         "secondary": round(sec, 12), "secondary_argmax_banned_alias": arg,
                         "content_tokens": sorted(content_tokens(s))})
    return rows


# ------------------------------------------------------------------------------------------- statistics helpers
def spearman(x: np.ndarray, y: np.ndarray) -> float:
    rx, ry = rankdata(x), rankdata(y)
    rx, ry = rx - rx.mean(), ry - ry.mean()
    den = math.sqrt(float((rx ** 2).sum() * (ry ** 2).sum()))
    return float((rx * ry).sum() / den) if den > 0 else float("nan")


def distinct_assignments(values: list[float]) -> np.ndarray:
    """All distinct arrangements of a multiset over len(values) positions (rows), by recursive placement."""
    vals, counts = np.unique(np.asarray(values, dtype=float), return_counts=True)
    n = len(values)
    rows: list[np.ndarray] = []

    def place(free: tuple[int, ...], k: int, cur: np.ndarray) -> None:
        if k == len(vals):
            rows.append(cur.copy())
            return
        for pos in itertools.combinations(free, int(counts[k])):
            cur[list(pos)] = vals[k]
            rest = tuple(f for f in free if f not in pos)
            place(rest, k + 1, cur)

    place(tuple(range(n)), 0, np.full(n, np.nan))
    return np.stack(rows)


def n_distinct_assignments(values: list[float]) -> int:
    _, counts = np.unique(np.asarray(values, dtype=float), return_counts=True)
    out = math.factorial(len(values))
    for c in counts:
        out //= math.factorial(int(c))
    return out


def null_rhos_exact(values: list[float]) -> np.ndarray:
    """Spearman of g-ranks 1..n (no ties) against every distinct arrangement of the overlap multiset. The distribution
    does not depend on which g-rank order is observed, so one table serves every observed g without ties."""
    A = distinct_assignments(values)
    R = np.apply_along_axis(rankdata, 1, A)
    R = R - R.mean(1, keepdims=True)
    gr = np.arange(1, A.shape[1] + 1, dtype=float)
    gr = gr - gr.mean()
    den = np.sqrt((R ** 2).sum(1) * (gr ** 2).sum())
    return (R @ gr) / den


def p_from_null(rho: float, null: np.ndarray) -> dict[str, float]:
    eps = 1e-12
    return {"p_lower": float(np.mean(null <= rho + eps)), "p_two_sided": float(np.mean(np.abs(null) >= abs(rho) - eps))}


def null_rhos_mc(g: np.ndarray, values: np.ndarray, n: int, rng: np.random.Generator) -> np.ndarray:
    rg = rankdata(g)
    rg = rg - rg.mean()
    out = np.empty(n)
    for s in range(0, n, 20_000):
        m = min(20_000, n - s)
        P = np.stack([rng.permutation(values) for _ in range(m)])
        R = np.apply_along_axis(rankdata, 1, P)
        R = R - R.mean(1, keepdims=True)
        out[s:s + m] = (R @ rg) / np.sqrt((R ** 2).sum(1) * (rg ** 2).sum())
    return out


# ------------------------------------------------------------------------------------------- geometry
def alias_sets(labels: np.ndarray, group: np.ndarray) -> dict[tuple[str, int], np.ndarray]:
    return {(a, j): np.where((labels == a) & (group % 3 == j))[0] for a in NONBANNED for j in range(N_ALIAS)}


def g_alias(geo: np.ndarray, labels: np.ndarray, group: np.ndarray) -> dict[tuple[str, int], float]:
    sets = alias_sets(labels, group)
    bidx = {b: np.where(labels == b)[0] for b in BANNED}
    return {k: min(float(geo[np.ix_(ix, bidx[b])].mean()) for b in BANNED) for k, ix in sets.items()}


def g_algorithm(geo: np.ndarray, labels: np.ndarray) -> dict[str, float]:
    bidx = {b: np.where(labels == b)[0] for b in BANNED}
    return {a: min(float(geo[np.ix_(np.where(labels == a)[0], bidx[b])].mean()) for b in BANNED) for a in NONBANNED}


def classify(g: dict[str, float]) -> dict[str, str]:
    med = float(np.median(list(g.values())))
    return {a: ("EDGE" if v < med - 1e-12 else "OPPOSITE" if v > med + 1e-12 else "MIDDLE") for a, v in g.items()}


def knn_geodesic_independent(X: np.ndarray, k: int) -> tuple[np.ndarray, int]:
    """Cross-check implementation (E4): cKDTree k-NN, symmetrised by union, every pair of components joined by its
    shortest inter-component edge, Dijkstra shortest paths. Shares no code with iteration 8's geometry.py."""
    n = len(X)
    tree = cKDTree(X)
    dist, idx = tree.query(X, k=k + 1)
    rows, cols, vals = [], [], []
    for i in range(n):
        nb = [(d, j) for d, j in zip(dist[i], idx[i]) if j != i][:k]
        for d, j in nb:
            rows += [i, j]
            cols += [j, i]
            vals += [max(d, 1e-12)] * 2
    W = csr_matrix((vals, (rows, cols)), shape=(n, n))
    W.sum_duplicates()
    W.data = np.where(W.data > 0, W.data, 1e-12)
    # union symmetrisation: duplicate entries were summed, so restore the plain distance on every stored edge
    W = W.tocoo()
    D_full = np.sqrt(np.maximum(((X[W.row] - X[W.col]) ** 2).sum(1), 0.0))
    W = csr_matrix((np.maximum(D_full, 1e-12), (W.row, W.col)), shape=(n, n))
    ncomp, lab = connected_components(W, directed=False)
    if ncomp > 1:
        W = W.tolil()
        comps = [np.where(lab == c)[0] for c in range(ncomp)]
        for a in range(ncomp):
            for b in range(a + 1, ncomp):
                Da = cdist(X[comps[a]], X[comps[b]])
                i, j = np.unravel_index(np.argmin(Da), Da.shape)
                W[comps[a][i], comps[b][j]] = W[comps[b][j], comps[a][i]] = max(float(Da[i, j]), 1e-12)
        W = W.tocsr()
    return shortest_path(W, method="D", directed=False), int(ncomp)


def classical_mds(D: np.ndarray, k: int = 2) -> tuple[np.ndarray, float]:
    n = len(D)
    J = np.eye(n) - 1.0 / n
    B = -0.5 * J @ (D ** 2) @ J
    w, V = np.linalg.eigh(B)
    i = np.argsort(w)[::-1][:k]
    Y = V[:, i] * np.sqrt(np.maximum(w[i], 0))
    d = np.sqrt(((Y[:, None, :] - Y[None, :, :]) ** 2).sum(-1))
    iu = np.triu_indices(n, 1)
    stress = float(np.sqrt(((D[iu] - d[iu]) ** 2).sum() / (D[iu] ** 2).sum()))
    return Y, stress


# ------------------------------------------------------------------------------------------- within-algorithm tests
def within_perms() -> list[tuple[int, ...]]:
    return list(itertools.permutations(range(N_ALIAS)))


def stratified_stat(g: np.ndarray, x: np.ndarray) -> tuple[float, int]:
    """Mean over algorithms with non-constant x of the 3-point within-algorithm Spearman (g, x). g, x: (5, 3)."""
    vals = [spearman(g[a], x[a]) for a in range(len(g)) if np.ptp(x[a]) > 1e-12]
    return (float(np.mean(vals)) if vals else float("nan")), len(vals)


def fe_beta(g: np.ndarray, x: np.ndarray) -> float:
    gc = g - g.mean(1, keepdims=True)
    xc = x - x.mean(1, keepdims=True)
    den = float((xc ** 2).sum())
    return float((gc * xc).sum() / den) if den > 0 else float("nan")


def stratified_exact(g: np.ndarray, x: np.ndarray) -> dict[str, Any]:
    """Exact null over (3!)^5 within-algorithm relabelings of x; reports both the nominal and effective spaces."""
    obs, n_used = stratified_stat(g, x)
    b_obs = fe_beta(g, x)
    P = within_perms()
    stats, betas = [], []
    for combo in itertools.product(P, repeat=len(g)):
        xp = np.stack([x[a][list(combo[a])] for a in range(len(g))])
        stats.append(stratified_stat(g, xp)[0])
        betas.append(fe_beta(g, xp))
    stats, betas = np.array(stats), np.array(betas)
    eps = 1e-12
    n_var = int(sum(np.ptp(x[a]) > 1e-12 for a in range(len(g))))
    return {"stat": obs, "n_algorithms_used": n_used, "n_constant_skipped": len(g) - n_used,
            "n_arrangements_nominal": len(stats), "n_arrangements_effective": 6 ** n_var,
            "floor_effective": 1.0 / 6 ** n_var if n_var else float("nan"),
            "p_lower": float(np.mean(stats <= obs + eps)), "p_upper": float(np.mean(stats >= obs - eps)),
            "p_two_sided": float(np.mean(np.abs(stats) >= abs(obs) - eps)),
            "beta": b_obs, "beta_p_lower": float(np.mean(betas <= b_obs + eps)),
            "beta_p_two_sided": float(np.mean(np.abs(betas) >= abs(b_obs) - eps)),
            "null_mean": float(np.nanmean(stats)), "null_sd": float(np.nanstd(stats))}


# ------------------------------------------------------------------------------------------- bootstrap over templates
def template_blocks(geo: np.ndarray, labels: np.ndarray, group: np.ndarray) -> dict[tuple[str, int, str], np.ndarray]:
    """For each non-banned alias set and banned algorithm: a 10 x 10 (template x template) matrix of mean geodesics,
    averaging the banned side over its 3 aliases, so that resampled means are quadratic forms in template weights."""
    tmpl = group // 3
    out = {}
    for a in NONBANNED:
        for j in range(N_ALIAS):
            ia = {int(t): int(i) for i in np.where((labels == a) & (group % 3 == j))[0] for t in [tmpl[i]]}
            for b in BANNED:
                M = np.zeros((N_TEMPLATES, N_TEMPLATES))
                for t in range(N_TEMPLATES):
                    for t2 in range(N_TEMPLATES):
                        ib = np.where((labels == b) & (tmpl == t2))[0]
                        M[t, t2] = geo[ia[t], ib].mean()
                out[(a, j, b)] = M
    return out


def boot_g(blocks: dict[tuple[str, int, str], np.ndarray], W: np.ndarray) -> np.ndarray:
    """(n_boot, 15) alias-level g under template weights W (n_boot, 10)."""
    tot = W.sum(1) ** 2
    G = np.empty((len(W), len(NONBANNED) * N_ALIAS))
    c = 0
    for a in NONBANNED:
        for j in range(N_ALIAS):
            m = np.stack([np.einsum("rt,ts,rs->r", W, blocks[(a, j, b)], W) / tot for b in BANNED], 1)
            G[:, c] = m.min(1)
            c += 1
    return G


def boot_g_alg(blocks: dict[tuple[str, int, str], np.ndarray], W: np.ndarray) -> np.ndarray:
    """(n_boot, 5) algorithm-level g: the algorithm's 30 points are the 3 alias sets pooled."""
    tot = W.sum(1) ** 2
    out = np.empty((len(W), len(NONBANNED)))
    for ai, a in enumerate(NONBANNED):
        m = np.stack([np.mean([np.einsum("rt,ts,rs->r", W, blocks[(a, j, b)], W) / tot for j in range(N_ALIAS)], 0)
                      for b in BANNED], 1)
        out[:, ai] = m.min(1)
    return out


def rowwise_spearman(G: np.ndarray, x: np.ndarray) -> np.ndarray:
    RG = np.apply_along_axis(rankdata, 1, G)
    RG = RG - RG.mean(1, keepdims=True)
    rx = rankdata(x)
    rx = rx - rx.mean()
    return (RG @ rx) / np.sqrt((RG ** 2).sum(1) * (rx ** 2).sum())


def verdict_sentence(rho: float, p: float, strat_p: float) -> tuple[str, str]:
    if rho <= -0.6 and strat_p < 0.05:
        return "NAMES", "names move the geometry even with the algorithm held fixed"
    if rho <= -0.6:
        return "CONFOUNDED", (f"g follows alias overlap across algorithms (rho {rho:.2f}, p {p:.3g}), but the pattern is "
                              "carried by algorithm identity; on this manifold names and mechanisms are confounded, "
                              "and TP-G's name-free manifolds decide")
    return "NOT_TRACKED", (f"iteration 8's geodesics do not track alias overlap at the committed strength "
                           f"(rho {rho:.2f})")


def convex_outline(P: np.ndarray) -> np.ndarray:
    try:
        h = ConvexHull(P)
        v = list(h.vertices) + [h.vertices[0]]
        return P[v]
    except Exception:  # noqa: BLE001 - degenerate hulls (collinear points) fall back to the points themselves
        return P


## Item 0e, part 2: `src/stage_0e.py` (unchanged)

`run()` reads the alias table from iteration 8's own source with `ast`, without importing it, then loads the stored
geodesics and the prompt index. It then does four things:

1. **Gates.** The alias-level numbers are reported only if E1–E3 pass.
   - E1: the stored labels equal the index's labels.
   - E2: the 300 (algorithm, template, alias) triples are exactly the forced set.
   - E3: the algorithm-level g recomputed here equals iteration 8's `geometry_M1.json` to within 1e-9, with the same
     edge/opposite classes.
   - E4: an independent kNN-geodesic recomputation from the 181 MB states file. It is skipped in this copy.
2. **Statistics at k = 15 and k = 10.**
   - the Spearman correlation of the 15 alias-level g with overlap, and its exact p over all 360,360 arrangements;
   - the 5-centroid test (120 permutations, with its attainable floor);
   - the within-algorithm test and the fixed-effects slope;
   - template-bootstrap 95% intervals.
3. **Controls.** Placebos must fail: shuffled point labels, random overlap vectors and within-algorithm shuffles. A
   planted effect must be recovered.
4. **Outputs.** The verdict by the committed rule, the tables `alias_g.csv` and `alias_stats.csv`, and the figure
   `figures/alias_overlap.png` with its activation-space panel (classical MDS of the geodesics).

In [ ]:
%%writefile eval_demo_run/art_iYh9zVP7myjJ/src/stage_0e.py
#!/usr/bin/env python3
"""Runner for item 0e (alias-named manifold). Called by eval.py after predictions/commit_s0e.json is on disk."""
from __future__ import annotations

import csv
import gc
import json
import math
from pathlib import Path
from typing import Any

import numpy as np
from loguru import logger
from scipy.stats import rankdata

import alias_geometry as AG

ART_TP = "art_R-HSmXq2t97f"


def _avg_rank_map(values: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    uniq, counts = np.unique(values, return_counts=True)
    starts = np.concatenate([[0], np.cumsum(counts)[:-1]])
    return uniq, starts + (counts + 1) / 2.0


def exact_null(values: np.ndarray) -> np.ndarray:
    """Spearman null over every distinct arrangement of the overlap multiset (g ranks 1..n, no ties in g)."""
    A = AG.distinct_assignments(list(values))
    uniq, avg = _avg_rank_map(values)
    R = avg[np.searchsorted(uniq, A)]
    R = R - R.mean(1, keepdims=True)
    gr = np.arange(1, A.shape[1] + 1, dtype=float)
    gr -= gr.mean()
    return (R @ gr) / np.sqrt((R ** 2).sum(1) * (gr ** 2).sum())


def mc_null(values: np.ndarray, n: int, rng: np.random.Generator) -> np.ndarray:
    rx = rankdata(values)
    gr = np.arange(1, len(values) + 1, dtype=float)
    gr -= gr.mean()
    out = np.empty(n)
    for s in range(0, n, 50_000):
        m = min(50_000, n - s)
        R = rng.permuted(np.tile(rx, (m, 1)), axis=1)
        R = R - R.mean(1, keepdims=True)
        out[s:s + m] = (R @ gr) / np.sqrt((R ** 2).sum(1) * (gr ** 2).sum())
    return out


def pvals(rho: float, null: np.ndarray) -> dict[str, float]:
    eps = 1e-12
    return {"p_lower": float(np.mean(null <= rho + eps)), "p_two_sided": float(np.mean(np.abs(null) >= abs(rho) - eps)),
            "n_null": int(len(null))}


def five_centroid(g_alg: np.ndarray, x_alg: np.ndarray) -> dict[str, Any]:
    import itertools
    obs = AG.spearman(g_alg, x_alg)
    rhos = np.array([AG.spearman(g_alg, x_alg[list(p)]) for p in itertools.permutations(range(len(x_alg)))])
    eps = 1e-12
    # attainable floor: the number of permutations that reproduce the most extreme (lowest) rho
    floor = float(np.mean(rhos <= rhos.min() + eps))
    return {"rho": obs, "p_lower": float(np.mean(rhos <= obs + eps)),
            "p_two_sided": float(np.mean(np.abs(rhos) >= abs(obs) - eps)), "n_perm": len(rhos),
            "floor_lower": floor, "floor_nominal": 1 / len(rhos)}


def run(art_dir: Path, out_dir: Path, rng_seed: int = AG.SEED) -> dict[str, Any]:
    src = art_dir / "src"
    res: dict[str, Any] = {"gates": {}, "flags": []}
    alias_table = AG.read_literal(src / "common.py", "ALIAS_TABLE")
    canon = AG.read_literal(src / "common.py", "CANON_ORDER")
    rows = AG.overlap_table(alias_table)
    prim_by = {a: [r["primary"] for r in rows if r["algorithm"] == a] for a in AG.NONBANNED}
    res["primary_vectors"] = prim_by
    res["primary_vectors_match_commit"] = prim_by == AG.EXPECTED_PRIMARY
    assert res["primary_vectors_match_commit"], f"detector code wrong: {prim_by} != {AG.EXPECTED_PRIMARY}"
    x1 = np.array([r["primary"] for r in rows], dtype=float)
    x2 = np.array([r["secondary"] for r in rows], dtype=float)
    x0 = (x1 > 0).astype(float)
    res["overlap_rows"] = rows
    res["primary_multiset"] = {int(v): int(c) for v, c in zip(*np.unique(x1, return_counts=True))}

    # ---------------------------------------------------------------------------------------- load stored state
    geo_p = art_dir / "cache/states/geo_C6_nf4.npz"
    sm_p = art_dir / "cache/states/smean_C6_nf4.npz"
    zg = np.load(geo_p, allow_pickle=False)
    geo = {15: zg["geo_k15"].astype(np.float64), 10: zg["geo_k10"].astype(np.float64)}
    labels_geo = zg["labels"].astype(str)
    L = int(zg["L"])
    have_states = sm_p.exists()
    if have_states:
        zs = np.load(sm_p, allow_pickle=False)
    else:   # published copy: the 181 MB states file is not published; use the vendored index (same rows, hashed)
        import json as _json
        zs = {k: np.array(v) for k, v in _json.loads((out_dir / "inputs/forced_index_C6_nf4.json").read_text()).items()
              if isinstance(v, list)}
        res["flags"].append("states file not available: E1-E3 run on the vendored index, E4 not run")
    kind = zs["kind"].astype(str)
    F = kind == "forced"
    labels = zs["algorithm"].astype(str)[F]
    group = zs["group"].astype(int)[F]
    template = zs["template"].astype(str)[F]
    blocks = [int(b) for b in zs["blocks"]]

    # E1 labels
    e1 = bool(len(labels) == len(labels_geo) and np.all(labels == labels_geo))
    res["gates"]["E1_labels_equal"] = e1
    # E2 multiset of (algorithm, template, alias) equals forced_set() of prompts_tp.py
    trip = list(zip(labels.tolist(), (group // 3).tolist(), (group % 3).tolist()))
    expect = [(a, t, j) for a in canon for t in range(10) for j in range(3)]
    e2 = sorted(trip) == sorted(expect) and len(set(trip)) == 300
    res["gates"]["E2_multiset_equal_forced_set"] = bool(e2)
    res["gates"]["E2_order_equal_forced_set"] = bool(trip == expect)
    res["gates"]["E2_template_field_consistent"] = bool(np.all(template == (group // 3).astype(str)))
    # E3 algorithm-level g against geometry_M1.json
    m1 = json.loads((art_dir / "results/geometry_M1.json").read_text())["M1"]
    e3 = {}
    for k in (15, 10):
        ga = AG.g_algorithm(geo[k], labels_geo)
        diff = max(abs(ga[a] - m1[f"k{k}"]["g"][a]) for a in AG.NONBANNED)
        cls = AG.classify(ga)
        e3[f"k{k}"] = {"max_abs_diff": diff, "classes_match": cls == m1[f"k{k}"]["classes"], "g": ga, "classes": cls}
    res["gates"]["E3_detail"] = e3
    res["gates"]["E3_g_matches_M1"] = bool(all(v["max_abs_diff"] < 1e-9 and v["classes_match"] for v in e3.values()))
    res["L"] = L
    # E4 independent cross-check from smean at block 31
    if not have_states:
        res["gates"]["E4_pass"] = None
        res["gates"]["E4_spearman_vs_stored_k15"] = None
    try:
        if not have_states:
            raise ValueError("states file not available")
        X = zs["smean"][F][:, blocks.index(L)].astype(np.float64)
        geo_x, ncomp = AG.knn_geodesic_independent(X, 15)
        iu = np.triu_indices(300, 1)
        rho_e4 = AG.spearman(geo_x[iu], geo[15][iu])
        rel = float(np.max(np.abs(geo_x[iu] - geo[15][iu]) / np.maximum(geo[15][iu], 1e-12)))
        res["gates"]["E4_spearman_vs_stored_k15"] = rho_e4
        res["gates"]["E4_max_rel_diff"] = rel
        res["gates"]["E4_components_before_join"] = ncomp
        res["gates"]["E4_pass"] = bool(rho_e4 >= 0.999)
        if rho_e4 < 0.999:
            res["flags"].append("E4_below_0.999")
        if rho_e4 < 0.99:
            res["flags"].append("geodesics not independently reproduced")
        del X, geo_x
    except (KeyError, ValueError, MemoryError) as exc:
        logger.error(f"E4 failed to run: {exc}")
        res["gates"]["E4_pass"] = None if not have_states else False
        res["flags"].append(f"E4_not_run: {exc}")
    del zs
    gc.collect()
    hard = res["gates"]["E1_labels_equal"] and res["gates"]["E2_multiset_equal_forced_set"] and res["gates"][
        "E3_g_matches_M1"]
    res["gates"]["hard_gates_pass"] = bool(hard)
    if not hard:
        res["flags"].append("hard gate failed: alias-level numbers withheld")
        return res

    rng = np.random.default_rng(rng_seed)
    null1 = exact_null(x1)
    null0 = exact_null(x0)
    n_sec = AG.n_distinct_assignments(list(x2))
    null2 = exact_null(x2) if n_sec <= 2_000_000 else mc_null(x2, AG.N_MC_SECONDARY, rng)
    res["null_sizes"] = {"primary_exact": int(len(null1)), "binary_exact": int(len(null0)),
                         "secondary_distinct": int(n_sec), "secondary_method": "exact" if n_sec <= 2_000_000 else
                         f"monte_carlo_{AG.N_MC_SECONDARY}"}
    order = [(a, j) for a in AG.NONBANNED for j in range(3)]
    xa1 = np.array([np.mean(prim_by[a]) for a in AG.NONBANNED])
    xa2 = np.array([np.mean([r["secondary"] for r in rows if r["algorithm"] == a]) for a in AG.NONBANNED])
    tblocks = {}
    per_k: dict[str, Any] = {}
    W = np.stack([np.bincount(rng.integers(0, 10, 10), minlength=10) for _ in range(AG.N_BOOT)]).astype(float)
    for k in (15, 10):
        ga = AG.g_alias(geo[k], labels_geo, group)
        g15 = np.array([ga[o] for o in order])
        galg = np.array([e3[f"k{k}"]["g"][a] for a in AG.NONBANNED])
        if len(np.unique(g15)) < 15:
            res["flags"].append(f"ties in alias-level g at k{k}")
        r1, r2, r0 = AG.spearman(g15, x1), AG.spearman(g15, x2), AG.spearman(g15, x0)
        st2 = AG.stratified_exact(g15.reshape(5, 3), x2.reshape(5, 3))
        st1 = AG.stratified_exact(g15.reshape(5, 3), x1.reshape(5, 3))
        c5 = five_centroid(galg, xa1)
        c5s = five_centroid(galg, xa2)
        tblocks[k] = AG.template_blocks(geo[k], labels_geo, group)
        GB = AG.boot_g(tblocks[k], W)
        GA = AG.boot_g_alg(tblocks[k], W)
        b1, b2 = AG.rowwise_spearman(GB, x1), AG.rowwise_spearman(GB, x2)
        b5 = np.array([AG.spearman(GA[i], xa1) for i in range(len(GA))])
        ci = lambda v: [float(np.nanpercentile(v, 2.5)), float(np.nanpercentile(v, 97.5))]  # noqa: E731
        per_k[f"k{k}"] = {
            "g_alias": {f"{a}|{j}": float(ga[(a, j)]) for a, j in order}, "g_algorithm": dict(zip(AG.NONBANNED, galg.tolist())),
            "rho_primary": r1, **{f"primary_{kk}": v for kk, v in pvals(r1, null1).items()},
            "rho_secondary": r2, **{f"secondary_{kk}": v for kk, v in pvals(r2, null2).items()},
            "rho_binary": r0, **{f"binary_{kk}": v for kk, v in pvals(r0, null0).items()},
            "ci95_primary": ci(b1), "ci95_secondary": ci(b2), "ci95_5centroid": ci(b5),
            "boot_frac_rho_primary_le_-0.6": float(np.mean(b1 <= -0.6)),
            "five_centroid_primary": c5, "five_centroid_secondary": c5s,
            "stratified_secondary": st2, "stratified_primary": st1}
        logger.info(f"0e k{k}: rho1 {r1:.3f} p {per_k[f'k{k}']['primary_p_lower']:.4f}; rho2 {r2:.3f}; "
                    f"5c {c5['rho']:.3f} p {c5['p_lower']:.3f}; strat2 {st2['stat']:.3f} p {st2['p_lower']:.3f}")
    res["per_k"] = per_k

    # ---------------------------------------------------------------------------------------- placebos and control
    g15 = np.array([per_k["k15"]["g_alias"][f"{a}|{j}"] for a, j in order])
    alias_lab = np.array([f"{a}|{j}" for a, j in zip(labels_geo, group % 3)])
    rho_p1, p_p1 = [], []
    for _ in range(AG.N_PLACEBO):
        perm = rng.permutation(300)
        lab_s = labels_geo[perm]
        grp_s = group[perm]
        gs = AG.g_alias(geo[15], lab_s, grp_s)
        v = np.array([gs[o] for o in order])
        r = AG.spearman(v, x1)
        rho_p1.append(r)
        p_p1.append(pvals(r, null1)["p_lower"])
    rho_p1, p_p1 = np.array(rho_p1), np.array(p_p1)
    p_p2 = np.array([pvals(AG.spearman(g15, rng.permutation(x1)), null1)["p_lower"] for _ in range(AG.N_PLACEBO)])
    s_p3 = []
    for _ in range(AG.N_PLACEBO):
        gp = np.stack([g15.reshape(5, 3)[a][rng.permutation(3)] for a in range(5)])
        s_p3.append(AG.stratified_stat(gp, x2.reshape(5, 3))[0])
    s_p3 = np.array(s_p3)
    planted = []
    for s in range(200):
        r_ = np.random.default_rng(10_000 + s)
        gp = 12 - 2 * x1 + r_.normal(0, 0.5, size=15)
        rr = AG.spearman(gp, x1)
        planted.append((rr, pvals(rr, null1)["p_lower"]))
    planted = np.array(planted)
    res["placebos"] = {
        "P1_shuffled_alias_labels": {"n": AG.N_PLACEBO, "rho_mean": float(rho_p1.mean()), "rho_sd": float(rho_p1.std()),
                                     "share_p_lt_0.05": float(np.mean(p_p1 < 0.05)),
                                     "fails_as_required": bool(np.mean(p_p1 < 0.05) <= 0.10 and abs(rho_p1.mean()) < 0.1)},
        "P2_random_overlap_vector": {"n": AG.N_PLACEBO, "share_p_lt_0.05": float(np.mean(p_p2 < 0.05)),
                                     "p_deciles": np.percentile(p_p2, np.arange(10, 100, 10)).round(3).tolist(),
                                     "fails_as_required": bool(np.mean(p_p2 < 0.05) <= 0.10)},
        "P3_within_algorithm_shuffles": {"n": AG.N_PLACEBO, "stat_mean": float(np.nanmean(s_p3)),
                                         "stat_sd": float(np.nanstd(s_p3)),
                                         "fails_as_required": bool(abs(np.nanmean(s_p3)) < 0.1)},
        "C1_planted": {"seed0_rho": float(planted[0, 0]), "seed0_p_lower": float(planted[0, 1]),
                       "seed0_succeeds": bool(planted[0, 0] <= -0.6 and planted[0, 1] < 0.05),
                       "power_200_seeds": float(np.mean((planted[:, 0] <= -0.6) & (planted[:, 1] < 0.05)))}}
    k15 = per_k["k15"]
    code, sentence = AG.verdict_sentence(k15["rho_primary"], k15["primary_p_lower"],
                                         k15["stratified_secondary"]["p_lower"])
    res["verdict"] = {"code": code, "sentence": sentence,
                      "rule_inputs": {"rho_k15_primary": k15["rho_primary"], "p_k15_primary": k15["primary_p_lower"],
                                      "stratified_secondary_p": k15["stratified_secondary"]["p_lower"]}}
    _write_tables(out_dir, per_k, rows, order)
    try:
        _figure(out_dir, geo[15], labels_geo, group, per_k, rows, order, m1)
        res["figure"] = ["figures/alias_overlap.pdf", "figures/alias_overlap.png"]
    except (ValueError, RuntimeError) as exc:
        logger.error(f"figure failed: {exc}")
        res["flags"].append(f"figure_failed: {exc}")
    return res


def _write_tables(out_dir: Path, per_k: dict[str, Any], rows: list[dict[str, Any]], order: list) -> None:
    with (out_dir / "tables/alias_g.csv").open("w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["level", "algorithm", "alias_index", "alias", "primary_overlap", "secondary_overlap", "g_k15",
                    "g_k10"])
        for r, (a, j) in zip(rows, order):
            w.writerow(["alias", a, j, r["alias"], r["primary"], round(r["secondary"], 6),
                        round(per_k["k15"]["g_alias"][f"{a}|{j}"], 6), round(per_k["k10"]["g_alias"][f"{a}|{j}"], 6)])
        for a in AG.NONBANNED:
            p = [r["primary"] for r in rows if r["algorithm"] == a]
            s = [r["secondary"] for r in rows if r["algorithm"] == a]
            w.writerow(["centroid", a, "", AG.DISPLAY[a], round(float(np.mean(p)), 6), round(float(np.mean(s)), 6),
                        round(per_k["k15"]["g_algorithm"][a], 6), round(per_k["k10"]["g_algorithm"][a], 6)])
    with (out_dir / "tables/alias_stats.csv").open("w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["k", "statistic", "value", "p_lower", "p_two_sided", "ci95_low", "ci95_high", "null", "note"])
        for k, d in per_k.items():
            w.writerow([k, "spearman_15_primary", d["rho_primary"], d["primary_p_lower"], d["primary_p_two_sided"],
                        *d["ci95_primary"], f"exact {d['primary_n_null']}", "committed primary"])
            w.writerow([k, "spearman_15_secondary", d["rho_secondary"], d["secondary_p_lower"],
                        d["secondary_p_two_sided"], *d["ci95_secondary"], f"{d['secondary_n_null']}", "content Jaccard"])
            w.writerow([k, "spearman_15_binary", d["rho_binary"], d["binary_p_lower"], d["binary_p_two_sided"], "", "",
                        f"exact {d['binary_n_null']}", "any overlap (sensitivity)"])
            c5 = d["five_centroid_primary"]
            w.writerow([k, "spearman_5_centroid_primary", c5["rho"], c5["p_lower"], c5["p_two_sided"],
                        *d["ci95_5centroid"], "exact 120", f"attainable floor {c5['floor_lower']:.4f}"])
            c5s = d["five_centroid_secondary"]
            w.writerow([k, "spearman_5_centroid_secondary", c5s["rho"], c5s["p_lower"], c5s["p_two_sided"], "", "",
                        "exact 120", f"attainable floor {c5s['floor_lower']:.4f}"])
            for nm in ("stratified_secondary", "stratified_primary"):
                s = d[nm]
                w.writerow([k, nm, s["stat"], s["p_lower"], s["p_two_sided"], "", "",
                            f"exact {s['n_arrangements_nominal']} (effective {s['n_arrangements_effective']})",
                            f"{s['n_algorithms_used']} algorithms used"])
                w.writerow([k, f"fe_beta_{nm.split('_')[1]}", s["beta"], s["beta_p_lower"], s["beta_p_two_sided"], "",
                            "", "same stratified null", "g = alpha_algorithm + beta * overlap"])


def _figure(out_dir: Path, geo15: np.ndarray, labels: np.ndarray, group: np.ndarray, per_k: dict[str, Any],
            rows: list[dict[str, Any]], order: list, m1: dict[str, Any]) -> None:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    plt.rcParams.update({"font.size": 9, "pdf.fonttype": 42, "ps.fonttype": 42})
    col = {"bibfs": "#0072B2", "biastar": "#56B4E9", "idastar": "#009E73", "jps": "#E69F00", "dijkstra_unit": "#D55E00"}
    mk = ["o", "s", "^"]
    fig, axes = plt.subplots(1, 3, figsize=(16, 5.2))
    jit = np.random.default_rng(7).uniform(-0.12, 0.12, size=15)
    for ax, k in zip(axes[:2], ("k15", "k10")):
        d = per_k[k]
        for i, ((a, j), r) in enumerate(zip(order, rows)):
            ax.scatter(r["primary"] + jit[i], d["g_alias"][f"{a}|{j}"], c=col[a], marker=mk[j], s=46,
                       edgecolor="k", linewidth=0.4, zorder=3)
        for a in AG.NONBANNED:
            xm = np.mean([r["primary"] for r in rows if r["algorithm"] == a])
            ax.scatter(xm, d["g_algorithm"][a], c=col[a], marker="*", s=330, edgecolor="k", linewidth=0.7, zorder=4,
                       alpha=0.85)
            ax.annotate(f"{AG.DISPLAY[a]} ({m1[k]['classes'][a].lower()})", (xm, d["g_algorithm"][a]),
                        xytext=(6, -3), textcoords="offset points", fontsize=7.5)
        ci = d["ci95_primary"]
        ax.set_title(f"{'A' if k == 'k15' else 'B'}. alias-level g at {k.replace('k', 'k = ')}: rho {d['rho_primary']:.2f}, "
                     f"exact p {d['primary_p_lower']:.3g}\n(template-bootstrap 95% CI {ci[0]:.2f} to {ci[1]:.2f}; "
                     f"5 centroids rho {d['five_centroid_primary']['rho']:.2f}, p {d['five_centroid_primary']['p_lower']:.3g})",
                     fontsize=8.5)
        ax.set_xlabel("primary lexical overlap with banned names (distinct detectors matched; jittered)")
        ax.set_ylabel("g: min over banned of mean geodesic (block 31)")
        ax.set_xticks([0, 1, 2, 3])
        ax.grid(alpha=0.3)
    Y, stress = AG.classical_mds(geo15)
    ax = axes[2]
    ban = np.isin(labels, AG.BANNED)
    ax.scatter(Y[ban, 0], Y[ban, 1], c="#BBBBBB", s=10, label="banned five (pooled)")
    for a in AG.NONBANNED:
        for j in range(3):
            ix = np.where((labels == a) & (group % 3 == j))[0]
            ax.scatter(Y[ix, 0], Y[ix, 1], c=col[a], marker=mk[j], s=16, edgecolor="k", linewidth=0.25,
                       label=f"{AG.DISPLAY[a]}" if j == 0 else None)
            hull = AG.convex_outline(Y[ix])
            ax.plot(hull[:, 0], hull[:, 1], color=col[a], lw=0.9, ls=["-", "--", ":"][j], alpha=0.8)
    ax.set_title(f"C. classical MDS of the stored k = 15 geodesics (300 forced prompts, block 31)\n"
                 f"stress-1 {stress:.2f}; outlines = 15 alias sets (line style = alias 0/1/2)", fontsize=8.5)
    ax.set_xlabel("MDS 1 (geodesic units)")
    ax.set_ylabel("MDS 2")
    ax.legend(fontsize=7, loc="best", markerscale=1.2)
    handles = [plt.Line2D([], [], marker=mk[j], ls="", color="k", label=f"alias {j}") for j in range(3)]
    axes[0].legend(handles=handles, fontsize=7, loc="upper right")
    fig.suptitle("Is iteration 8's 'opposite' landing a property of the alias-named manifold? (iteration-9 Step 0, "
                 "item 0e; stored geodesics of art_R-HSmXq2t97f, Qwen3-32B NF4)", fontsize=10)
    fig.tight_layout()
    fig.savefig(out_dir / "figures/alias_overlap.pdf")
    fig.savefig(out_dir / "figures/alias_overlap.png", dpi=160)
    plt.close(fig)
    per_k["k15"]["mds_stress1"] = stress


## Item 0e, optional sensitivity: `src/tertiary_bpe.py` (unchanged)

This module scores overlap on the model's own BPE tokens: the Jaccard of the token-id set of " alias" with that of each
banned alias. It uses the tokenizer of the checkpoint whose states built the manifold (`unsloth/Qwen3-32B-bnb-4bit`),
read offline from the Hugging Face cache. The sensitivity is committed as optional and is skipped without the
tokenizer.

In [ ]:
%%writefile eval_demo_run/art_iYh9zVP7myjJ/src/tertiary_bpe.py
#!/usr/bin/env python3
"""Optional tertiary lexical score of item 0e (committed as optional in predictions/commit_s0e.json): Qwen3 BPE
token-set Jaccard, using the tokenizer of the very checkpoint whose states built the manifold (unsloth/Qwen3-32B-bnb-4bit),
read offline from the run's shared HF cache. Sensitivity only; the committed verdict uses the primary score.

uv run src/tertiary_bpe.py   (after eval.py has written results/s0e.json)
"""
from __future__ import annotations

import csv
import glob
import json
import os
import sys
from pathlib import Path

import numpy as np
from loguru import logger
from tokenizers import Tokenizer

sys.path.insert(0, str(Path(__file__).resolve().parent))
import alias_geometry as AG  # noqa: E402
import stage_0e as S0E  # noqa: E402

ROOT = Path(__file__).resolve().parent.parent
import recompute as RC  # noqa: E402

TP_SRC = RC.ART[RC.TP] / "src"
CACHE = str(Path(os.environ.get("HF_HUB_CACHE") or Path(os.environ.get("HF_HOME", "")) / "hub")
            / "models--unsloth--Qwen3-32B-bnb-4bit" / "snapshots" / "*" / "tokenizer.json")


@logger.catch(reraise=True)
def main() -> dict | None:
    paths = sorted(glob.glob(CACHE))
    if not paths:
        logger.warning("tokenizer not cached offline: tertiary score not run (as committed)")
        return None
    tok = Tokenizer.from_file(paths[0])
    alias_table = AG.read_literal(TP_SRC / "common.py", "ALIAS_TABLE")
    ids = lambda s: set(tok.encode(" " + s, add_special_tokens=False).ids)  # noqa: E731 - alias as it sits in the prompt
    banned = [ids(s) for b in AG.BANNED for s in alias_table[b]]
    rows = []
    for a in AG.NONBANNED:
        for j, s in enumerate(alias_table[a]):
            t = ids(s)
            jac = max(len(t & b) / len(t | b) for b in banned)
            rows.append({"algorithm": a, "alias_index": j, "alias": s, "bpe_tokens": tok.encode(" " + s, add_special_tokens=False).tokens,
                         "tertiary": jac})
    x3 = np.array([r["tertiary"] for r in rows])
    z = json.loads((ROOT / "results/s0e.json").read_text())
    rng = np.random.default_rng(AG.SEED + 3)
    n_dist = AG.n_distinct_assignments(list(x3))
    null = S0E.exact_null(x3) if n_dist <= 2_000_000 else S0E.mc_null(x3, AG.N_MC_SECONDARY, rng)
    out = {"tokenizer": paths[0].split("hub/")[1], "rows": rows, "n_distinct_arrangements": n_dist,
           "null": "exact" if n_dist <= 2_000_000 else f"monte_carlo_{AG.N_MC_SECONDARY}", "per_k": {}}
    for k in ("k15", "k10"):
        g = np.array([z["per_k"][k]["g_alias"][f"{a}|{j}"] for a in AG.NONBANNED for j in range(3)])
        r = AG.spearman(g, x3)
        st = AG.stratified_exact(g.reshape(5, 3), x3.reshape(5, 3))
        out["per_k"][k] = {"rho_tertiary": r, **S0E.pvals(r, null), "stratified": st}
        logger.info(f"tertiary {k}: rho {r:.3f} p {out['per_k'][k]['p_lower']:.4f}; stratified {st['stat']:.3f} "
                    f"p {st['p_lower']:.3f}")
    (ROOT / "results/s0e_tertiary.json").write_text(json.dumps(out, indent=1, default=float))
    with (ROOT / "tables/alias_tertiary.csv").open("w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["algorithm", "alias_index", "alias", "bpe_tokens", "tertiary_bpe_jaccard", "g_k15", "g_k10"])
        for r in rows:
            key = f"{r['algorithm']}|{r['alias_index']}"
            w.writerow([r["algorithm"], r["alias_index"], r["alias"], " ".join(r["bpe_tokens"]), round(r["tertiary"], 6),
                        round(z["per_k"]["k15"]["g_alias"][key], 6), round(z["per_k"]["k10"]["g_alias"][key], 6)])
    return out


if __name__ == "__main__":
    main()


The original read that tokenizer from the run's shared Hugging Face cache. The next cell fetches the one file into the
standard cache, which is where `tertiary_bpe.py` looks. If `HF_HUB_CACHE` is unset, it is first set to the library's
default location.

In [ ]:
# demo: tertiary_bpe.py globs <HF_HUB_CACHE or HF_HOME/hub>/models--unsloth--Qwen3-32B-bnb-4bit/snapshots/*/tokenizer.json
if not os.environ.get("HF_HUB_CACHE"):
    os.environ["HF_HUB_CACHE"] = str(Path(os.environ.get("HF_HOME", Path.home() / ".cache" / "huggingface")) / "hub")
if FETCH_TOKENIZER:
    try:
        from huggingface_hub import hf_hub_download
        tok_path = hf_hub_download("unsloth/Qwen3-32B-bnb-4bit", "tokenizer.json")
        print("tokenizer cached in HF_HUB_CACHE at", os.path.relpath(tok_path, os.environ["HF_HUB_CACHE"]))
    except Exception as exc:  # noqa: BLE001 - optional input: without it tertiary_bpe.main() skips the sensitivity
        print(f"tokenizer not fetched ({type(exc).__name__}: {exc}); the BPE sensitivity will be skipped")


## Items 0a, 0b, 0c and 0f: `src/recompute.py` (unchanged)

This is the value ledger. `Ledger.add` records a value with its source key path. When the report printed the number
(`recorded`), the recomputed value is formatted the way the report printed it and compared, giving MATCH or CORRECTED.
`get()` navigates key paths such as `validation.C1forced.per_node.ablate[node=mlp|L0|H-1|COND].mean_d`.

- **0a, the circuit.**
  - Each node satisfies the identity mean × 30 = pair mean × 20 + held-out mean × 10.
  - One node has an exactly zero held-out effect, and this module establishes why. The held-out prompts use templates
    F11–F15, none of which any pair prompt uses. So `MeanCache.get` returns no mean, and the held-out edit is never
    built. The raw rows replicate it: the node and all 50 matched random components are exactly 0.
  - It also recomputes the values of Tables 395 and 396 and the committed motif bands.
- **0b, DAS.** The verdicts and the BFS counts at Qwen3-8B's fork: 29 programs written as BFS, 5 correct, McNemar 5:0
  with p 0.0625. `classify_29of30` classifies every "29 of 30" in the report.
- **0c.** Per unit, whether the fork really is the highest-entropy position; and the temperature law against its flat
  and tempering rivals.
- **0f.** Which of Gemma's sampling arms actually ran.

In [ ]:
%%writefile eval_demo_run/art_iYh9zVP7myjJ/src/recompute.py
#!/usr/bin/env python3
"""Recompute every number the corrections quote, from the iteration-8 result files, into a value ledger.

Each ledger entry: name, item, recorded (as printed in the report, or None), recomputed (raw), printed (recomputed,
formatted like the report), status MATCH / CORRECTED / NEW / CARRIED, tolerance, source
'<artifact_id>:<relative path>#<key path>'.
"""
from __future__ import annotations

import csv
import json
import os
import math
import re
import statistics
from functools import lru_cache
from pathlib import Path
from typing import Any

ROOT = Path(__file__).resolve().parents[1]
# The iteration-8 artifacts this repair reads. ONE location setting: S0_DEPS_DIR (default: the run layout,
# <loop>/iter_8/gen_art). Each artifact is found by its id or by its original folder name, under S0_DEPS_DIR or as a
# sibling folder of this one (the layout of the published repository).
DEPS_DIR = Path(os.environ.get("S0_DEPS_DIR", ROOT.parents[2] / "iter_8" / "gen_art"))
FOLDERS = {"art_vnyi0Fm5SyNH": "gen_art_experiment_39", "art_e-_q5mSJGrSb": "gen_art_experiment_40",
           "art_18XvfcJJZi1O": "gen_art_experiment_41", "art_FJMIE9T9_PMv": "gen_art_experiment_42",
           "art_lzuosbrb34wB": "gen_art_experiment_43", "art_R-HSmXq2t97f": "gen_art_experiment_44"}


def dep_dir(art_id: str) -> Path:
    for base in (DEPS_DIR, ROOT.parent):
        for name in (art_id, FOLDERS[art_id]):
            if (base / name).is_dir():
                return base / name
    return DEPS_DIR / FOLDERS[art_id]


ART = {a: dep_dir(a) for a in FOLDERS}
FORK, DAS, CIRC, GEM, LAW, TP = list(ART)


@lru_cache(maxsize=None)
def load(art: str, rel: str) -> Any:
    return json.loads((ART[art] / rel).read_text())


def get(art: str, rel: str, keypath: str) -> Any:
    """Navigate 'a.b[3].c[node=mlp|L0|H-1|COND].d' (dots inside [] are literal)."""
    x = load(art, rel)
    for part in re.findall(r"[^.\[\]]+|\[[^\]]*\]", keypath):
        if part.startswith("["):
            sel = part[1:-1]
            if "=" in sel:
                f, v = sel.split("=", 1)
                x = next(e for e in x if str(e.get(f)) == v)
            else:
                x = x[int(sel)]
        else:
            x = x[part]
    return x


def _half_up(v: float, d: int) -> str:
    """Fixed-point rounding half away from zero, as printed tables round (0.0625 -> 0.063)."""
    from decimal import ROUND_HALF_UP, Decimal
    q = Decimal(1).scaleb(-d)
    return str(Decimal(repr(float(v))).quantize(q, rounding=ROUND_HALF_UP)) if d > 0 else str(
        Decimal(repr(float(v))).quantize(Decimal(1), rounding=ROUND_HALF_UP))


def fmt(v: Any, spec: str) -> str:
    if isinstance(v, (int, float)) and re.fullmatch(r"\.\d+f", spec):
        return _half_up(v, int(spec[1:-1]))
    if spec == "int":
        return str(int(round(v)))
    if spec == "str":
        return str(v)
    if spec.startswith("sig"):           # significant figures, plain or scientific like the report prints p-values
        n = int(spec[3:])
        if v == 0:
            return "0"
        if v == 1:
            return "1"
        if abs(v) < 1e-4:
            return f"{v:.{n - 1}e}"
        return f"{v:.{n}g}"
    if spec.startswith("e"):
        return f"{v:.{int(spec[1:])}e}"
    return format(v, spec)


def like(recorded: str, v: float) -> str:
    """Format v the way `recorded` is printed (decimals or scientific mantissa)."""
    r = recorded.strip().lstrip("+")
    if r in ("1", "0") and isinstance(v, (int, float)):
        return str(int(round(v))) if abs(v - round(v)) < 1e-12 else f"{v:.2g}"
    if "e" in r:
        mant = r.split("e")[0]
        d = len(mant.split(".")[1]) if "." in mant else 0
        return f"{v:.{d}e}"
    if "." in r:
        return _half_up(v, len(r.split(".")[1]))
    return str(int(round(v)))


class Ledger:
    def __init__(self) -> None:
        self.rows: list[dict[str, Any]] = []
        self.by: dict[str, dict[str, Any]] = {}

    def add(self, name: str, item: str, value: Any, source: str, spec: str = ".2f", recorded: str | None = None,
            status: str | None = None, note: str = "") -> str:
        printed = fmt(value, spec) if not isinstance(value, str) else value
        if status is None:
            if recorded is None:
                status = "NEW"
            else:
                cmp = like(recorded, value) if not isinstance(value, str) else value
                status = "MATCH" if cmp == recorded.strip().lstrip("+") else "CORRECTED"
                if status == "MATCH":
                    printed = recorded.strip()
        tol = ("exact" if spec in ("int", "str") or isinstance(value, str) else
               "last printed digit" if recorded is not None else "printed precision")
        row = {"name": name, "item": item, "recorded": recorded, "recomputed": value, "printed": printed,
               "status": status, "tolerance": tol, "source": source, "note": note}
        if name in self.by:
            raise ValueError(f"duplicate ledger name {name}")
        self.rows.append(row)
        self.by[name] = row
        return printed

    def p(self, name: str) -> str:
        return self.by[name]["printed"]


def src(art: str, rel: str, key: str) -> str:
    return f"{art}:{rel}#{key}"


# ================================================================================================ 0a circuit
NODE_NAMES = {"mlp|L0|H-1|COND": "MLP, layer 0, condition span"}


def node_label(k: str) -> str:
    t, L, H, c = k.split("|")
    pos = {"COND": "condition span", "SPEC": "task specification", "OPEN": "opening", "DEC": "fork"}[c]
    if t == "mlp":
        return f"MLP, layer {L[1:]}, {pos}"
    return f"Head {H[1:]} of layer {L[1:]}, {pos}"


def recompute_0a(led: Ledger, out_dir: Path) -> dict[str, Any]:
    rel = "results/summary.json"
    res: dict[str, Any] = {"tables": {}, "identity": [], "zero_heldout": [], "zero_entries": []}
    table_csv = []
    rows_by_contrast = {}
    for c in ("C1forced", "C2a", "C1nat"):
        V = get(CIRC, rel, f"validation.{c}")
        n_h = V["n_heldout"]
        vrows = [json.loads(l) for l in (ART[CIRC] / f"data/validation_rows_{c}.jsonl").read_text().splitlines() if l]
        rows_by_contrast[c] = vrows
        held_mask = [bool(r["heldout"]) for r in vrows]
        for mode in ("ablate", "steer"):
            for i, rec in enumerate(V["per_node"][mode]):
                k = rec["node"]
                n = rec["n"]
                lhs = rec["mean_d"] * n
                rhs = rec["mean_d_pair_prompts"] * (n - n_h) + rec["mean_d_heldout"] * n_h
                ok = abs(lhs - rhs) <= 1e-6 * max(1.0, abs(lhs))
                res["identity"].append({"contrast": c, "mode": mode, "node": k, "lhs": lhs, "rhs": rhs, "pass": ok})
                d = rec["d_per_prompt"]
                held_d = [x for x, h in zip(d, held_mask) if h] if len(d) == len(held_mask) else None
                n_zero = sum(1 for x in (held_d or []) if x == 0.0)
                if rec["mean_d_heldout"] == 0.0:
                    res["zero_heldout"].append({"contrast": c, "mode": mode, "node": k})
                if n_zero:
                    res["zero_entries"].append({"contrast": c, "mode": mode, "node": k, "n_zero_heldout_entries": n_zero})
                table_csv.append({"contrast": c, "mode": mode, "rank": i + 1, "node": k, "label": node_label(k),
                                  "n": n, "n_heldout": n_h, "mean_d": rec["mean_d"],
                                  "mean_d_pair_prompts": rec["mean_d_pair_prompts"],
                                  "mean_d_heldout": rec["mean_d_heldout"],
                                  "p_sign_flip_heldout": rec["p_sign_flip_heldout"], "p_holm_20": rec["p_holm_20"],
                                  "identity_pass": ok, "n_exact_zero_heldout_entries": n_zero})
    with (out_dir / "tables/circuit_heldout.csv").open("w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(table_csv[0].keys()))
        w.writeheader()
        w.writerows(table_csv)
    res["table_rows"] = table_csv
    res["identity_check_pass"] = all(r["pass"] for r in res["identity"])
    res["n_exact_zero_heldout_nodes"] = len({(z["contrast"], z["node"]) for z in res["zero_heldout"]})
    # cause: held-out prompts of C1forced share no template with any pair prompt
    vr = rows_by_contrast["C1forced"]
    pair_t = [r["template"] for r in vr if not r["heldout"]]
    held = [r for r in vr if r["heldout"]]
    matches = [sum(1 for t in pair_t if t == r["template"]) for r in held]
    res["cond_span_matches_heldout"] = {"per_heldout_prompt": dict(zip([r["vid"] for r in held], matches)),
                                        "n_heldout_with_any_match": sum(1 for m in matches if m > 0),
                                        "n_heldout": len(held), "pair_templates": sorted(set(pair_t)),
                                        "heldout_templates": sorted({r["template"] for r in held})}
    # replication from stored raw margins: node AND every matched random component are exactly 0 on held-out prompts
    key = "mlp|L0|H-1|COND"
    zero_all = []
    for r in held:
        for mode in ("ablate", "steer"):
            vals = [v for kk, v in r["res"].items() if kk.startswith(f"{mode}|{key}|")]
            zero_all.append(all(v == 0.0 for v in vals) and len(vals) == 51)
    res["replication"] = {"heldout_rows_all_node_and_random_entries_zero": sum(zero_all), "of": len(zero_all)}
    res["cause_evidence"] = "replicated" if all(zero_all) and res["cond_span_matches_heldout"][
        "n_heldout_with_any_match"] == 0 else "code-and-pattern"
    code = (ART[CIRC] / "src/k3circuit.py").read_text().splitlines()
    res["code_quotes"] = {str(n): code[n - 1].strip() for n in (482, 485, 488)}
    # ledger: the layer-0 node and the natural contrast
    base = "validation.C1forced.per_node"
    for mode, tag in (("ablate", "abl"), ("steer", "ste")):
        r0 = get(CIRC, rel, f"{base}.{mode}[node={key}]")
        led.add(f"0a.L0.{tag}.mean_d", "0a", r0["mean_d"], src(CIRC, rel, f"{base}.{mode}[node={key}].mean_d"), ".2f",
                recorded={"abl": "5.43", "ste": "4.94"}[tag])
        led.add(f"0a.L0.{tag}.pair", "0a", r0["mean_d_pair_prompts"],
                src(CIRC, rel, f"{base}.{mode}[node={key}].mean_d_pair_prompts"), ".2f")
        led.add(f"0a.L0.{tag}.heldout", "0a", r0["mean_d_heldout"],
                src(CIRC, rel, f"{base}.{mode}[node={key}].mean_d_heldout"), ".2f")
        rn = get(CIRC, rel, f"validation.C1nat.per_node.{mode}[node={key}]")
        led.add(f"0a.nat.{tag}.heldout", "0a", rn["mean_d_heldout"],
                src(CIRC, rel, f"validation.C1nat.per_node.{mode}[node={key}].mean_d_heldout"), ".2f")
        led.add(f"0a.nat.{tag}.p_heldout", "0a", rn["p_sign_flip_heldout"],
                src(CIRC, rel, f"validation.C1nat.per_node.{mode}[node={key}].p_sign_flip_heldout"), ".3f")
    led.add("0a.nat.n_heldout", "0a", get(CIRC, rel, "validation.C1nat.n_heldout"),
            src(CIRC, rel, "validation.C1nat.n_heldout"), "int")
    led.add("0a.C1forced.n_heldout", "0a", get(CIRC, rel, "validation.C1forced.n_heldout"),
            src(CIRC, rel, "validation.C1forced.n_heldout"), "int", recorded="10")
    led.add("0a.C1forced.n_pair", "0a", get(CIRC, rel, "validation.C1forced.n_prompts") - get(
        CIRC, rel, "validation.C1forced.n_heldout"), src(CIRC, rel, "validation.C1forced.n_prompts-n_heldout"), "int")
    led.add("0a.cond_matches", "0a", res["cond_span_matches_heldout"]["n_heldout_with_any_match"],
            f"{CIRC}:data/validation_rows_C1forced.jsonl#template (pair vs heldout)", "int")
    led.add("0a.n_zero_nodes", "0a", res["n_exact_zero_heldout_nodes"], src(CIRC, rel, "validation.*.per_node"), "int")
    # Tables 395/396 recorded values (MATCH checks) and new held-out columns
    rec395 = {"mlp|L0|H-1|COND": ("5.429", "3.8e-06", "4.938", "3.8e-06"), "head|L33|H14|DEC": ("2.050", "1.9e-08", "4.251", "1.9e-08"),
              "mlp|L35|H-1|DEC": ("-2.638", "1", "-5.068", "1"), "head|L34|H28|DEC": ("1.964", "1.9e-08", "3.824", "1.9e-08"),
              "mlp|L27|H-1|DEC": ("0.777", "1.9e-08", "1.896", "1.9e-08"), "head|L29|H6|DEC": ("1.403", "1.9e-08", "2.827", "1.9e-08"),
              "mlp|L30|H-1|DEC": ("0.716", "1.9e-08", "1.529", "1.9e-08"), "head|L23|H10|OPEN": ("0.783", "1.9e-08", "1.531", "1.9e-08"),
              "mlp|L28|H-1|DEC": ("0.414", "3.4e-07", "0.904", "1.9e-08"), "mlp|L11|H-1|SPEC": ("1.162", "1.9e-08", "1.288", "1.9e-08")}
    rec396 = {"mlp|L27|H-1|DEC": ("9.596", "1.8e-11", "16.058", "1.8e-11"), "mlp|L26|H-1|DEC": ("5.088", "1.8e-11", "10.606", "1.8e-11"),
              "mlp|L32|H-1|DEC": ("1.924", "3.7e-10", "7.967", "1.8e-11"), "mlp|L30|H-1|DEC": ("0.738", "0.0015", "5.851", "1.8e-11"),
              "head|L23|H10|DEC": ("3.627", "1.8e-11", "7.964", "1.8e-11"), "head|L27|H31|DEC": ("3.047", "1.8e-11", "6.869", "1.8e-11"),
              "mlp|L28|H-1|DEC": ("0.485", "0.0077", "1.685", "3.4e-10"), "mlp|L31|H-1|DEC": ("-1.527", "1", "0.684", "0.0079"),
              "mlp|L13|H-1|SPEC": ("1.712", "2.4e-06", "2.847", "1.3e-07"), "head|L11|H13|SPEC": ("-1.802", "1", "-3.351", "1")}
    for c, recs, tab in (("C1forced", rec395, "T395"), ("C2a", rec396, "T396")):
        for k, (ae, ap, se, sp) in recs.items():
            pa = f"validation.{c}.per_node.ablate[node={k}]"
            ps = f"validation.{c}.per_node.steer[node={k}]"
            ra, rs = get(CIRC, rel, pa), get(CIRC, rel, ps)
            led.add(f"0a.{tab}.{k}.abl", "0a", ra["mean_d"], src(CIRC, rel, pa + ".mean_d"), ".3f", recorded=ae)
            led.add(f"0a.{tab}.{k}.abl_p", "0a", ra["p_holm_20"], src(CIRC, rel, pa + ".p_holm_20"), "sig2", recorded=ap)
            led.add(f"0a.{tab}.{k}.ste", "0a", rs["mean_d"], src(CIRC, rel, ps + ".mean_d"), ".3f", recorded=se)
            led.add(f"0a.{tab}.{k}.ste_p", "0a", rs["p_holm_20"], src(CIRC, rel, ps + ".p_holm_20"), "sig2", recorded=sp)
            led.add(f"0a.{tab}.{k}.abl_h", "0a", ra["mean_d_heldout"], src(CIRC, rel, pa + ".mean_d_heldout"), ".2f")
            led.add(f"0a.{tab}.{k}.ste_h", "0a", rs["mean_d_heldout"], src(CIRC, rel, ps + ".mean_d_heldout"), ".2f")
            led.add(f"0a.{tab}.{k}.abl_hp", "0a", ra["p_sign_flip_heldout"],
                    src(CIRC, rel, pa + ".p_sign_flip_heldout"), "sig2")
            led.add(f"0a.{tab}.{k}.ste_hp", "0a", rs["p_sign_flip_heldout"],
                    src(CIRC, rel, ps + ".p_sign_flip_heldout"), "sig2")
    led.add("0a.C2a.n_prompts", "0a", get(CIRC, rel, "validation.C2a.n_prompts"), src(CIRC, rel, "validation.C2a.n_prompts"),
            "int", recorded="40")
    led.add("0a.C2a.n_heldout", "0a", get(CIRC, rel, "validation.C2a.n_heldout"), src(CIRC, rel, "validation.C2a.n_heldout"),
            "int", recorded="10")
    res["C2a_top10_has_COND"] = any(k.endswith("|COND") for k in rec396)
    res["rec395"], res["rec396"] = rec395, rec396
    # motif
    pc = "motif.primary_comparison"
    led.add("0a.motif.label", "0a", get(CIRC, rel, pc + ".label"), src(CIRC, rel, pc + ".label"), "str",
            recorded="DIFFERENT MOTIF")
    led.add("0a.motif.qwen_cond", "0a", get(CIRC, rel, pc + ".qwen.COND"), src(CIRC, rel, pc + ".qwen.COND"), ".2f",
            recorded="0.01")
    led.add("0a.motif.qwen_depth", "0a", get(CIRC, rel, pc + ".qwen_dec_depth"), src(CIRC, rel, pc + ".qwen_dec_depth"),
            ".2f", recorded="0.86")
    led.add("0a.motif.gemma_depth", "0a", get(CIRC, rel, pc + ".gemma_dec_depth"), src(CIRC, rel, pc + ".gemma_dec_depth"),
            ".2f", recorded="0.69")
    led.add("0a.motif.gemma_cond", "0a", get(CIRC, rel, pc + ".gemma.COND"), src(CIRC, rel, pc + ".gemma.COND"), ".2f",
            recorded="0.36")
    led.add("0a.motif.gemma_label", "0a", get(CIRC, rel, "motif.gemma.label_vs_bands.label"),
            src(CIRC, rel, "motif.gemma.label_vs_bands.label"), "str")
    led.add("0a.motif.sweep_cond", "0a", get(CIRC, rel, "motif.qwen.C1forced.extended_top60_by_exact.COND"),
            src(CIRC, rel, "motif.qwen.C1forced.extended_top60_by_exact.COND"), ".2f", recorded="0.17")
    depths = []
    for c in ("C1nat", "C1forced", "C1mix", "C1natB"):
        q = get(CIRC, rel, f"motif.qwen.{c}")
        for kk in ("dec_stack", "extended_top60_dec_stack"):
            if kk in q and q[kk] and q[kk].get("depth") is not None:
                depths.append((c, kk, q[kk]["depth"]))
    res["mst_depths"] = depths
    led.add("0a.motif.mst_depth_min", "0a", min(d for *_, d in depths), src(CIRC, rel, "motif.qwen.{C1nat,C1forced,C1mix,"
            "C1natB}.{dec_stack,extended_top60_dec_stack}.depth (min)"), ".2f")
    led.add("0a.motif.mst_depth_max", "0a", max(d for *_, d in depths), src(CIRC, rel, "motif.qwen.{C1nat,C1forced,C1mix,"
            "C1natB}.{dec_stack,extended_top60_dec_stack}.depth (max)"), ".2f")
    bands_line = [(i + 1, l) for i, l in enumerate((ART[CIRC] / "src/k3motif.py").read_text().splitlines())
                  if l.startswith("BANDS =")][0]
    bands = eval(bands_line[1].split("=", 1)[1], {"__builtins__": {}})  # literal dict of tuples, from the source line
    res["bands"] = {"line": bands_line[0], "text": bands_line[1], "values": bands}
    for kk, nm in (("COND", "cond"), ("depth", "depth")):
        lo, hi = bands[kk]
        led.add(f"0a.band.{nm}.lo", "0a", lo, f"{CIRC}:src/k3motif.py#L{bands_line[0]} BANDS['{kk}'][0]", ".2f")
        led.add(f"0a.band.{nm}.hi", "0a", hi, f"{CIRC}:src/k3motif.py#L{bands_line[0]} BANDS['{kk}'][1]", ".2f")
    return res


# ================================================================================================ 0b DAS
def recompute_0b(led: Ledger) -> dict[str, Any]:
    rel = "results/analysis.json"
    b = "gen_T10_dec.classes.bfs"
    d = "gen_T10_dec.classes.dfs"
    g = "gen_GRID_dec.classes.bfs"
    led.add("0b.T10.verdict", "0b", get(DAS, rel, "verdicts.T10-dec.verdict"), src(DAS, rel, "verdicts.T10-dec.verdict"),
            "str", recorded="NO HANDLE")
    led.add("0b.GRID.verdict", "0b", get(DAS, rel, "verdicts.GRID-dec.verdict"), src(DAS, rel, "verdicts.GRID-dec.verdict"),
            "str", recorded="SUFFICIENT HANDLE")
    led.add("0b.band.verdict", "0b", get(DAS, rel, "verdicts.T10-band.verdict"), src(DAS, rel, "verdicts.T10-band.verdict"),
            "str", recorded="SUFFICIENT HANDLE")
    led.add("0b.bfs.written", "0b", get(DAS, rel, f"{b}.arms.DAS.attempted_target"),
            src(DAS, rel, f"{b}.arms.DAS.attempted_target"), "int", recorded="29")
    led.add("0b.bfs.correct", "0b", get(DAS, rel, f"{b}.arms.DAS.correct_target"),
            src(DAS, rel, f"{b}.arms.DAS.correct_target"), "int", recorded="5")
    led.add("0b.bfs.fail", "0b", get(DAS, rel, f"{b}.arms.DAS.attempted_target") - get(DAS, rel, f"{b}.arms.DAS.correct_target"),
            src(DAS, rel, f"{b}.arms.DAS.attempted_target - correct_target"), "int", recorded="24")
    led.add("0b.bfs.pass", "0b", get(DAS, rel, f"{b}.arms.DAS.pass_rate"), src(DAS, rel, f"{b}.arms.DAS.pass_rate"), ".2f",
            recorded="0.20")
    led.add("0b.bfs.base_pass", "0b", get(DAS, rel, f"{b}.arms.BASE.pass_rate"), src(DAS, rel, f"{b}.arms.BASE.pass_rate"),
            ".2f", recorded="0.83")
    led.add("0b.bfs.pass_ratio", "0b", get(DAS, rel, "verdicts.T10-dec.V0_by_class.bfs.pass_ratio"),
            src(DAS, rel, "verdicts.T10-dec.V0_by_class.bfs.pass_ratio"), ".2f")
    led.add("0b.V0", "0b", str(get(DAS, rel, "verdicts.T10-dec.V0")), src(DAS, rel, "verdicts.T10-dec.V0"), "str")
    mc = get(DAS, rel, f"{b}.mcnemar_DAS_vs_RAND")
    led.add("0b.bfs.mc_a", "0b", mc["a_only"], src(DAS, rel, f"{b}.mcnemar_DAS_vs_RAND.a_only"), "int", recorded="5")
    led.add("0b.bfs.mc_b", "0b", mc["b_only"], src(DAS, rel, f"{b}.mcnemar_DAS_vs_RAND.b_only"), "int", recorded="0")
    exact = 2 * 0.5 ** 5
    led.add("0b.bfs.mc_p", "0b", mc["p_two_sided"], src(DAS, rel, f"{b}.mcnemar_DAS_vs_RAND.p_two_sided"), ".3f",
            recorded="0.063", note=f"exact two-sided McNemar 2*(1/2)^5 = {exact}")
    led.add("0b.dfs.correct", "0b", get(DAS, rel, f"{d}.arms.DAS.correct_target"),
            src(DAS, rel, f"{d}.arms.DAS.correct_target"), "int", recorded="29")
    led.add("0b.dfs.full", "0b", get(DAS, rel, f"{d}.arms.FULL.correct_target"),
            src(DAS, rel, f"{d}.arms.FULL.correct_target"), "int", recorded="2")
    led.add("0b.dfs.random", "0b", get(DAS, rel, f"{d}.arms.RAND.correct_target"),
            src(DAS, rel, f"{d}.arms.RAND.correct_target"), "int", recorded="0")
    led.add("0b.grid.das", "0b", get(DAS, rel, f"{g}.arms.DAS.correct_target"), src(DAS, rel, f"{g}.arms.DAS.correct_target"),
            "int", recorded="22")
    led.add("0b.grid.base", "0b", get(DAS, rel, f"{g}.arms.BASE.correct_target"),
            src(DAS, rel, f"{g}.arms.BASE.correct_target"), "int", recorded="6")
    led.add("0b.grid.random", "0b", get(DAS, rel, f"{g}.arms.RAND.correct_target"),
            src(DAS, rel, f"{g}.arms.RAND.correct_target"), "int", recorded="5")
    led.add("0b.p_K2", "0b", get(DAS, rel, "K2.p_K2"), src(DAS, rel, "K2.p_K2"), ".3f", recorded="0.037")
    return {"mcnemar_exact_two_sided": exact, "pooled_rule_V0_parts": get(DAS, rel, "verdicts.T10-dec.V0_parts")}


def classify_29of30(text: str) -> list[dict[str, Any]]:
    """Every '29 of 30' / '29/30' occurrence in the report, classified by the nearest preceding keyword in its line
    (and the few words after it). Lines before the iteration-8 heading are marked pre-iteration-8."""
    kws = [("counting sort", "counting sort"), ("toward bfs", "DAS-fork-BFS"), ("toward dfs", "DAS-fork-DFS"),
           ("top five", "circuit top-five"), ("circuit", "circuit top-five"), ("full-state", "full-state"),
           ("full state", "full-state"), ("das", "DAS"), ("rank-4", "DAS"), ("block 22", "DAS"), ("rank 2:", "DAS"),
           ("rank 1:", "DAS")]
    lines = text.splitlines()
    it8 = next((i + 1 for i, l in enumerate(lines) if l.startswith("# Iteration 8:")), 10 ** 9)
    out = []
    for i, l in enumerate(lines):
        for m in re.finditer(r"29 of 30|29/30", l):
            pre = l[max(0, m.start() - 160): m.start()].lower()
            post = l[m.end(): m.end() + 40].lower()
            if "counting sort" in post or "counting sort" in pre[-80:]:
                cls = "counting sort"
            elif re.match(r"\s*(written|write bfs|programs, but)", post) or "toward bfs" in pre[-120:] and \
                    "toward dfs" not in pre[-120:]:
                cls = "DAS-fork-BFS"
            else:
                best, cls = -1, "other"
                for kw, c in kws:
                    pos = pre.rfind(kw)
                    if pos > best:
                        best, cls = pos, c
                if cls == "DAS":
                    cls = "DAS-fork-DFS" if "dfs" in pre + post else "DAS-other"
            if i + 1 < it8:
                cls = f"pre-iteration-8 {cls}"
            out.append({"line": i + 1, "class": cls, "context": l[max(0, m.start() - 140): m.end() + 80]})
    return out


# ================================================================================================ 0c entropy and law
UNIT_NAMES = {"T10": "connected components", "B3": "minimum spanning tree", "A3": "trapped rain water",
              "B6": "valid anagram", "G": "Gemma grid path"}


def recompute_0c(led: Ledger, out_dir: Path) -> dict[str, Any]:
    rel = "results/k1_verdict.json"
    dose = "predictions/commit1_dose_table.json"
    rec_hits = {"T10": ("2", "50"), "B3": ("32", "39"), "A3": ("0", "37"), "B6": ("21", "39"), "G": ("44", "44")}
    med = {}
    with (ART[FORK] / "data/samples.jsonl").open() as f:
        for line in f:
            r = json.loads(line)
            if r.get("arm") == "A0" and r.get("stage") == "S0" and r.get("reached_decision") and r.get("decision_H1") is not None:
                med.setdefault(r["unit"], []).append(r["decision_H1"])
    rows = []
    for u in ("G", "B3", "B6", "T10", "A3"):
        k = get(FORK, rel, f"K1c.{u}")
        led.add(f"0c.{u}.hits", "0c", k["A7_hits_decision"], src(FORK, rel, f"K1c.{u}.A7_hits_decision"), "int",
                recorded=rec_hits[u][0])
        led.add(f"0c.{u}.n", "0c", k["n"], src(FORK, rel, f"K1c.{u}.n"), "int", recorded=rec_hits[u][1])
        led.add(f"0c.{u}.above", "0c", 1 - k["twin_dec_H_below_q90_frac"],
                src(FORK, rel, f"K1c.{u}.1-twin_dec_H_below_q90_frac"), ".2f")
        q = get(FORK, dose, f"units.{u}.q90")
        led.add(f"0c.{u}.q90", "0c", q, src(FORK, dose, f"units.{u}.q90"), "sig2")
        m = statistics.median(med[u])
        led.add(f"0c.{u}.median_H", "0c", m, f"{FORK}:data/samples.jsonl#arm=A0,stage=S0,reached_decision -> "
                f"median(decision_H1) [{u}]", ".2f")
        rows.append({"unit": u, "task": UNIT_NAMES[u], "n_twin_reaching_fork": k["n"],
                     "share_above_q90": 1 - k["twin_dec_H_below_q90_frac"], "q90_nats": q,
                     "argmax_entropy_hits_first64": k["A7_hits_decision"], "median_fork_entropy_nats": m,
                     "n_samples_with_entropy": len(med[u])})
    led.add("0c.n_total", "0c", sum(r["n_twin_reaching_fork"] for r in rows), src(FORK, rel, "K1c.*.n (sum)"), "int")
    q90s = [r["q90_nats"] for r in rows]
    led.add("0c.q90.min", "0c", min(q90s), src(FORK, dose, "units.*.q90 (min)"), ".4f", recorded="0.0003")
    led.add("0c.q90.max", "0c", max(q90s), src(FORK, dose, "units.*.q90 (max)"), ".2f", recorded="0.05")
    led.add("0c.T10.median_H_rec", "0c", med_v := statistics.median(med["T10"]),
            f"{FORK}:data/samples.jsonl#median decision_H1 [T10]", ".2f", recorded="0.13")
    mixed = [statistics.median(med[u]) for u in ("B3", "A3", "B6", "G")]
    led.add("0c.mixed_median.min", "0c", min(mixed), f"{FORK}:data/samples.jsonl#median decision_H1 [B3,A3,B6,G] (min)",
            ".2f", recorded="0.53")
    led.add("0c.mixed_median.max", "0c", max(mixed), f"{FORK}:data/samples.jsonl#median decision_H1 [B3,A3,B6,G] (max)",
            ".2f", recorded="0.81")
    # temperature law
    relL = "results/analysis_L.json"
    rec_mae = {"A3": "0.029", "B6": "0.054", "B3": "0.043", "G-T04": "0.033", "G-R6": "0.014", "L-T1": "0.142"}
    lrows = []
    for u, r in rec_mae.items():
        pu = get(LAW, relL, f"verdict.per_unit.{u}")
        led.add(f"0c.law.{u}.mae", "0c", pu["mae_L"], src(LAW, relL, f"verdict.per_unit.{u}.mae_L"), ".3f", recorded=r)
        led.add(f"0c.law.{u}.H", "0c", pu["H"], src(LAW, relL, f"verdict.per_unit.{u}.H"), ".3f" if pu["H"] > 0.01 else "sig1")
        mae = get(LAW, relL, f"units.{u}.mae.p1")
        lrows.append({"unit": u, "law": pu["mae_L"], "flat_B0": mae["B0"], "tempering_B1": mae["B1"],
                      "single_fork_B2": mae["B2"], "entropy_BH": mae["BH"], "H_bits": pu["H"], "pass": pu["pass"]})
    hmax = max(get(LAW, relL, f"verdict.per_unit.{u}.H") for u in ("A3", "B6", "B3", "G-T04", "G-R6"))
    led.add("0c.law.Hmax5", "0c", hmax, src(LAW, relL, "verdict.per_unit.{A3,B6,B3,G-T04,G-R6}.H (max)"), ".3f",
            recorded="0.004")
    led.add("0c.law.H_llama", "0c", get(LAW, relL, "verdict.per_unit.L-T1.H"), src(LAW, relL, "verdict.per_unit.L-T1.H"),
            ".2f", recorded="0.63")
    led.add("0c.law.B6.flat", "0c", get(LAW, relL, "units.B6.mae.p1.B0"), src(LAW, relL, "units.B6.mae.p1.B0"), ".3f",
            recorded="0.030")
    led.add("0c.law.GT04.temper", "0c", get(LAW, relL, "units.G-T04.mae.p1.B1"), src(LAW, relL, "units.G-T04.mae.p1.B1"),
            ".3f", recorded="0.023")
    led.add("0c.law.beats_flat", "0c", get(LAW, relL, "verdict.L_vs_baselines.B0.n_L_better"),
            src(LAW, relL, "verdict.L_vs_baselines.B0.n_L_better"), "int", recorded="4")
    led.add("0c.law.sign_p", "0c", get(LAW, relL, "verdict.holm_input_p_L_sign_test_vs_B0"),
            src(LAW, relL, "verdict.holm_input_p_L_sign_test_vs_B0"), ".2f", recorded="0.34")
    led.add("0c.law.n_pass", "0c", get(LAW, relL, "verdict.n_pass"), src(LAW, relL, "verdict.n_pass"), "int", recorded="5")
    led.add("0c.law.n_scored", "0c", get(LAW, relL, "verdict.n_scored"), src(LAW, relL, "verdict.n_scored"), "int",
            recorded="6")
    with (out_dir / "tables/entropy_scope.csv").open("w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        w.writeheader()
        w.writerows(rows)
    with (out_dir / "tables/law_scope.csv").open("w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(lrows[0].keys()))
        w.writeheader()
        w.writerows(lrows)
    return {"entropy_rows": rows, "law_rows": lrows}


# ================================================================================================ 0f Gemma arms
ARM_NAMES = {"A0": "twin T 1", "A1": "global T 1.3, top-p 0.95", "A2": "global T 1.6, top-p 0.95",
             "A3g": "global T 2.0, top-p 0.95", "A4": "min-p 0.1, T 3.0", "A5": "entropy-triggered (EDT-style) T 2",
             "A6": "block-start (AdapT-style) T 2", "A7": "high dose at highest-entropy position (placebo)",
             "A8": "high dose at random position (placebo)", "A9": "fork dose, low", "A10": "fork dose, high",
             "A11": "class-uniform forcing", "A9f": "filtered fork dose, low (amendment)",
             "A10f": "filtered fork dose, high (amendment)"}
FAMILIES = {"global_T": ["A1", "A2", "A3g"], "min_p": ["A4"], "EDT": ["A5"], "AdapT": ["A6"]}


def recompute_0f(led: Ledger, out_dir: Path) -> dict[str, Any]:
    cells = load(FORK, "results/k1_cells.json")
    am = load(FORK, "results/amendment.json")
    code = (ART[FORK] / "src/analysis_k1.py").read_text()
    fam_line = [i + 1 for i, l in enumerate(code.splitlines()) if l.startswith("COMP_FAMILIES")][0]
    doc_line = [i + 1 for i, l in enumerate(code.splitlines()) if "twin Hill if none" in l][0]
    presence = {u: sorted(cells[u].keys()) for u in ("T10", "B3", "A3", "B6", "G")}
    rows = []
    for u, arms in presence.items():
        for a in sorted(set(ARM_NAMES) - {"A9f", "A10f"} | set(arms)):
            c = cells[u].get(a) or (am["units"].get(u, {}).get("cells", {}).get(a) if u in am["units"] else None)
            rows.append({"unit": u, "arm": a, "arm_name": ARM_NAMES.get(a, a), "present": a in arms or c is not None,
                         "pass_rate": None if c is None else c.get("pass_rate"),
                         "pass_cost": None if c is None else c.get("pass_cost"),
                         "hill1_correct": None if c is None else c.get("hill1_correct_r25")})
    with (out_dir / "tables/gemma_arms.csv").open("w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        w.writeheader()
        w.writerows(rows)
    g_present = set(presence["G"])
    fam_status = {}
    bv = am["units"]["G"]["bootstrap_vs"]
    for fam, arms in FAMILIES.items():
        run = [a for a in arms if a in g_present]
        within = [a for a in run if (cells["G"][a].get("pass_cost") or 0) <= 0.05]
        status = "run arm" if within else "twin-level fallback"
        why = ("not run on Gemma" if not run else "run, but pass cost above 5 points on the full slot set" if not within
               else "run within the 5-point pass-cost rule")
        fam_status[fam] = {"arms_present": run, "arms_within_cost": within, "status": status, "why": why,
                           "mean_diff": bv[fam]["mean_diff"], "p_one_sided": bv[fam]["p_one_sided"],
                           "lb_97_5": bv[fam]["lb_97_5"]}
    rel = "results/amendment.json"
    led.add("0f.G.globalT.diff", "0f", bv["global_T"]["mean_diff"], src(FORK, rel, "units.G.bootstrap_vs.global_T.mean_diff"),
            ".2f", recorded="0.08")
    led.add("0f.G.globalT.p", "0f", bv["global_T"]["p_one_sided"], src(FORK, rel, "units.G.bootstrap_vs.global_T.p_one_sided"),
            ".2f", recorded="0.23")
    led.add("0f.G.EDT.p", "0f", bv["EDT"]["p_one_sided"], src(FORK, rel, "units.G.bootstrap_vs.EDT.p_one_sided"), ".3f",
            recorded="0.023")
    led.add("0f.G.minp.p", "0f", bv["min_p"]["p_one_sided"], src(FORK, rel, "units.G.bootstrap_vs.min_p.p_one_sided"),
            ".3f", recorded="0.016")
    led.add("0f.G.adapt.p", "0f", bv["AdapT"]["p_one_sided"], src(FORK, rel, "units.G.bootstrap_vs.AdapT.p_one_sided"),
            ".3f")
    led.add("0f.G.A5.pass_cost", "0f", cells["G"]["A5"]["pass_cost"], src(FORK, "results/k1_cells.json", "G.A5.pass_cost"),
            ".2f", recorded="0.18")
    led.add("0f.G.A3g.pass_cost", "0f", cells["G"]["A3g"]["pass_cost"], src(FORK, "results/k1_cells.json", "G.A3g.pass_cost"),
            ".2f", recorded="0.08")
    led.add("0f.G.twin_hill", "0f", am["units"]["G"]["point_best_hill"]["EDT"],
            src(FORK, rel, "units.G.point_best_hill.EDT (= twin Hill)"), ".2f", recorded="1.64")
    led.add("0f.G.filtered_hill", "0f", am["units"]["G"]["point_best_hill"]["filtered"],
            src(FORK, rel, "units.G.point_best_hill.filtered"), ".2f", recorded="1.98")
    return {"presence": presence, "family_status_G": fam_status, "comp_families_line": fam_line,
            "fallback_docstring_line": doc_line,
            "missing": {u: [a for a in ("A1", "A2", "A3g", "A4", "A5", "A6") if a not in presence[u]] for u in presence}}


## Items 0d and 0g, and the 0e text: `src/s0_extra.py` (unchanged)

- **0d, positioning.** Adapters read the six iteration-8 `positioning.json` schemas. The live checks grep each
  neighbour's raw page text (`audit/web/`, fetched on 2026-10-23) for content, venue and a quote, giving VERIFIED,
  VERIFIED-ABSTRACT or NOT-FOUND. The output is section 8.10b, with Table 416b and references [142]–[146].
- **0g, figure markers.** Each marker gets a six-field caption (positions, block, projection, N, centroids, caveat) and
  the sha256 of each figure file. The iteration-8 images are not shipped, so their hashes print as "missing" here.
- **The 0e text.** The report paragraph is built from `results/s0e.json` through the ledger, so every number it prints
  is a ledger value.

In [ ]:
%%writefile eval_demo_run/art_iYh9zVP7myjJ/src/s0_extra.py
#!/usr/bin/env python3
"""Builders for item 0d (positioning table from six positioning.json adapters plus live fetch_grep checks), item 0g
(figure markers with six-field captions and sha256) and the 0e report text (from results/s0e.json)."""
from __future__ import annotations

import csv
import hashlib
import json
import re
from pathlib import Path
from typing import Any

from recompute import ART, CIRC, DAS, FORK, GEM, LAW, TP, Ledger, load

M = "(iteration-9 correction)"


def sha(p: Path) -> str:
    h = hashlib.sha256()
    with p.open("rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


# ================================================================================================ 0d positioning
def adapt_positioning() -> list[dict[str, Any]]:
    """One adapter per artifact schema -> rows {artifact, claim, neighbour, verdict, quote, url, source}."""
    rows = []
    a = load(FORK, "results/positioning.json")
    for p in a["papers"]:
        rows.append({"artifact": FORK, "claim": "fork dosing", "neighbour": p["title"], "verdict": p["verdict"],
                     "quote": p.get("quote", ""), "url": p.get("id", ""), "source": f"{FORK}:results/positioning.json#papers"})
    a = load(DAS, "results/positioning.json")
    for q in a["queries"]:
        rows.append({"artifact": DAS, "claim": "matched DAS", "neighbour": q["source"], "verdict": q["verdict"],
                     "quote": q.get("quote", ""), "url": q["source"], "source": f"{DAS}:results/positioning.json#queries"})
    a = load(CIRC, "results/positioning.json")
    for w in a["works"]:
        rows.append({"artifact": CIRC, "claim": "circuit", "neighbour": w["work"], "verdict": w["verdict"],
                     "quote": w.get("quote", ""), "url": "", "source": f"{CIRC}:results/positioning.json#works"})
    a = load(GEM, "results/positioning.json")
    for e in a["entries"]:
        qs = e.get("quotes") or []
        rows.append({"artifact": GEM, "claim": "Gemma dial", "neighbour": e["title"], "verdict": e["verdict"],
                     "quote": (qs[0] if qs and isinstance(qs[0], str) else json.dumps(qs[:1]))[:400], "url": e.get("id", ""),
                     "source": f"{GEM}:results/positioning.json#entries"})
    a = load(LAW, "results/positioning.json")
    for s in a["sources"]:
        claim = "base sub-arm" if re.search(r"base sub-arm", s["verdict"], re.I) else "temperature law"
        rows.append({"artifact": LAW, "claim": claim, "neighbour": s["work"], "verdict": s["verdict"],
                     "quote": s.get("quote", ""), "url": s.get("url", ""), "source": f"{LAW}:results/positioning.json#sources"})
    a = load(TP, "results/positioning.json")
    for it in a["items"]:
        rows.append({"artifact": TP, "claim": "top-five prohibition", "neighbour": it["title"], "verdict": it["verdict"],
                     "quote": it.get("quote", ""), "url": it.get("id", ""), "source": f"{TP}:results/positioning.json#items"})
    return rows


NEIGHBOURS = [
    {"key": "cantarero", "ref": 142, "title": "The RLVR Policy Shift Is Sparse and Predictable from the Base Model",
     "authors": "Cantarero, Garcia, Marín, Tiene, Orús", "files": ["cantarero_172378", "cantarero_164442"],
     "url": "https://neurips.cc/virtual/2026/172378 ; https://neurips.cc/virtual/2026/164442",
     "venue_patterns": [r"Poster in Workshop"], "content_patterns": [r"18\.4--57\.1", r"1\.06--1\.70", r"32\.8", r"4\.7"],
     "quote_rx": r"per-position Jensen--Shannon divergence between the two policies is 18\.4--57\.1\$\\times\$ higher inside this fork set than outside",
     "claims": ["base sub-arm", "fork dosing"], "record_venue": "NeurIPS 2026 workshop (temperature-law positioning)",
     "venue_found": "two NeurIPS 2026 workshop posters: 'Transitioning from Pre-training to Post-training' (172378) and "
                    "'ELLIS Workshop on the Foundations of LLM Post-Training in Changing Environments' (164442); no "
                    "main-track page found"},
    {"key": "sparse_but_critical", "ref": 143, "title": "Sparse but Critical: A Token-Level Analysis of Distributional "
     "Shifts in RLVR Fine-Tuning of LLMs", "authors": "Meng et al. (Qwen Pilot Team)", "files": ["sparse_but_critical"],
     "url": "https://arxiv.org/abs/2603.22446", "venue_patterns": [r"ICLR 2026"], "content_patterns": [r"Sparse but Critical"],
     "quote_rx": r"Published as a conference paper at the International Conference on Learning Representations \(ICLR 2026\)",
     "claims": ["base sub-arm"], "record_venue": "ICLR 2026", "venue_found": "ICLR 2026 (arXiv comment); arXiv v1, 2026-03-23"},
    {"key": "self_distillation", "ref": 144, "title": "Embarrassingly Simple Self-Distillation Improves Code Generation",
     "authors": "Zhang et al. (Apple)", "files": ["self_distillation", "self_distillation_html"],
     "url": "https://arxiv.org/abs/2604.01193", "venue_patterns": [], "content_patterns": [r"precision-exploration conflict",
                                                                                           r"forks?\b", r"locks?\b"],
     "quote_rx": r"code generation mixes precision-bound locks and exploration-bound forks",
     "claims": ["fork dosing", "temperature law"], "record_venue": "not in the record",
     "venue_found": "arXiv preprint, v2 (2026-06-24); no venue stated"},
    {"key": "position_aware", "ref": 145, "title": "Position-aware Automatic Circuit Discovery",
     "authors": "Haklay, Orgad, Bau, Mueller, Belinkov", "files": ["position_aware_acl", "position_aware_arxiv"],
     "url": "https://aclanthology.org/2025.acl-long.141/ ; https://arxiv.org/abs/2502.04577",
     "venue_patterns": [r"2025\.acl-long\.141"], "content_patterns": [r"dataset schema"],
     "quote_rx": r"we introduce the concept of a dataset schema, which defines token spans with similar semantics across examples",
     "claims": ["circuit"], "record_venue": "not in the record", "venue_found": "ACL 2025 long paper (2025.acl-long.141); arXiv v1 2025-02-07"},
    {"key": "many_circuits", "ref": 146, "title": "Many Circuits, One Mechanism: Input Variation and Evaluation "
     "Granularity in Circuit Discovery", "authors": "Bayat Makou, Niu, Dutta, Gurevych", "files": ["many_circuits",
                                                                                                   "many_circuits_html"],
     "url": "https://arxiv.org/abs/2606.06267", "venue_patterns": [r"Journal reference: \| Transactions on Machine Learning Research, 2026"],
     "content_patterns": [r"phantom specialization"],
     "quote_rx": r"We term this mismatch _?phantom specialization_?",
     "claims": ["circuit"], "record_venue": "TMLR 2026 (iteration-6 positioning, Table 92c row 19)",
     "venue_found": "TMLR 2026 (arXiv journal reference); arXiv v2 2026-08-26; a NeurIPS 2026 poster page was not fetched"},
    {"key": "mib", "ref": 52, "title": "MIB: A Mechanistic Interpretability Benchmark", "authors": "Mueller et al.",
     "files": ["mib"], "url": "https://arxiv.org/abs/2504.13151", "venue_patterns": [r"Accepted to ICML 2025"],
     "content_patterns": [r"MIB"], "quote_rx": r"Accepted to ICML 2025", "claims": ["matched DAS", "circuit"],
     "record_venue": "ICML 2025", "venue_found": "ICML 2025 (arXiv comment); v2 2025-06-09"},
    {"key": "invisible_lottery", "ref": 1, "title": "The Invisible Lottery: How Subtle Cues Steer Algorithm Choice in "
     "LLM Code Generation", "authors": "Narula et al.", "files": ["invisible_lottery"], "url": "https://arxiv.org/abs/2606.04057",
     "venue_patterns": [r"ICML 2026"], "content_patterns": [r"Invisible Lottery"], "quote_rx": r"How Subtle Cues Steer Algorithm Choice in LLM Code Generation",
     "claims": ["top-five prohibition"], "record_venue": "ICML 2026 (iteration-6 positioning)",
     "venue_found": "arXiv v1 (2026-06-02); no venue stated on the arXiv page"},
    {"key": "denial_prompting", "ref": 23, "title": "Benchmarking Language Model Creativity: A Case Study on Code "
     "Generation (denial prompting)", "authors": "Lu et al.", "files": ["denial_prompting"], "url": "https://arxiv.org/abs/2407.09007",
     "venue_patterns": [r"NAACL"], "content_patterns": [r"Model Creativity"], "quote_rx": r"Model Creativity: A Case Study on Code Generation",
     "claims": ["top-five prohibition"], "record_venue": "NAACL (reference [23] gives 2024; the plan gives 2025)",
     "venue_found": "arXiv v2; no venue stated on the arXiv page"},
    {"key": "manifold_steering", "ref": 27, "title": "Manifold Steering Reveals the Shared Geometry of Neural Network "
     "Representation and Behavior", "authors": "Wurgaft et al.", "files": ["manifold_steering"],
     "url": "https://arxiv.org/abs/2605.05115", "venue_patterns": [r"ICML", r"[Ss]potlight"], "content_patterns": [r"manifold"],
     "quote_rx": r"we test whether interventions that respect the geometry of activation space will yield behaviors close to those the model exhibits naturally",
     "claims": ["top-five prohibition"], "record_venue": "ICML 2026 MI-workshop spotlight (iteration-6 positioning)",
     "venue_found": "arXiv abstract page fetched; no venue stated"},
    {"key": "gravity_wells", "ref": 7, "title": "Semantic Gravity Wells: Why Negative Constraints Backfire",
     "authors": "Rana", "files": ["gravity_wells"], "url": "https://arxiv.org/abs/2601.08070", "venue_patterns": [],
     "content_patterns": [r"Semantic Gravity Wells"],
     "quote_rx": r"the very act of naming a forbidden word primes the model to produce it",
     "claims": ["top-five prohibition"], "record_venue": "arXiv 2026", "venue_found": "arXiv v1 (2026-01-12)"},
    {"key": "white_bear", "ref": 53, "title": "Don't Think of the White Bear: Ironic Negation in Transformer Models "
     "Under Cognitive Load", "authors": "Mann et al.", "files": ["white_bear"], "url": "https://arxiv.org/abs/2511.12381",
     "venue_patterns": [], "content_patterns": [r"White Bear"], "quote_rx": r"Ironic Negation in Transformer Models Under Cognitive Load",
     "claims": ["top-five prohibition"], "record_venue": "arXiv 2025", "venue_found": "arXiv v1 (2025-11-15)"},
]


def verify_live(web_dir: Path) -> list[dict[str, Any]]:
    out = []
    utc = (web_dir / "_run_utc.txt").read_text().split("\n")[0].strip() if (web_dir / "_run_utc.txt").exists() else ""
    for n in NEIGHBOURS:
        texts = []
        for f in n["files"]:
            p = web_dir / f"{f}.txt"
            if p.exists():
                raw = p.read_text(errors="replace")
                raw = raw.split("--- Content ---", 1)[1] if "--- Content ---" in raw else ""
                texts.append(re.sub(r"\s+", " ", raw))
        blob = " ".join(texts)
        fetched = bool(blob.strip())
        content_ok = fetched and all(re.search(pt, blob) for pt in n["content_patterns"])
        venue_ok = bool(n["venue_patterns"]) and all(re.search(pt, blob) for pt in n["venue_patterns"])
        m = re.search(n["quote_rx"], blob)
        status = ("NOT-FOUND" if not content_ok else "VERIFIED" if (venue_ok or not n["venue_patterns"]) else
                  "VERIFIED-ABSTRACT")
        out.append({**{k: v for k, v in n.items() if k not in ("quote_rx", "venue_patterns", "content_patterns")},
                    "status": status, "venue_verified": venue_ok, "quote": m.group(0) if m else "",
                    "checked_utc": utc, "tool": "aii-web-tools aii_fast_web_fetch.py grep",
                    "raw": [f"audit/web/{f}.txt" for f in n["files"]],
                    "record_check": ("new" if n["record_venue"] == "not in the record" else
                                     "confirmed" if (venue_ok or not n["venue_patterns"]) else "unverified")})
    return out


CLAIM_ROWS = [
    ("Fork-dosed temperature", "fork dosing", "PARTLY OCCUPIED",
     "Self-Distillation's account of code as precision-bound locks and exploration-bound forks, where raising "
     "temperature helps forks but floods locks [144]; Cantarero et al.'s Fork-Patched Decoding at base-entropy forks "
     "[142]; EDT [137], AdapT [32], min-p [138]; high-entropy minority tokens [139]",
     "temperature at a counterfactually located algorithm fork in code, with doses committed from teacher-forced logits, "
     "algorithm shares predicted before sampling and hidden-test correctness"),
    ("Matched DAS at the fork", "matched DAS", "OCCUPIED (method)",
     "DAS [110], Boundless DAS [111], CDAS [112]; MIB (ICML 2025, arXiv 2504.13151 v2) ranks DAS without specificity "
     "controls [52]; Makelov et al. [122]; Sutter et al. [132]",
     "seven specificity controls under one protocol in two families and per-direction generation verdicts (sufficient "
     "handle in Gemma, NO HANDLE at Qwen3-8B's fork)"),
    ("Circuit at Qwen3-8B's fork", "circuit", "PARTLY OCCUPIED (method)",
     "path patching [135], attribution patching [136]; Position-aware Automatic Circuit Discovery (ACL 2025 long paper) "
     "makes position classes and dataset schemas explicit [145]; MIB evaluates faithfulness on held-out inputs [52]; "
     "Many Circuits, One Mechanism (TMLR 2026, arXiv 2606.06267 v2): structural differences between circuits need not "
     "be different mechanisms [146]",
     "a position-classed circuit for an algorithm fork validated against matched random components and steered in "
     "generation; the motif comparison is structural only, and the layer-0 node's held-out arm is untested"),
    ("Gemma dial against prompting", "Gemma dial", "PARTLY OCCUPIED",
     "CDAS validates DAS steering by generation against prompting [112]; prefix steering [140]; AxBench [34]",
     "prefill-matched prompting cells on the same slots, a committed free-generation placement rule, and the dormancy "
     "and implementation-channel analysis"),
    ("Temperature law over forks", "temperature law", "PARTLY OCCUPIED",
     "forking-path resampling (Bigelow et al., ICLR 2025, the locator's method source); high-entropy forks [139]; "
     "algorithm-level diversity under temperature measured post hoc (Lee et al., EMNLP Findings 2025); the fork and lock "
     "temperature asymmetry of [144]",
     "a law over blindly located forks committed before sampling and scored on held-out families against flat and "
     "tempering rivals, which stay close"),
    ("Base sub-arm", "base sub-arm", "PARTLY OCCUPIED",
     "Cantarero et al.: per-position Jensen-Shannon divergence 18.4 to 57.1 times higher inside the base model's fork set, "
     "1.06 to 1.70 times after entropy adjustment, Fork-Patched Decoding recovers 32.8% of distinct answers against 4.7% "
     "at random placement (two NeurIPS 2026 workshop posters) [142]; Sparse but Critical (ICLR 2026): only a small "
     "fraction of token distributions diverge after RLVR [143]; URIAL [100]",
     "the concentration of post-training shifts at base-entropy forks is theirs; the run adds the algorithm-level "
     "outcome on code at a counterfactually located fork and the reading that the default is the base model's second or "
     "third choice, sharpened"),
    ("Top-five prohibition", "top-five prohibition", "ADJACENT-ONLY (landing); OCCUPIED (bans backfire)",
     "The Invisible Lottery (arXiv 2606.04057) [1]; denial prompting (arXiv 2407.09007) [23]; Semantic Gravity Wells: "
     "naming a forbidden word primes it (arXiv 2601.08070) [7]; Don't Think of the White Bear (arXiv 2511.12381) [53]; "
     "Manifold Steering (arXiv 2605.05115) [27]",
     "a screen of ten configurations and the landing of survivors on the obeying model's manifold, which is alias-named, "
     "so names and mechanisms are confounded there"),
]

NEW_REFS = [
    "[142] Cantarero et al. The RLVR Policy Shift Is Sparse and Predictable from the Base Model. NeurIPS 2026 Workshops "
    "(Transitioning from Pre-training to Post-training; ELLIS Workshop on the Foundations of LLM Post-Training in "
    "Changing Environments). 2026.",
    "[143] Meng et al. Sparse but Critical: A Token-Level Analysis of Distributional Shifts in RLVR Fine-Tuning of LLMs. "
    "International Conference on Learning Representations. 2026.",
    "[144] Zhang et al. Embarrassingly Simple Self-Distillation Improves Code Generation. arXiv preprint arXiv:2604.01193. "
    "2026.",
    "[145] Haklay et al. Position-aware Automatic Circuit Discovery. Annual Meeting of the Association for Computational "
    "Linguistics. 2025.",
    "[146] Bayat Makou et al. Many Circuits, One Mechanism: Input Variation and Evaluation Granularity in Circuit "
    "Discovery. Transactions on Machine Learning Research. 2026.",
]


def positioning(out_dir: Path) -> dict[str, Any]:
    carried = adapt_positioning()
    live = verify_live(out_dir / "audit/web")
    rows = []
    for title, claim, verdict, nb, adds in CLAIM_ROWS:
        for c in carried:
            if c["claim"] == claim:
                rows.append({"claim": title, "neighbour": c["neighbour"][:200], "authors": "", "venue_track": "",
                             "arxiv_id_version": "", "url": c["url"], "quote": c["quote"][:400],
                             "verdict": c["verdict"][:200], "adds": adds, "status": "CARRIED", "source": c["source"]})
        for n in live:
            if claim in n["claims"]:
                rows.append({"claim": title, "neighbour": n["title"], "authors": n["authors"],
                             "venue_track": n["venue_found"], "arxiv_id_version": n["url"], "url": n["url"],
                             "quote": n["quote"], "verdict": verdict, "adds": adds, "status": n["status"],
                             "source": ";".join(n["raw"])})
    with (out_dir / "tables/positioning_rows.csv").open("w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        w.writeheader()
        w.writerows(rows)
    status_counts: dict[str, int] = {}
    for n in live:
        status_counts[n["status"]] = status_counts.get(n["status"], 0) + 1
    n_corr = sum(1 for n in live if n["record_check"] == "unverified")
    utc = live[0]["checked_utc"][:10] if live else "unknown"
    table = ["| Claim of iteration 8 | Nearest neighbours (venues and versions checked live) | Verdict | What this run "
             "adds |", "|-|-|-|-|"]
    for title, claim, verdict, nb, adds in CLAIM_ROWS:
        table.append(f"| {title} | {nb} | {verdict} | {adds} |")
    canta = next(n for n in live if n["key"] == "cantarero")
    section = (
        f"## 8.10b Positioning of the iteration-8 claims {M}\n\n"
        f"The iteration-8 artifacts each searched their own neighbours, but the record never set the seven claims against "
        f"the closest published work in one place. Table 416b does, with every venue and version checked live on {utc} "
        f"by fetching each page and grepping it for the quoted text (raw outputs in the Step-0 artifact's audit/web; "
        f"{status_counts.get('VERIFIED', 0)} neighbours verified with their venue, {status_counts.get('VERIFIED-ABSTRACT', 0)} "
        f"verified from the abstract page only, {status_counts.get('NOT-FOUND', 0)} not found).\n\n"
        f"**Table 416b.** The claims of iteration 8 against their nearest neighbours {M}.\n\n" + "\n".join(table) +
        f"\n\nBoth NeurIPS 2026 pages of Cantarero et al. are posters in workshops ({canta['venue_found'].split(': ', 1)[1]}), "
        f"and the abstract states that the divergence is \"18.4--57.1x higher inside this fork set than outside\", that "
        f"\"an excess of 1.06--1.70x remains\" after the entropy adjustment, and that Fork-Patched Decoding moves the number "
        f"of distinct answers \"32.8% of the way back toward the base\" while random placement recovers \"only 4.7%\". "
        f"Sparse but Critical is an ICLR 2026 paper, MIB an ICML 2025 paper (arXiv v2), and Many Circuits, One Mechanism "
        f"carries the journal reference Transactions on Machine Learning Research, 2026 (arXiv v2), which confirms the "
        f"venue Table 92c gives it. The arXiv pages of Manifold Steering, The Invisible Lottery and the denial-prompting "
        f"study state no venue, so the venues that earlier sections give them (an ICML 2026 workshop spotlight, ICML 2026, "
        f"and NAACL, which reference [23] dates 2024) are carried, not verified ({n_corr} venue statements left "
        f"unverified, none contradicted). No iteration-8 result is called first.")
    return {"rows": rows, "live": live, "carried_n": len(carried), "status_counts": status_counts,
            "n_venue_or_version_corrections": n_corr, "section": section, "new_refs": NEW_REFS}


# ================================================================================================ 0g figure markers
def figure_markers(out_dir: Path, s0e: dict[str, Any]) -> list[dict[str, Any]]:
    f8 = load(FORK, "results/geometry_f8.json")
    dg = load(DAS, "results/geometry.json")
    m1 = load(TP, "results/geometry_M1.json")
    m2 = load(TP, "results/geometry_M2.json")
    impl = m1["baseline_implemented"]
    un = {"T10": "connected components", "B3": "minimum spanning tree weight", "A3": "trapped rain water",
          "B6": "valid anagram", "G": "Gemma grid path", "T1": "Qwen3-8B grid path"}
    n_f8 = ", ".join(f"{un.get(u, u)} {f8[u]['n_states']}" for u in f8 if isinstance(f8[u], dict) and "n_states" in f8[u])
    agree = ", ".join(f"{un.get(u, u)} {f8[u]['nearest_centroid_agreement_fork_dosed']:.3f}" for u in f8
                      if isinstance(f8[u], dict) and f8[u].get("nearest_centroid_agreement_fork_dosed") is not None)
    evs = [sum(f8[u]["explained_variance_pc1_pc2"]) for u in f8 if isinstance(f8[u], dict) and
           f8[u].get("explained_variance_pc1_pc2")]
    ev_rng = f"{min(evs):.0%} to {max(evs):.0%}"
    nice = lambda d: ", ".join(f"{k.replace('_', '-')} {v}" for k, v in sorted(d.items(), key=lambda kv: -kv[1]))  # noqa: E731
    ic = {u: dg[u]["implemented_counts"] for u in ("T10", "GRID")}
    dec = m2["label_decodability_balanced_acc"]
    specs = [
        {"id": "fig_forkdose_states", "art": FORK, "files": ["figures/F8_activation_geometry"], "section": "8.3",
         "edit_id": "S0-0g-01", "anchor": "No correct program of 5,543 is novel (rule-of-three upper bound 0.0005).",
         "code": f"{FORK}:src/figures8.py#f8",
         "fields": {"positions": "mean of the 16 positions from the fork token on (fork to fork + 15)",
                    "block": "final-norm hidden state (after the last block)",
                    "projection": f"PCA per unit, first two components ({ev_rng} of variance; linear, no k or stress)",
                    "N": f"states per unit: {n_f8} (arms twin, low and high dose, class-uniform forcing, global T 1.3 and 2.0)",
                    "centroids": "no centroids drawn, so none sized by implemented count; colour = program class, marker = "
                                 f"arm; the nearest-centroid check (class-forced centroids) agrees on {agree}",
                    "caveat": "correlational; states after the fork partly encode the tokens already written"}},
        {"id": "fig_das_handle_coords", "art": DAS, "files": ["figures/F3_geometry_T10", "figures/F3_geometry_GRID"],
         "section": "8.4", "edit_id": "S0-0g-02", "anchor": 'The figure titles say "handle coordinates (not certified)".',
         "code": f"{DAS}:src/figures8.py#F3",
         "fields": {"positions": "the fork token (index 11 after ' def' on connected components; index 32 in Gemma)",
                    "block": f"chosen block {dg['T10']['L']} (Qwen3-8B) and {dg['GRID']['L']} (Gemma)",
                    "projection": f"every prompt's coordinates in the fitted rank-8 DAS subspace, centred on the twin mean, "
                                  f"embedded by metric MDS (Kruskal stress-1 {dg['T10']['mds_stress1']:.3f} and "
                                  f"{dg['GRID']['mds_stress1']:.3f}) and Isomap (k 8), beside a PCA of the raw block states",
                    "N": f"{dg['T10']['n_prompts']} prompts (connected components) and {dg['GRID']['n_prompts']} (Gemma)",
                    "centroids": f"class centroids (stars) sized by implemented, test-passing programs: connected "
                                 f"components {nice(ic['T10'])}; Gemma {nice(ic['GRID'])}",
                    "caveat": "off-manifold DAS coordinates: both fork fits raise the off-manifold flag and fail the label "
                              "and random-initialisation controls, so these are handle coordinates, not a certified variable"}},
        {"id": "fig_das_handle_axis", "art": DAS, "files": ["figures/F3b_handle_axis_T10", "figures/F3b_handle_axis_GRID"],
         "section": "8.4", "edit_id": None, "anchor": None, "code": f"{DAS}:src/figures8.py#F3b",
         "fields": {"positions": "the fork token, as in fig_das_handle_coords",
                    "block": f"{dg['T10']['L']} (Qwen3-8B) and {dg['GRID']['L']} (Gemma)",
                    "projection": "the first fitted handle coordinate per prompt group (one axis; no k or stress)",
                    "N": f"{dg['T10']['n_prompts']} and {dg['GRID']['n_prompts']} prompts",
                    "centroids": "no centroids drawn; implemented counts as in fig_das_handle_coords",
                    "caveat": "off-manifold DAS coordinates (not certified)"}},
        {"id": "fig_gemma_dial_geometry", "art": GEM, "files": ["figures/F4_geometry"], "section": "8.6",
         "edit_id": "S0-0g-03", "anchor": "The subspace moves the algorithm, BFS against DFS, but not the forced prompt's "
                                         "choice of implementation.", "code": f"{GEM}:src/outputs8.py#F4",
         "fields": {"positions": "the fork (decision) position, index 32 after the shared opening",
                    "block": "block 17 of gemma-3-4b-it",
                    "projection": "(R1, R2) of the rank-4 DAS map (seed 0) and (PC1, PC2) of the 120 twin variants "
                                  "(linear; no k or stress)",
                    "N": "120 twin variants, 40 forced sources (BFS and DFS), the free twin at its decision position, and "
                         "the same free-twin states after DAS toward BFS, DAS toward DFS and difference in means toward BFS",
                    "centroids": "no centroids drawn, so none sized by implemented count; the twin implements DFS in 32 and "
                                 "BFS in 18 of 50 slots (Table 386); colour = fork margin of each twin variant",
                    "caveat": "dormant or off-manifold handle by retention (0.18 to 0.28): the edited states are DAS "
                              "coordinates, correlational as geometry"}},
        {"id": "fig_tp_mds", "art": TP, "files": ["figures/F3_mds_C6_nf4"], "section": "8.8", "edit_id": "S0-0g-04",
         "anchor": "the only catalogue algorithm the model writes under the ban, forms its own cloud at the bottom, away "
                   "from the ban prompts.", "code": f"{TP}:src/figures.py#manifold_figures",
         "fields": {"positions": "S-mean over the task-specification tokens of each forced prompt",
                    "block": f"block {m1['L_star']} of Qwen3-32B (NF4 weights)",
                    "projection": f"classical MDS of the 300 points (Kruskal stress-1 {m1['M1']['mds_points_stress1']:.2f}) "
                                  f"and of the 10 centroids ({m1['M1']['mds_centroids_stress1']:.2f})",
                    "N": "300 forced prompts (10 algorithms x 10 templates x 3 aliases), with baseline and ban prompts "
                         "projected",
                    "centroids": f"sized by baseline implemented count ({nice(impl)}; every other algorithm 0)",
                    "caveat": "alias-named manifold (each prompt names its algorithm) and correlational (in-manifold pushes "
                              "moved 0 of 300 programs)"}},
        {"id": "fig_tp_isomap", "art": TP, "files": ["figures/F4_isomap_C6_nf4"], "section": "8.8", "edit_id": None,
         "anchor": None, "code": f"{TP}:src/figures.py#manifold_figures",
         "fields": {"positions": "S-mean over the task-specification tokens", "block": f"block {m1['L_star']}",
                    "projection": "Isomap on the symmetrised kNN geodesic graph at k 5, 10 and 15 (8, 3 and 1 components "
                                  "before joining); overlays at the primary k 15",
                    "N": "300 forced prompts plus projected baseline, ban and confirmation prompts",
                    "centroids": f"sized by baseline implemented count ({nice(impl)}; every other algorithm 0)",
                    "caveat": "alias-named and correlational"}},
        {"id": "fig_tp_program_states", "art": TP, "files": ["figures/F5_M2_C6"], "section": "8.8", "edit_id": "S0-0g-05",
         "anchor": "Iteration 7 found that pushes inside the manifold moved 0 of 300 programs, so none of this geometry is "
                   "causal.", "code": f"{TP}:src/figures.py#m2_figure",
         "fields": {"positions": "p*, the first token that signals the program's class (label-dependent)",
                    "block": f"block {m2['L_star']}",
                    "projection": "PCA and Isomap (k 10) of per-program states",
                    "N": f"{m2['n_programs']} programs (all valid ban programs and sampled baseline programs)",
                    "centroids": "no centroids; correct survivors starred",
                    "caveat": f"the label is decodable at p* (balanced accuracy {dec['p_star']:.2f}) but only at "
                              f"{dec['g16']:.2f} at the label-independent 16th code token (chance 0.25), so the separation "
                              f"is largely written into the tokens; correlational"}},
        {"id": "fig_alias_overlap", "art": "this artifact (iteration-9 Step 0)", "files": ["figures/alias_overlap"],
         "section": "8.8", "edit_id": None, "anchor": None, "code": "src/stage_0e.py#_figure",
         "fields": {"positions": "S-mean over the task-specification tokens (iteration 8's stored states)",
                    "block": "block 31 of Qwen3-32B (NF4)",
                    "projection": f"panels A-B: alias-level g against lexical overlap at k 15 and k 10; panel C: classical "
                                  f"MDS of the stored k 15 geodesics (stress-1 {s0e['per_k']['k15'].get('mds_stress1', float('nan')):.2f})",
                    "N": "15 alias sets of 10 prompts (non-banned) plus 150 banned prompts; 300 points in panel C",
                    "centroids": "algorithm centroids (stars) are not sized: g is a distance, not a count; the screen's "
                                 "catalogue survivors were unit-weight Dijkstra 6 of 6",
                    "caveat": "alias-named and correlational; between algorithms names and mechanisms are confounded"}},
    ]
    out = []
    for s in specs:
        base = ART[s["art"]] if s["art"] in ART else out_dir
        hashes = {}
        for f in s["files"]:
            for ext in ("png", "pdf"):
                p = base / f"{f}.{ext}"
                hashes[f"{f}.{ext}"] = sha(p) if p.exists() else None
        fl = s["fields"]
        cap = (f"Positions: {fl['positions']}. Block: {fl['block']}. Projection: {fl['projection']}. N: {fl['N']}. "
               f"Centroids: {fl['centroids']}. Caveat: {fl['caveat']}. Files ({s['art']}): "
               + ", ".join(f"{k} sha256 {v[:12] if v else 'missing'}" for k, v in hashes.items()) + f"; code {s['code']}.")
        out.append({**s, "caption": cap, "sha256": hashes, "n_fields": sum(1 for v in fl.values() if v),
                    "block_md": f"[FIGURE:{s['id']}]\n\n*Figure {s['id']} {M}.* {cap}"})
    return out


def marker_edits(markers: list[dict[str, Any]]) -> list[dict[str, Any]]:
    by = {m["id"]: m for m in markers}
    gem_par = (f"**The dial's geometry {M}.** Figure fig_gemma_dial_geometry plots Gemma's block-17 fork states in actual "
               f"activation coordinates: the 120 twin variants (coloured by fork margin), the forced-BFS and forced-DFS "
               f"sources, the free twin at its decision position, and the same free-twin states after the rank-4 DAS edit "
               f"toward BFS or DFS and after the difference-in-means edit toward BFS. In the DAS plane the twin states form "
               f"one tight cluster; the DAS edits carry them onto the forced-BFS or forced-DFS sources, while the "
               f"difference-in-means edit throws them far beyond every natural state. In the twins' own first two "
               f"principal components the DAS-edited states stay inside the twin columns and the difference-in-means "
               f"states leave them. This is the retention picture of Table 406: a subspace that the twins' noun-driven "
               f"variation barely uses but that prompts use, and an edit along it that stays near the natural states "
               f"where the difference-in-means edit does not.")
    return [
        {"edit_id": "S0-0g-01", "section": "8.3", "anchor": by["fig_forkdose_states"]["anchor"],
         "block": by["fig_forkdose_states"]["block_md"], "why": "Geometry at the fork had no figure marker"},
        {"edit_id": "S0-0g-02", "section": "8.4", "anchor": by["fig_das_handle_coords"]["anchor"],
         "block": by["fig_das_handle_coords"]["block_md"] + "\n\n" + by["fig_das_handle_axis"]["block_md"],
         "why": "DAS handle geometry had no figure markers"},
        {"edit_id": "S0-0g-03", "section": "8.6", "anchor": by["fig_gemma_dial_geometry"]["anchor"],
         "block": gem_par + "\n\n" + by["fig_gemma_dial_geometry"]["block_md"],
         "why": "Gemma F4 had no marker and no description"},
        {"edit_id": "S0-0g-04", "section": "8.8", "anchor": by["fig_tp_mds"]["anchor"],
         "block": by["fig_tp_mds"]["block_md"] + "\n\n" + by["fig_tp_isomap"]["block_md"],
         "why": "MDS and Isomap of the prohibition manifold had no markers"},
        {"edit_id": "S0-0g-05", "section": "8.8", "anchor": by["fig_tp_program_states"]["anchor"],
         "block": by["fig_tp_program_states"]["block_md"], "why": "Per-program states had no marker"},
    ]


# ================================================================================================ 0e text
def f2(x: float) -> str:
    return f"{x:.2f}".replace("-0.00", "0.00")


def pfmt(p: float) -> str:
    return f"{p:.3f}" if p >= 0.001 else f"{p:.1e}"


def alias_text(Z: dict[str, Any], led: Ledger, alias_marker_md: str, T: dict[str, Any] | None = None) -> dict[str, Any]:
    k15, k10 = Z["per_k"]["k15"], Z["per_k"]["k10"]
    src = "S0:results/s0e.json"
    v = {}

    def a(name: str, val: Any, key: str, spec: str = ".2f") -> str:
        v[name] = led.add(name, "0e", val, f"{src}#{key}", spec)
        return v[name]

    rho15 = a("0e.k15.rho_primary", k15["rho_primary"], "per_k.k15.rho_primary")
    p15 = a("0e.k15.p_primary", k15["primary_p_lower"], "per_k.k15.primary_p_lower", ".3f")
    ci = k15["ci95_primary"]
    ci_lo = a("0e.k15.ci_lo", ci[0], "per_k.k15.ci95_primary[0]")
    ci_hi = a("0e.k15.ci_hi", ci[1], "per_k.k15.ci95_primary[1]")
    rho10 = a("0e.k10.rho_primary", k10["rho_primary"], "per_k.k10.rho_primary")
    p10 = a("0e.k10.p_primary", k10["primary_p_lower"], "per_k.k10.primary_p_lower", ".4f")
    rho2 = a("0e.k15.rho_secondary", k15["rho_secondary"], "per_k.k15.rho_secondary")
    c5 = k15["five_centroid_primary"]
    rho5 = a("0e.k15.rho_5c", c5["rho"], "per_k.k15.five_centroid_primary.rho")
    p5 = a("0e.k15.p_5c", c5["p_lower"], "per_k.k15.five_centroid_primary.p_lower")
    fl5 = a("0e.k15.floor_5c", c5["floor_lower"], "per_k.k15.five_centroid_primary.floor_lower", ".3f")
    st = k15["stratified_secondary"]
    strat = a("0e.k15.strat", st["stat"], "per_k.k15.stratified_secondary.stat")
    pstrat = a("0e.k15.strat_p", st["p_lower"], "per_k.k15.stratified_secondary.p_lower")
    neff = a("0e.k15.strat_neff", st["n_arrangements_effective"], "per_k.k15.stratified_secondary.n_arrangements_effective", "int")
    nalg = a("0e.k15.strat_nalg", st["n_algorithms_used"], "per_k.k15.stratified_secondary.n_algorithms_used", "int")
    beta = a("0e.k15.beta", st["beta"], "per_k.k15.stratified_secondary.beta")
    pbeta = a("0e.k15.beta_p", st["beta_p_two_sided"], "per_k.k15.stratified_secondary.beta_p_two_sided")
    st10 = k10["stratified_secondary"]
    strat10 = a("0e.k10.strat_p", st10["p_lower"], "per_k.k10.stratified_secondary.p_lower")
    pl = Z["placebos"]
    p1 = a("0e.P1", pl["P1_shuffled_alias_labels"]["share_p_lt_0.05"], "placebos.P1_shuffled_alias_labels.share_p_lt_0.05", ".3f")
    p2 = a("0e.P2", pl["P2_random_overlap_vector"]["share_p_lt_0.05"], "placebos.P2_random_overlap_vector.share_p_lt_0.05", ".3f")
    prho = a("0e.C1.rho", pl["C1_planted"]["seed0_rho"], "placebos.C1_planted.seed0_rho")
    pp = a("0e.C1.p", pl["C1_planted"]["seed0_p_lower"], "placebos.C1_planted.seed0_p_lower", "sig1")
    e4v = Z["gates"].get("E4_spearman_vs_stored_k15")
    e4 = a("0e.E4", e4v, "gates.E4_spearman_vs_stored_k15", ".3f") if e4v is not None else None
    bpe_sent, bpe_short = "", ""
    if T:
        tsrc = "S0:results/s0e_tertiary.json"
        v["0e.k15.bpe_rho"] = led.add("0e.k15.bpe_rho", "0e", T["per_k"]["k15"]["rho_tertiary"],
                                      f"{tsrc}#per_k.k15.rho_tertiary", ".2f")
        v["0e.k15.bpe_strat"] = led.add("0e.k15.bpe_strat", "0e", T["per_k"]["k15"]["stratified"]["stat"],
                                        f"{tsrc}#per_k.k15.stratified.stat", ".2f")
        v["0e.k15.bpe_strat_p"] = led.add("0e.k15.bpe_strat_p", "0e", T["per_k"]["k15"]["stratified"]["p_lower"],
                                          f"{tsrc}#per_k.k15.stratified.p_lower", ".3f")
        v["0e.k10.bpe_strat"] = led.add("0e.k10.bpe_strat", "0e", T["per_k"]["k10"]["stratified"]["stat"],
                                        f"{tsrc}#per_k.k10.stratified.stat", ".2f")
        v["0e.k10.bpe_strat_p"] = led.add("0e.k10.bpe_strat_p", "0e", T["per_k"]["k10"]["stratified"]["p_lower"],
                                          f"{tsrc}#per_k.k10.stratified.p_lower", ".2f")
        bpe_sent = (f" A sensitivity committed as optional scores overlap with the model's own BPE tokens (function words "
                    f"included, so every algorithm's aliases differ): the 15-set correlation is {v['0e.k15.bpe_rho']}, and the "
                    f"within-algorithm statistic is {v['0e.k15.bpe_strat']} (p {v['0e.k15.bpe_strat_p']} over 7,776 relabelings; "
                    f"{v['0e.k10.bpe_strat']}, p {v['0e.k10.bpe_strat_p']} at k 10), so a small within-algorithm name effect "
                    f"cannot be excluded at this resolution.")
        bpe_short = f"; {v['0e.k15.bpe_strat_p']} on the model's BPE tokens"
    g = k15["g_alias"]
    rng = lambda al: f"{min(g[f'{al}|{j}'] for j in range(3)):.1f} to {max(g[f'{al}|{j}'] for j in range(3)):.1f}"  # noqa: E731
    code = Z["verdict"]["code"]
    rule = {"CONFOUNDED": (f"g follows alias overlap across algorithms (rho {rho15}, exact p {p15}), but the pattern is "
                           f"carried by algorithm identity; on this manifold names and mechanisms are confounded, and "
                           f"TP-G's name-free manifolds decide"),
            "NAMES": "names move the geometry even with the algorithm held fixed",
            "NOT_TRACKED": f"iteration 8's geodesics do not track alias overlap at the committed strength (rho {rho15})"}[code]
    par = (
        f"**Is 'opposite' a property of the names? {M}** Each of the 300 forced prompts names its algorithm by one of "
        f"three aliases, and the aliases of the edge algorithms share words with the banned names. On iteration 8's own "
        f"stored geodesics at block 31 (every gate passes: the stored labels, prompt groups and algorithm-level g "
        f"reproduce to 1e-9, and " + (f"an independent kNN-geodesic recomputation from the stored states correlates with "
                                      f"the stored k 15 matrix at {e4}" if e4 else "the independent kNN-geodesic "
                                      "recomputation was not run in this copy") + f"), each non-banned algorithm was split into its three alias sets of 10 prompts and each "
        f"set's g was computed (unit-weight Dijkstra {rng('dijkstra_unit')}, jump point search {rng('jps')}, IDA* "
        f"{rng('idastar')}, bidirectional A* {rng('biastar')}, bidirectional BFS {rng('bibfs')}). The lexical overlap was "
        f"committed by hash before any alias-level number: the number of banned-name detectors an alias matches (0 for "
        f"every unit-weight Dijkstra and jump point search alias, 1 for every bidirectional BFS and bidirectional A* alias, "
        f"1 to 3 for IDA*). Across the 15 alias sets at k 15 the Spearman correlation is {rho15} (exact one-sided p {p15} "
        f"over all 360,360 assignments; 95% interval over template resamples {ci_lo} to {ci_hi}), {rho10} at k 10 (p "
        f"{p10}), and {rho2} with content-word overlap. That statistic treats the 15 sets as exchangeable although they "
        f"are nested in five algorithms, and between algorithms lexical overlap is confounded with mechanism: "
        f"bidirectional BFS is next to BFS both in name and in mechanism. At the level of algorithms the correlation is "
        f"{rho5} with p {p5} (exact over 120 permutations; the tied overlap vector cannot go below {fl5}). Holding the "
        f"algorithm fixed and varying only the name, the within-algorithm test on content-word overlap ({nalg} algorithms "
        f"whose aliases differ; exact null over {neff} distinct relabelings) gives a mean within-algorithm Spearman of "
        f"{strat} (p {pstrat}; p {strat10} at k 10), and the fixed-effects slope of g on overlap is {beta} (two-sided "
        f"stratified p {pbeta}), the wrong sign for a name effect.{bpe_sent} Placebos fail as they must (at the 0.05 level, shuffled "
        f"point labels reject in a share {p1} of draws and random overlap vectors in {p2}) and a planted effect is "
        f"recovered (rho {prho}, p {pp}). By the committed "
        f"wording rule: {rule}. The planner's expectation of about -0.70 was public before the computation, so only the "
        f"within-algorithm test and the interval were unseen; tables/alias_g.csv of the Step-0 artifact gives every set."
        f"\n\n{alias_marker_md}")
    short = (f"its alias analysis (Section 8.8) finds that g follows alias overlap across algorithms (rho {rho15}, exact p "
             f"{p15}) but not clearly within them (stratified p {pstrat} on content words{bpe_short}): names and mechanisms "
             f"are confounded there, and the name-free manifolds of TP-G decide")
    return {"paragraph": par, "short": short, "value_names": list(v.keys()), "printed": v, "code": code}


## The 49 edits: `src/edits_spec.py` (unchanged)

Each edit is `{id, item, section, old_text, new_text, rationale, values}`:
- `old_text` is cut verbatim from the report. A table is cut as one caption-plus-table block, so that the marker sits in
  its caption.
- Every number inside `new_text` comes from the ledger through `P(name)`; none is retyped.
- `values` lists the ledger names the edit cites, and `eval.py` checks that each one exists.

In [ ]:
%%writefile eval_demo_run/art_iYh9zVP7myjJ/src/edits_spec.py
#!/usr/bin/env python3
"""The iteration-9 Step-0 edits of the iteration-8 record: verbatim old_text -> new_text tagged '(iteration-9 correction)'.

Every number inside a new_text is taken from the recomputed value ledger (P(name) returns the printed recomputed
value), from the 0e results (results/s0e.json), or from the positioning and figure tables built in eval.py; none is
retyped. Table edits replace caption + table as one unit so that the marker sits in the caption.
"""
from __future__ import annotations

from typing import Any, Callable

M = "(iteration-9 correction)"


def _table_block(report: str, caption_start: str) -> str:
    """Verbatim caption + table block starting at caption_start (through the last '|' line)."""
    i = report.index(caption_start)
    j = report.index("\n\n", i) + 2           # end of caption paragraph
    lines = report[j:].split("\n")
    k = 0
    while k < len(lines) and lines[k].startswith("|"):
        k += 1
    return report[i:j] + "\n".join(lines[:k])


def build(report: str, P: Callable[[str], str], Z: dict[str, Any], ctx: dict[str, Any]) -> list[dict[str, Any]]:
    E: list[dict[str, Any]] = []

    def add(eid: str, item: str, section: str, old: str, new: str, why: str, values: list[str]) -> None:
        E.append({"id": eid, "item": item, "section": section, "old_text": old, "new_text": new, "rationale": why,
                  "values": values})

    k15, k10 = Z["per_k"]["k15"], Z["per_k"]["k10"]

    # ============================================================================================ 0a circuit
    t395 = _table_block(report, "**Table 395.**")
    lines = t395.split("\n")
    cap = lines[0]
    rows395 = [l for l in lines if l.startswith("| ") and not l.startswith("| Node")]
    keys395 = list(ctx["rec395"].keys())
    new_rows, vals = [], []
    for row, k in zip(rows395, keys395):
        base = row.rstrip()
        if k == "mlp|L0|H-1|COND":
            new_rows.append(base + " 0.00 / 0.00 (edit not applied; see text) | n/a (edit not applied) |")
        else:
            new_rows.append(base + f" {P(f'0a.T395.{k}.abl_h')} / {P(f'0a.T395.{k}.ste_h')} | "
                                   f"{P(f'0a.T395.{k}.abl_hp')} / {P(f'0a.T395.{k}.ste_hp')} |")
        vals += [f"0a.T395.{k}.{s}" for s in ("abl", "abl_p", "ste", "ste_p", "abl_h", "ste_h", "abl_hp", "ste_hp")]
    q = ctx["code_quotes"]
    para = (
        f"\n\n**The held-out arm of the layer-0 node {M}.** The layer-0 MLP at the condition span has a held-out effect "
        f"of exactly {P('0a.L0.abl.heldout')} nats under ablation and steering because the held-out edit was never "
        f"applied. The class means that the edit writes are taken, for a condition-span node, only over pair prompts of "
        f"the held-out prompt's own template (`MeanCache.get` in `src/k3circuit.py`, line 482: `{q['482']}`; line 485 "
        f"also skips prompts whose span length differs). The 10 held-out prompts are templates F11 to F15 and the 20 "
        f"pair prompts F1 to F10, so {P('0a.cond_matches')} of the 10 held-out prompts has a matching pair prompt, the "
        f"mean is None (line 488), no edit is built, and the edited and unedited margins coincide; the stored rows show "
        f"the node and all 50 matched random components of the same class at exactly 0 on every held-out prompt "
        f"(20 of 20 prompt-by-mode rows). The node's {P('0a.L0.abl.mean_d')} nats (ablation) and {P('0a.L0.ste.mean_d')} "
        f"nats (steering) over 30 prompts are therefore {P('0a.L0.abl.pair')} and {P('0a.L0.ste.pair')} nats over the "
        f"{P('0a.C1forced.n_pair')} pair prompts plus ten zeros ({P('0a.L0.abl.pair')} x 20 / 30 = "
        f"{P('0a.L0.abl.mean_d')}); the identity mean x 30 = pair mean x 20 + held-out mean x 10 holds for every node of "
        f"both tables, and no other node has an exact-zero held-out effect ({P('0a.n_zero_nodes')} node in all, under "
        f"both modes). Its held-out status is untested, not failed: on the descriptive natural contrast, whose held-out "
        f"prompts share templates with its pairs, the same node gives {P('0a.nat.abl.heldout')} nats under ablation and "
        f"{P('0a.nat.ste.heldout')} under steering on {P('0a.nat.n_heldout')} held-out prompts (sign-flip p "
        f"{P('0a.nat.abl.p_heldout')}, the smallest one-sided p that five prompts allow, 1/32). BASE (B3) of iteration 9 "
        f"reruns the held-out arm with template-matched means.")
    hdr_old = "| Node | Ablation effect | Ablation p | Steering effect | Steering p | Both pass |\n|-|-|-|-|-|-|"
    hdr_new = ("| Node | Ablation effect | Ablation p | Steering effect | Steering p | Both pass | Held-out mean effect, "
               "ablation / steering (nats) | Held-out sign-flip p, ablation / steering |\n|-|-|-|-|-|-|-|-|")
    new395 = (cap + f" The two added columns give the mean effect over the 10 held-out prompts and its exact one-sided "
              f"sign-flip p (smallest possible 1/1024) {M}.\n\n" + hdr_new + "\n" + "\n".join(new_rows) + para)
    assert hdr_old in t395
    add("S0-0a-01", "0a", "8.5", t395, new395,
        "Table 395 gains the held-out columns; the layer-0 condition node's exact-zero held-out effect is an unapplied "
        "edit (MeanCache template filter), not a held-out failure", vals + ["0a.L0.abl.mean_d", "0a.L0.ste.mean_d",
                                                                          "0a.L0.abl.pair", "0a.L0.ste.pair",
                                                                          "0a.L0.abl.heldout", "0a.nat.abl.heldout",
                                                                          "0a.nat.ste.heldout", "0a.nat.abl.p_heldout",
                                                                          "0a.nat.n_heldout", "0a.cond_matches",
                                                                          "0a.n_zero_nodes", "0a.C1forced.n_pair"])
    t396 = _table_block(report, "**Table 396.**")
    lines = t396.split("\n")
    rows396 = [l for l in lines if l.startswith("| ") and not l.startswith("| Node")]
    keys396 = list(ctx["rec396"].keys())
    new_rows, vals = [], []
    for row, k in zip(rows396, keys396):
        new_rows.append(row.rstrip() + f" {P(f'0a.T396.{k}.abl_h')} / {P(f'0a.T396.{k}.ste_h')} | "
                                       f"{P(f'0a.T396.{k}.abl_hp')} / {P(f'0a.T396.{k}.ste_hp')} |")
        vals += [f"0a.T396.{k}.{s}" for s in ("abl", "abl_p", "ste", "ste_p", "abl_h", "ste_h", "abl_hp", "ste_hp")]
    new396 = (lines[0] + f" Held-out columns as in Table 395; no node here sits at the condition span, and every "
              f"held-out edit was applied {M}. {ctx['weak396']}\n\n" + hdr_new + "\n" + "\n".join(new_rows))
    add("S0-0a-02", "0a", "8.5", t396, new396, "Table 396 gains the held-out columns (connected components)",
        vals + ["0a.C2a.n_prompts", "0a.C2a.n_heldout"])
    add("S0-0a-03", "0a", "8.5",
        "The committed composition depends on the attribution screen, which missed Qwen3-8B's largest node; with the "
        "exhaustive sweep, the condition share of the primary contrast rises to 0.17 and the picture matches Gemma "
        "qualitatively: a layer-0 MLP at the condition span injects the instruction, and a late stack of fork-position "
        "MLPs and heads turns it into the fork logit. In Qwen3-8B that stack sits deeper (effect-weighted depth about "
        "0.86 against 0.69).",
        f"The committed composition depends on the attribution screen, which missed Qwen3-8B's largest node; with the "
        f"exhaustive sweep, the condition share of the primary contrast rises to {P('0a.motif.sweep_cond')}, still below "
        f"Gemma's band [{P('0a.band.cond.lo')}, {P('0a.band.cond.hi')}], and every minimum spanning tree row keeps a "
        f"fork-stack depth of {P('0a.motif.mst_depth_min')} to {P('0a.motif.mst_depth_max')}, above the band "
        f"[{P('0a.band.depth.lo')}, {P('0a.band.depth.hi')}] (the bands are constants of the committed motif code, "
        f"`src/k3motif.py` line {ctx['bands_line']}), so the post hoc sweep does not turn the committed different-motif "
        f"reading into a match {M}. What the two families share is a late stack of fork-position MLPs and heads that "
        f"carries the fork logit, deeper in Qwen3-8B (effect-weighted depth about {P('0a.motif.qwen_depth')} against "
        f"{P('0a.motif.gemma_depth')}); the layer-0 MLP at the condition span carries much of the forced contrast on the "
        f"20 pair prompts, but its held-out edit was never applied (see the paragraph after Table 395), and a structural "
        f"likeness or difference between discovered circuits is not by itself a likeness or difference of mechanism, the "
        f"phantom specialization that Bayat Makou et al. describe [146].",
        "The post hoc sweep was read as a qualitative match; recomputed, it still misses both committed bands",
        ["0a.motif.sweep_cond", "0a.band.cond.lo", "0a.band.cond.hi", "0a.motif.mst_depth_min",
         "0a.motif.mst_depth_max", "0a.band.depth.lo", "0a.band.depth.hi", "0a.motif.qwen_depth",
         "0a.motif.gemma_depth"])
    add("S0-0a-04", "0a", "8.9 Table 414",
        "| Shape of the circuit | Layer-0 condition MLP plus a late fork stack, depth about 0.86 | The same shape, depth "
        "0.69 | Section 8.5 |",
        f"| Shape of the circuit {M} | Late fork stack, depth {P('0a.motif.qwen_depth')}; condition share "
        f"{P('0a.motif.qwen_cond')}; committed motif test: {P('0a.motif.label').lower()} (condition share and depth "
        f"outside Gemma's bands [{P('0a.band.cond.lo')}, {P('0a.band.cond.hi')}] and [{P('0a.band.depth.lo')}, "
        f"{P('0a.band.depth.hi')}]); the layer-0 condition MLP rests on the 20 pair prompts (held-out edit not applied) "
        f"| Condition share {P('0a.motif.gemma_cond')}, depth {P('0a.motif.gemma_depth')} (motif match for its own "
        f"set) | Section 8.5 |",
        "Table 414 said 'the same shape'; the committed motif test says different motif",
        ["0a.motif.qwen_depth", "0a.motif.qwen_cond", "0a.motif.label", "0a.motif.gemma_cond", "0a.motif.gemma_depth",
         "0a.motif.gemma_label"])
    add("S0-0a-05", "0a", "8.10 Established",
        "carry the fork logit, with a layer-0 MLP at the condition span injecting the instruction, the same shape as "
        "Gemma's circuit.",
        f"carry the fork logit; the committed motif test reads {P('0a.motif.label').lower()} from Gemma's circuit "
        f"(condition share {P('0a.motif.qwen_cond')} against Gemma's band [{P('0a.band.cond.lo')}, "
        f"{P('0a.band.cond.hi')}]; fork-stack depth {P('0a.motif.qwen_depth')} against [{P('0a.band.depth.lo')}, "
        f"{P('0a.band.depth.hi')}]), and the layer-0 MLP at the condition span rests on the 20 pair prompts because its "
        f"held-out edit was never applied {M}.",
        "'Established' claimed Gemma's shape", ["0a.motif.label", "0a.motif.qwen_cond", "0a.motif.qwen_depth"])
    add("S0-0a-06", "0a", "What we have learned",
        "Path-patching circuits for the fork margin, validated against matched random components in the real model, "
        "have the same shape in both families: a layer-0 MLP at the condition span injects the instruction, and a late "
        "stack of fork-position MLPs and heads turns it into the fork logit, deeper in Qwen3-8B (effect-weighted depth "
        "about 0.86) than in Gemma (0.69); 9 of 10 and 8 of 10 Qwen3-8B nodes pass both tests.",
        f"Path-patching circuits for the fork margin, validated against matched random components in the real model, "
        f"show that a late stack of fork-position MLPs and heads carries the fork logit in both families; the committed "
        f"motif test reads {P('0a.motif.label').lower()} (condition share {P('0a.motif.qwen_cond')} against Gemma's band "
        f"[{P('0a.band.cond.lo')}, {P('0a.band.cond.hi')}]; fork-stack depth {P('0a.motif.qwen_depth')} against "
        f"[{P('0a.band.depth.lo')}, {P('0a.band.depth.hi')}]), and the post hoc sweep still misses both bands "
        f"(forced-Prim condition share {P('0a.motif.sweep_cond')}; every minimum spanning tree row depth "
        f"{P('0a.motif.mst_depth_min')} to {P('0a.motif.mst_depth_max')}). The layer-0 condition-span MLP rests on the "
        f"20 pair prompts ({P('0a.L0.abl.pair')} and {P('0a.L0.ste.pair')} nats under ablation and steering) plus the "
        f"descriptive natural contrast, because its held-out edit was never applied, and BASE (B3) reruns the held-out "
        f"arm; 9 of 10 and 8 of 10 Qwen3-8B nodes pass both tests {M}.",
        "The closing repeated 'the same shape'", ["0a.motif.label", "0a.motif.qwen_cond", "0a.motif.qwen_depth",
                                                  "0a.motif.sweep_cond", "0a.motif.mst_depth_min",
                                                  "0a.motif.mst_depth_max", "0a.L0.abl.pair", "0a.L0.ste.pair"])
    add("S0-0a-07", "0a", "8.10 Table 416",
        "but the matched random five also flip at dose 2 (7:1, p 0.070; 0:1); motif differs on condition share and depth |",
        f"but the matched random five also flip at dose 2 (7:1, p 0.070; 0:1); motif differs on condition share and "
        f"depth; the layer-0 condition MLP's held-out edit was never applied ({P('0a.L0.abl.heldout')} by construction), "
        f"so that node rests on its 20 pair prompts {M} |",
        "Table 416's circuit row gains the held-out note", ["0a.L0.abl.heldout"])

    # ============================================================================================ 0b DAS verdict
    das_sent = (f"Under the committed rule it is a sufficient handle in Gemma and {P('0b.T10.verdict')} at Qwen3-8B's "
                f"fork, where the edit toward BFS writes BFS in {P('0b.bfs.written')} of 30 programs but "
                f"{P('0b.bfs.fail')} fail (pass {P('0b.bfs.pass')} against {P('0b.bfs.base_pass')}; "
                f"{P('0b.bfs.mc_a')}:{P('0b.bfs.mc_b')} against random, p {P('0b.bfs.mc_p')}).")
    dasv = ["0b.T10.verdict", "0b.bfs.written", "0b.bfs.fail", "0b.bfs.pass", "0b.bfs.base_pass", "0b.bfs.mc_a",
            "0b.bfs.mc_b", "0b.bfs.mc_p"]
    add("S0-0b-01", "0b", "8.4 Verdict",
        "What the protocol finds in both families is a strong steering handle of low rank at the fork, not a certified "
        "variable: a certified variable would pass every control, while a steering handle moves the output without "
        "passing them.",
        f"What the protocol finds is a rank-4 edit at the fork that moves the written algorithm in both families, not a "
        f"certified variable: a certified variable would pass every control. {das_sent} Neither passes the specificity "
        f"controls {M}.",
        "The verdict paragraph called both forks a strong steering handle; the committed generation rule says NO HANDLE "
        "at Qwen3-8B's fork", dasv + ["0b.GRID.verdict"])
    add("S0-0b-02", "0b", "8.9",
        "the rank-4 DAS map at block 22 (29 of 30 correct), the top five circuit components at dose 2 (29 of 30)",
        f"the rank-4 DAS map at block 22 ({P('0b.dfs.correct')} of 30 correct toward DFS; toward BFS it writes BFS in "
        f"{P('0b.bfs.written')} of 30 programs but only {P('0b.bfs.correct')} correct, {P('0b.T10.verdict')} by the "
        f"committed rule) {M}, the top five circuit components at dose 2 (29 of 30)",
        "A DAS-fork-DFS '29 of 30' without the BFS direction", ["0b.dfs.correct", "0b.bfs.written", "0b.bfs.correct",
                                                                "0b.T10.verdict"])
    add("S0-0b-03", "0b", "8.9 Table 414",
        "| Steering handle at the fork in generation | Toward DFS 29 of 30 correct; toward BFS 29 of 30 written, 5 correct "
        "| Toward BFS 22 of 30 correct (twin 6) | Section 8.4 |",
        f"| Steering handle at the fork in generation | Toward DFS {P('0b.dfs.correct')} of 30 correct; toward BFS "
        f"{P('0b.bfs.written')} of 30 written, {P('0b.bfs.correct')} correct (pass {P('0b.bfs.pass')} against "
        f"{P('0b.bfs.base_pass')}): {P('0b.T10.verdict')} by the committed rule {M} | Toward BFS {P('0b.grid.das')} of "
        f"30 correct (twin {P('0b.grid.base')}): {P('0b.GRID.verdict').lower()} | Section 8.4 |",
        "Table 414's steering row lacked the committed verdicts", ["0b.dfs.correct", "0b.bfs.written", "0b.bfs.correct",
                                                                   "0b.bfs.pass", "0b.bfs.base_pass", "0b.T10.verdict",
                                                                   "0b.grid.das", "0b.grid.base", "0b.GRID.verdict"])
    add("S0-0b-04", "0b", "8.10 Established",
        "A rank-4 linear edit at the fork moves the written algorithm in both families, and under one protocol neither "
        "family's handle is a certified variable: permuted labels, an untrained network or matched random components "
        "reproduce most of each effect.",
        f"A rank-4 edit at the fork moves the written algorithm in both families. {das_sent} Neither passes the "
        f"specificity controls: permuted labels, an untrained network or matched random components reproduce most of "
        f"each effect {M}.", "'Established' pooled the two directions", dasv)
    add("S0-0b-05", "0b", "8.10 Table 416",
        "off-manifold overshoot at both forks; the band is a sufficient handle that fails the label control |",
        f"off-manifold overshoot at both forks; the band is a sufficient handle that fails the label control; in "
        f"generation Gemma's fork is a {P('0b.GRID.verdict').lower()} and Qwen3-8B's fork {P('0b.T10.verdict')} (toward "
        f"BFS {P('0b.bfs.written')} of 30 written, {P('0b.bfs.fail')} fail; pass {P('0b.bfs.pass')} against "
        f"{P('0b.bfs.base_pass')}) {M} |",
        "Table 416's DAS row lacked the per-site generation verdicts", ["0b.GRID.verdict", "0b.T10.verdict",
                                                                        "0b.bfs.written", "0b.bfs.fail", "0b.bfs.pass",
                                                                        "0b.bfs.base_pass"])
    add("S0-0b-06", "0b", "What we have learned",
        "At the fork itself, one DAS protocol with seven committed controls found a rank-4 handle in both families that "
        "writes the algorithm in generation (29 of 30 correct DFS on connected components; 22 of 30 correct BFS in Gemma "
        "against 6 unedited).",
        f"At the fork itself, one DAS protocol with seven committed controls found a rank-4 edit that moves the written "
        f"algorithm in both families ({P('0b.dfs.correct')} of 30 correct DFS on connected components; "
        f"{P('0b.grid.das')} of 30 correct BFS in Gemma against {P('0b.grid.base')} unedited). {das_sent[:-1]} {M}.",
        "The closing called it a handle in both families", dasv + ["0b.dfs.correct", "0b.grid.das", "0b.grid.base"])

    # ============================================================================================ 0c entropy and law
    ent = (f"the single highest-entropy position among the first 64 tokens on {P('0c.G.hits')} of {P('0c.G.n')} "
           f"(Gemma), {P('0c.B3.hits')} of {P('0c.B3.n')} (minimum spanning tree) and {P('0c.B6.hits')} of "
           f"{P('0c.B6.n')} (valid anagram) slots, but on {P('0c.T10.hits')} of {P('0c.T10.n')} (connected components) "
           f"and {P('0c.A3.hits')} of {P('0c.A3.n')} (trapped rain water), the two units where a highest-entropy rule "
           f"misses it")
    entv = [f"0c.{u}.{s}" for u in ("G", "B3", "B6", "T10", "A3") for s in ("hits", "n")]
    add("S0-0c-01", "0c", "8.3 entropy paragraph",
        "Code tokens are nearly deterministic, so the 90th percentile of the twin's per-token entropies is only 0.0003 to "
        "0.05 nats, and the fork's entropy at T 1 lies above it in every sample of every unit (median 0.13 nats on "
        "connected components and 0.53 to 0.81 nats on the mixed units). The fork is also the single highest-entropy "
        "position among the first 64 tokens on 44 of 44 Gemma slots that reach it, 32 of 39 minimum spanning tree slots "
        "and 21 of 39 valid anagram slots, but on only 2 of 50 connected components slots and 0 of 37 trapped rain water "
        "slots. A rule that doses the highest-entropy position therefore finds the fork on the mixed units (in Gemma it "
        "reproduces the fork-dosed arm exactly) and misses it on the collapsed one.",
        f"Code tokens are nearly deterministic, so the 90th percentile of the twin's per-token entropies is only "
        f"{P('0c.q90.min')} to {P('0c.q90.max')} nats, and the fork's entropy at T 1 lies above it in all "
        f"{P('0c.n_total')} twin samples that reach the fork on the five located units (median {P('0c.T10.median_H_rec')} "
        f"nats on connected components and {P('0c.mixed_median.min')} to {P('0c.mixed_median.max')} nats on the mixed "
        f"units). The fork is also {ent}. A rule that doses the highest-entropy position therefore finds the fork on most "
        f"slots of three of the four mixed units (on every Gemma slot, where it reproduces the fork-dosed arm exactly, but "
        f"on only {P('0c.B6.hits')} of {P('0c.B6.n')} valid anagram slots) and misses it on trapped rain water and on "
        f"the collapsed connected components {M}.",
        "The highest-entropy rule was said to find the fork on 'the mixed units'; it misses trapped rain water (0 of 37)",
        entv + ["0c.q90.min", "0c.q90.max", "0c.n_total", "0c.T10.median_H_rec", "0c.mixed_median.min",
                "0c.mixed_median.max"])
    add("S0-0c-02", "0c", "8.10 Table 416",
        "| Fork entropy prediction | Opposite holds | The fork lies above the 90th percentile of per-token entropy in every "
        "sample; it is the highest-entropy position on 44 of 44 Gemma slots but 2 of 50 on connected components |",
        f"| Fork entropy prediction | Opposite holds | The fork lies above the 90th percentile of per-token entropy in all "
        f"{P('0c.n_total')} twin samples that reach it; it is {ent} {M} |",
        "Table 416's entropy row carried a superlative", entv + ["0c.n_total"])
    add("S0-0c-03", "0c", "8.10 Established",
        "The fork is a high-entropy position relative to the near-deterministic code around it, though its entropy can "
        "be small in absolute terms (0.13 nats on connected components).",
        f"The fork's entropy lies above the 90th percentile of the twin's per-token entropies in all {P('0c.n_total')} "
        f"twin samples that reach it, though it can be small in absolute terms (median {P('0c.T10.median_H_rec')} nats "
        f"on connected components); it is {ent} {M}.",
        "'Established' needed the per-unit scope", entv + ["0c.n_total", "0c.T10.median_H_rec"])
    add("S0-0c-04", "0c", "What we have learned",
        "The fork is the highest-entropy position of the near-deterministic code around it in every sample, though its "
        "entropy can be small in absolute terms (median 0.13 nats on connected components).",
        f"The fork's entropy lies above the 90th percentile of per-token entropy in all {P('0c.n_total')} twin samples "
        f"that reach it, though it can be small in absolute terms (median {P('0c.T10.median_H_rec')} nats on connected "
        f"components); it is {ent} {M}.",
        "The closing claimed the highest-entropy position in every sample", entv + ["0c.n_total", "0c.T10.median_H_rec"])
    add("S0-0c-05", "0c", "What we have learned",
        "and at most 0.004 bits under iteration 8's blind locator, which adds a second fork inside impure branches",
        f"and at most {P('0c.law.Hmax5')} bits on the five Qwen3-8B and Gemma units of iteration 8's blind locator "
        f"({P('0c.law.H_llama')} bits on Llama's grid task, where the fork is reached through many openings) {M}, which "
        f"adds a second fork inside impure branches",
        "The 0.004-bit scope omitted Llama's 0.63 bits", ["0c.law.Hmax5", "0c.law.H_llama"])
    law = (f"on {P('0c.law.n_pass')} of {P('0c.law.n_scored')} units the fork tokens predict temperature's effect on the "
           f"algorithm mix, but the flat law is better on valid anagram ({P('0c.law.B6.flat')} against "
           f"{P('0c.law.B6.mae')}), algorithm-level tempering is better on substring search ({P('0c.law.GT04.temper')} "
           f"against {P('0c.law.G-T04.mae')}), the law beats the flat law on only {P('0c.law.beats_flat')} of 6 units "
           f"(sign test p {P('0c.law.sign_p')}), and on Llama's grid task every predictor fails ({P('0c.law.L-T1.mae')})")
    lawv = ["0c.law.n_pass", "0c.law.n_scored", "0c.law.B6.flat", "0c.law.B6.mae", "0c.law.GT04.temper",
            "0c.law.G-T04.mae", "0c.law.beats_flat", "0c.law.sign_p", "0c.law.L-T1.mae"]
    add("S0-0c-06", "0c", "8.9",
        "the temperature-law experiment predicted global temperature sweeps within 0.014 to 0.054 on five of six units, "
        "two of them in a held-out family, and both found nothing outside the catalogue.",
        f"the temperature-law experiment predicted global temperature sweeps within {P('0c.law.G-R6.mae')} to "
        f"{P('0c.law.B6.mae')} on five of six units, two of them in a held-out family, and both found nothing outside "
        f"the catalogue; {law} {M}.", "8.9 stated the law without its rivals", lawv + ["0c.law.G-R6.mae"])
    add("S0-0c-07", "0c", "8.10 Established",
        "and a law composed over one or two forks found blind predicts global temperature sweeps on held-out Gemma units "
        "(0.033 and 0.014) as well as on Qwen3-8B (0.029 to 0.054).",
        f"and a law composed over one or two forks found blind predicts global temperature sweeps on held-out Gemma units "
        f"({P('0c.law.G-T04.mae')} and {P('0c.law.G-R6.mae')}) as well as on Qwen3-8B ({P('0c.law.A3.mae')} to "
        f"{P('0c.law.B6.mae')}): {law} {M}.", "'Established' stated the law without its rivals",
        lawv + ["0c.law.G-R6.mae", "0c.law.A3.mae"])
    add("S0-0c-08", "0c", "8.10 Table 416",
        "| Temperature-law claim | Partial | 5 of 6 units pass including both held-out Gemma units; only 3 held-out units "
        "scored (4 required); Llama fails (0.142) |",
        f"| Temperature-law claim | Partial | {P('0c.law.n_pass')} of {P('0c.law.n_scored')} units pass including both "
        f"held-out Gemma units; only 3 held-out units scored (4 required); Llama fails ({P('0c.law.L-T1.mae')}); the "
        f"flat law beats the composed law on valid anagram ({P('0c.law.B6.flat')} against {P('0c.law.B6.mae')}) and "
        f"algorithm-level tempering on substring search ({P('0c.law.GT04.temper')} against {P('0c.law.G-T04.mae')}); "
        f"the law beats the flat law on {P('0c.law.beats_flat')} of 6 (sign test p {P('0c.law.sign_p')}) {M} |",
        "Table 416's law row omitted the rivals", lawv)
    add("S0-0c-09", "0c", "What we have learned",
        "predicted the minority share across T 0.3 to 1.3 within a mean total variation distance of 0.029 to 0.054 on "
        "three Qwen3-8B units and 0.033 and 0.014 on two held-out Gemma units.",
        f"predicted the minority share across T 0.3 to 1.3 within a mean total variation distance of "
        f"{P('0c.law.A3.mae')} to {P('0c.law.B6.mae')} on three Qwen3-8B units and {P('0c.law.G-T04.mae')} and "
        f"{P('0c.law.G-R6.mae')} on two held-out Gemma units; {law} {M}.", "The closing stated the law without rivals",
        lawv + ["0c.law.A3.mae", "0c.law.G-R6.mae"])

    # ============================================================================================ 0d positioning
    add("S0-0d-01", "0d", "8.10b (new)", "## 8.11 What the next iteration takes from this one",
        ctx["positioning_section"] + "\n\n## 8.11 What the next iteration takes from this one",
        "Iteration 8's claims were never positioned against the verified neighbours as a table", [])
    add("S0-0d-02", "0d", "8.7 base sub-arm",
        "The instruct default is therefore not the base model's mean: on minimum spanning tree weight",
        f"The instruct default is therefore not the base model's mean (that post-training shifts concentrate at the base "
        f"model's high-entropy forks is already shown by Cantarero et al. [142] and Sparse but Critical [143]; what this "
        f"sub-arm adds is the algorithm-level outcome on code at a counterfactually located fork) {M}: on minimum "
        f"spanning tree weight", "Base sub-arm stated without the neighbours that occupy the concentration result", [])
    add("S0-0d-03", "0d", "8.10 Established",
        "so the instruct default is a sharpened second or third choice of the base model, not its mean.",
        f"so the instruct default is a sharpened second or third choice of the base model, not its mean; the "
        f"concentration of post-training shifts at base-entropy forks is prior work (Cantarero et al. [142]; Sparse but "
        f"Critical [143]), and the contribution here is the algorithm-level reading on code {M}.",
        "'Established' needed the attribution", [])
    add("S0-0d-04", "0d", "What we have learned",
        "the instruct default is the base model's second or third choice at the fork, sharpened, not its mean answer.",
        f"the instruct default is the base model's second or third choice at the fork, sharpened, not its mean answer; "
        f"that post-training shifts concentrate at forks is shown by Cantarero et al. [142] and Sparse but Critical "
        f"[143], and the run adds the algorithm-level outcome on code at a counterfactually located fork {M}.",
        "The closing needed the attribution", [])
    last_ref = [l for l in report.rstrip("\n").split("\n") if l.startswith("[141] ")][0]
    add("S0-0d-05", "0d", "References", last_ref, last_ref + "\n" + "\n".join(f"{r} {M}" for r in ctx["new_refs"]),
        "References for the neighbours added by the positioning table", [])

    # ============================================================================================ 0e alias-named manifold
    zv = ctx["z_printed"]
    alias_facts = ("every bidirectional BFS alias contains 'BFS' or 'breadth-first', every bidirectional A* alias "
                   "contains 'A*', IDA*'s aliases contain 'A*', 'iterative deepening' or 'Manhattan heuristic', and jump "
                   "point search and unit-weight Dijkstra share no algorithm-specific term with any banned alias")
    add("S0-0e-01", "0e", "8.8 intro",
        "then placed that model's survivors on its own manifold of ten forced algorithms, in actual activation "
        "coordinates.",
        f"then placed that model's survivors on its own manifold of ten forced algorithms, in actual activation "
        f"coordinates; each forced prompt names its algorithm by one of three aliases, so the manifold is alias-named "
        f"{M}.", "The manifold's prompts name the algorithm", [])
    add("S0-0e-02", "0e", "8.8 classes",
        "and middle at the median; the landing statistic is the mean g over the correct catalogue survivors (Table 412).",
        f"and middle at the median; the landing statistic is the mean g over the correct catalogue survivors (Table "
        f"412). The manifold is built from 300 prompts that each name the algorithm by one of three aliases, so "
        f"'opposite' may reflect the distance between names: {alias_facts} {M}.",
        "The classes were stated without the alias caveat", [])
    add("S0-0e-03", "0e", "8.8 Table 412",
        "**Table 412.** Geodesic classes of the non-banned algorithms at block 31, k 15, and their stability under 1,000 "
        "bootstrap resamples of paraphrase groups.",
        f"**Table 412.** Geodesic classes of the non-banned algorithms at block 31, k 15, and their stability under 1,000 "
        f"bootstrap resamples of paraphrase groups, on the alias-named manifold {M}.", "Table caption", [])
    add("S0-0e-04", "0e", "8.8",
        "so edge and opposite are classes of geodesic distance on a manifold that is not a ring.",
        f"so edge and opposite are classes of geodesic distance on a manifold that is not a ring and whose prompts name "
        f"each algorithm by one of three aliases {M}.", "Alias caveat", [])
    add("S0-0e-05", "0e", "8.8 Landing",
        "and the committed verdict is opposite: the preregistered edge prediction is not supported.",
        f"and the committed verdict is opposite on the alias-named manifold {M}: the preregistered edge prediction is "
        f"not supported.", "Landing statement without the alias caveat", [])
    add("S0-0e-06", "0e", "8.8 alias analysis (new)",
        "**Held-out confirmation.** The fourth commit froze the block, k, the point set, the geodesic matrices,",
        ctx["alias_paragraph"] + "\n\n**Held-out confirmation.** The fourth commit froze the block, k, the point set of "
        f"alias-named prompts {M}, the geodesic matrices,",
        "The CPU computation answering the fifth MUST-FIX", ctx["alias_value_names"])
    add("S0-0e-07", "0e", "8.8",
        "Both lie above the 0.8 flag, and in Qwen3-8B's bf16 geometry unit-weight Dijkstra is also opposite the banned "
        "five.",
        f"Both lie above the 0.8 flag, and in Qwen3-8B's bf16 geometry unit-weight Dijkstra is also opposite the banned "
        f"five (that manifold is built from the same alias-named prompts) {M}.", "Alias caveat", [])
    add("S0-0e-08", "0e", "8.8 Other descriptive geometry",
        "and the off-catalogue survivors (Bellman-Ford, random walk, random-priority search) sit far from every class "
        "cloud (231 to 394).",
        f"and the off-catalogue survivors (Bellman-Ford, random walk, random-priority search) sit far from every class "
        f"cloud (231 to 394). At the one position whose choice does not depend on the label, the unit-Dijkstra survivors "
        f"therefore sit nearest a banned cloud (greedy best-first) rather than opposite the banned five, but that position "
        f"decodes the label at only {P('0e.M2.g16')} against chance 0.25 {M}.", "Connects the F5 fact to the alias caveat", ["0e.M2.g16"])
    add("S0-0e-09", "0e", "8.8 audits",
        "and gives the exact uniform p for opposite as 6.4e-5, (1/5) to the sixth.",
        f"and gives the exact uniform p for opposite as 6.4e-5, (1/5) to the sixth, on the alias-named manifold {M}.",
        "Alias caveat", [])
    add("S0-0e-10", "0e", "8.9",
        "and its catalogue survivors are all unit-weight Dijkstra, the farthest non-banned algorithm from the banned "
        "five, on the screen and on held-out templates.",
        f"and its catalogue survivors are all unit-weight Dijkstra, the farthest non-banned algorithm from the banned "
        f"five, on the screen and on held-out templates. That manifold is built from 300 prompts that each name the "
        f"algorithm by one of three aliases, and {ctx['alias_short']} {M}.", "Alias caveat with the computed result",
        ctx["alias_value_names"])
    add("S0-0e-11", "0e", "8.10 Table 416",
        "| Top-five prohibition claim | One configuration obeys; survivors land opposite, not at the edge; not robust; "
        "confirmed on held-out templates |",
        f"| Top-five prohibition claim | One configuration obeys; survivors land opposite, not at the edge, on the "
        f"alias-named manifold, where names and mechanisms are confounded (Section 8.8) {M}; not robust; confirmed on "
        f"held-out templates |", "Alias caveat", [])
    add("S0-0e-12", "0e", "8.10 Established",
        "and its catalogue survivors are all the algorithm farthest from the banned five on its manifold.",
        f"and its catalogue survivors are all the algorithm farthest from the banned five on its alias-named manifold, "
        f"where {ctx['alias_short']} {M}.", "Alias caveat", ctx["alias_value_names"])
    add("S0-0e-13", "0e", "8.10 Ruled out",
        "The edge prediction for the survivors of the top-five prohibition on the model that obeys.",
        f"The edge prediction for the survivors of the top-five prohibition on the model that obeys, on the alias-named "
        f"manifold {M}.", "Alias caveat", [])
    add("S0-0e-14", "0e", "8.11",
        "tasks where an obeying model writes several catalogue survivors are needed for the edge-against-opposite "
        "question.",
        f"tasks where an obeying model writes several catalogue survivors are needed for the edge-against-opposite "
        f"question, on a manifold whose prompts do not name the algorithm (TP-G), because on the alias-named manifold "
        f"names and mechanisms are confounded {M}.", "Third lead needs a name-free manifold", [])
    add("S0-0e-15", "0e", "What we have learned",
        "Its catalogue survivors are all unit-weight Dijkstra, the farthest non-banned algorithm from the banned five on "
        "its manifold of forced prompts at block 31, so they land on the opposite side, not at the edge as preregistered,",
        f"Its catalogue survivors are all unit-weight Dijkstra, the farthest non-banned algorithm from the banned five on "
        f"its manifold of forced prompts at block 31, so they land on the opposite side, not at the edge as "
        f"preregistered; that manifold is built from prompts that each name the algorithm by one of three aliases, and "
        f"{ctx['alias_short']} {M},", "Alias caveat with the computed result", ctx["alias_value_names"])
    add("S0-0e-16", "0e", "What we have learned",
        "Third, the edge-against-opposite question of the top-five prohibition needs tasks where an obeying model writes "
        "several catalogue survivors.",
        f"Third, the edge-against-opposite question of the top-five prohibition needs tasks where an obeying model writes "
        f"several catalogue survivors, on manifolds whose prompts do not name the algorithm (TP-G), because the alias-named "
        f"manifold confounds names with mechanisms {M}.", "Open question needs a name-free manifold", [])

    # ============================================================================================ 0f Gemma comparators
    add("S0-0f-01", "0f", "8.3",
        "The filtered dial beats global T by +0.08 (p 0.23) and min-p, entropy-triggered and block-start temperature "
        "with p 0.016 to 0.023.",
        f"The filtered dial beats the twin-level fallback of the entropy-triggered family (p {P('0f.G.EDT.p')}) and ties "
        f"global T (+{P('0f.G.globalT.diff')}, p {P('0f.G.globalT.p')}). Min-p and block-start temperature were not run "
        f"on Gemma, and the entropy-triggered arm that was run costs {P('0f.G.A5.pass_cost')} in pass rate, above the "
        f"5-point (0.05) rule, so every family except global T is scored at the twin's Hill of {P('0f.G.twin_hill')} "
        f"(`boot_family_best`, `src/analysis_k1.py`) {M}.",
        "Min-p and block-start were never run on Gemma; their 'wins' were twin-level fallbacks",
        ["0f.G.EDT.p", "0f.G.globalT.diff", "0f.G.globalT.p", "0f.G.A5.pass_cost", "0f.G.twin_hill"])

    # ============================================================================================ 0g figure markers
    for m in ctx["markers"]:
        add(m["edit_id"], "0g", m["section"], m["anchor"], m["anchor"] + "\n\n" + m["block"], m["why"], [])
    return E


## Apply test and lint: `src/apply_lint.py` (unchanged)

- **The apply test** passes when:
  - each `old_text` occurs exactly once and is at least 40 characters long;
  - each `new_text` carries the marker;
  - no two spans overlap;
  - after application, every `new_text` is present and no `old_text` survives.
- **The lint (L1–L10)** is a set of regex rules for the residual claims the review flagged. For example, L1 is "same
  shape" near "circuit" or "Gemma"; L4 is a "29 of 30" DAS claim without the BFS failure markers; L8 is "opposite" from
  8.8 on without the alias caveat. Hits in pre-iteration-8 text, or in 8.1's prediction statements, are allowlisted
  with a reason.

In [ ]:
%%writefile eval_demo_run/art_iYh9zVP7myjJ/src/apply_lint.py
#!/usr/bin/env python3
"""Apply test (exact-once old_text, no overlapping spans, every new_text present after application, unified diff)
and the whole-report residual-claim lint L1-L10 with a reasoned allowlist."""
from __future__ import annotations

import difflib
import re
from typing import Any

MARK = "(iteration-9 correction)"
ITER8_START_HEADING = "# Iteration 8: temperature at the fork"
PREDICTION_LINE_PREFIX = "- *The top-five prohibition claim*"


# ================================================================================================ apply
def apply_edits(original: str, edits: list[dict[str, Any]]) -> tuple[str, dict[str, Any]]:
    problems: list[str] = []
    spans = []
    for e in edits:
        n = original.count(e["old_text"])
        if n != 1:
            problems.append(f"{e['id']}: old_text occurs {n} times")
            continue
        if len(e["old_text"]) < 40:
            problems.append(f"{e['id']}: old_text shorter than 40 characters")
        if MARK not in e["new_text"]:
            problems.append(f"{e['id']}: new_text lacks the iteration-9 marker")
        s = original.index(e["old_text"])
        spans.append((s, s + len(e["old_text"]), e))
    spans.sort(key=lambda t: t[0])
    for (s1, e1, a), (s2, e2, b) in zip(spans, spans[1:]):
        if s2 < e1:
            problems.append(f"overlap: {a['id']} [{s1},{e1}) and {b['id']} [{s2},{e2})")
    if problems:
        return original, {"apply_test_pass": False, "problems": problems, "n_edits": len(edits)}
    out, last = [], 0
    for s, e, ed in spans:
        out.append(original[last:s])
        out.append(ed["new_text"])
        last = e
    out.append(original[last:])
    corrected = "".join(out)
    missing = [e["id"] for e in edits if e["new_text"] not in corrected]
    gone = [e["id"] for e in edits if e["old_text"] in corrected and e["old_text"] not in e["new_text"]]
    ok = not missing and not gone
    return corrected, {"apply_test_pass": ok, "problems": [f"new_text missing: {m}" for m in missing] +
                       [f"old_text survives: {g}" for g in gone], "n_edits": len(edits),
                       "line_of_edit": {ed["id"]: original[:s].count("\n") + 1 for s, _, ed in spans}}


def unified_diff(a: str, b: str) -> str:
    return "".join(difflib.unified_diff(a.splitlines(keepends=True), b.splitlines(keepends=True),
                                        fromfile="inputs/report/current_report.md", tofile="report_corrected.md", n=1))


# ================================================================================================ lint
def paragraphs(text: str) -> list[dict[str, Any]]:
    """Blocks separated by blank lines; a '**Table N.**' caption block is merged with the table block after it."""
    lines = text.split("\n")
    blocks, cur, start = [], [], 0
    for i, l in enumerate(lines):
        if l.strip() == "":
            if cur:
                blocks.append({"start": start + 1, "text": "\n".join(cur)})
                cur = []
        else:
            if not cur:
                start = i
            cur.append(l)
    if cur:
        blocks.append({"start": start + 1, "text": "\n".join(cur)})
    merged = []
    for b in blocks:
        if merged and merged[-1]["text"].startswith("**Table") and b["text"].lstrip().startswith("|") and \
                "\n" not in merged[-1]["text"]:
            merged[-1] = {"start": merged[-1]["start"], "text": merged[-1]["text"] + "\n" + b["text"]}
        else:
            merged.append(b)
    heading = ""
    for b in merged:
        if b["text"].startswith("#"):
            heading = b["text"].split("\n")[0].lstrip("# ").strip()
        b["section"] = heading
    return merged


def sentences(par: str) -> list[tuple[int, str]]:
    out, pos = [], 0
    for m in re.finditer(r"[^.!?]*(?:\d\.\d[^.!?]*)*[.!?](?=\s|$)|[^.!?]+$", par):
        out.append((m.start(), m.group(0)))
        pos = m.end()
    return out or [(0, par)]


def _sent_split(par: str) -> list[tuple[int, str]]:
    """Split on '. ' boundaries that are not decimal points or abbreviations; keeps offsets."""
    res, last = [], 0
    for m in re.finditer(r"(?<=[a-z0-9)\]'\"*])[.;]\s+(?=[A-Z*(\[])|\n", par):
        res.append((last, par[last:m.end()]))
        last = m.end()
    res.append((last, par[last:]))
    return [r for r in res if r[1].strip()]


BFS_MARKERS = re.compile(r"5 correct|5 of 30|24 fail|24 of them fail|pass 0\.20|0\.20 against 0\.83|NO HANDLE")
COUNTS = re.compile(r"\d+ of \d+|\d+/\d+")


def lint(text: str) -> list[dict[str, Any]]:
    hits: list[dict[str, Any]] = []
    iter8_line = next((i + 1 for i, l in enumerate(text.split("\n")) if l.startswith(ITER8_START_HEADING)), 10 ** 9)
    s88 = next((i + 1 for i, l in enumerate(text.split("\n")) if l.startswith("## 8.8 ")), 10 ** 9)

    def add(pid: str, b: dict[str, Any], off: int, snippet: str) -> None:
        line = b["start"] + b["text"][:off].count("\n")
        hits.append({"pattern": pid, "line": line, "section": b["section"], "snippet": snippet[:240].replace("\n", " "),
                     "iteration8_text": line >= iter8_line})

    for b in paragraphs(text):
        t = b["text"]
        low = t.lower()
        # L1
        for m in re.finditer(r"same shape", t):
            w = t[max(0, m.start() - 300): m.end() + 300]
            if "circuit" in w or "Gemma" in w:
                add("L1", b, m.start(), t[max(0, m.start() - 120): m.end() + 80])
        # L2
        for m in re.finditer(r"in every sample", t):
            w = t[max(0, m.start() - 200): m.end() + 200]
            if "highest-entropy" in w:
                add("L2", b, m.start(), t[max(0, m.start() - 120): m.end() + 120])
        sents = _sent_split(t)
        for off, s in sents:
            sl = s.lower()
            # L3
            if "in both families" in s and re.search(r"handle|edit", sl) and re.search(r"move|writ", sl) and \
                    "NO HANDLE" not in t:
                add("L3", b, off, s)
            # L5
            if "0.004 bits" in s and not ("five Qwen3-8B and Gemma units" in s and "0.63" in s):
                add("L5", b, off, s)
            # L6
            if "highest-entropy position" in s and re.search(r"\bevery (sample|slot|unit|case)|all (slots|samples|units)",
                                                            s) and not COUNTS.search(s):
                add("L6", b, off, s)
            # L7
            if "min-p" in s and "block-start" in s and ("Gemma" in s or "Gemma" in t[:off]) and \
                    re.search(r"\bp \d", s) and "not run" not in s:
                add("L7", b, off, s)
        # L4
        if re.search(r"DAS|rank-4|block 22", t) and "DFS" in t and not BFS_MARKERS.search(t):
            for m in re.finditer(r"29 of 30|29/30", t):
                near = low[max(0, m.start() - 120): m.end() + 60]
                if "counting sort" in near or "top five" in near or "full-state" in near or "circuit" in near:
                    continue
                add("L4", b, m.start(), t[max(0, m.start() - 160): m.end() + 80])
        # L8 (8.8 to the end)
        if b["start"] >= s88 and re.search(r"\bopposite\b|farthest non-banned", low) and not re.search(
                r"alias|named", low):
            m = re.search(r"\bopposite\b|farthest non-banned", low)
            add("L8", b, m.start(), t[max(0, m.start() - 160): m.end() + 80])
        # L9
        if b["start"] >= iter8_line:
            for m in re.finditer(r"not its mean|not the base model's mean", t):
                if "Cantarero" not in t and "Sparse but Critical" not in t:
                    add("L9", b, m.start(), t[max(0, m.start() - 160): m.end() + 60])
        # L10
        if re.search(r"predict|\blaw\b", low) and re.search(r"five of six|5 of 6", low) and not (
                "flat" in low and "tempering" in low):
            m = re.search(r"five of six|5 of 6", low)
            add("L10", b, m.start(), t[max(0, m.start() - 160): m.end() + 80])
    return hits


def allow_reason(h: dict[str, Any], text_lines: list[str]) -> str | None:
    if not h["iteration8_text"]:
        return "pre-iteration-8 text: outside the iteration-8 record under review (earlier iterations were audited by " \
               "their own Step 0)"
    line = text_lines[h["line"] - 1] if h["line"] - 1 < len(text_lines) else ""
    if line.startswith(PREDICTION_LINE_PREFIX) or line.startswith("- *The circuit claim*") or \
            line.startswith("- *The temperature-law claim*") or line.startswith("- *The fork-dosing claim*"):
        return "iteration-8 prediction statement (8.1): records what was preregistered, not a result"
    return None


def lint_report(text: str) -> dict[str, Any]:
    hits = lint(text)
    lines = text.split("\n")
    allowed, live = [], []
    for h in hits:
        r = allow_reason(h, lines)
        (allowed if r else live).append({**h, "allow_reason": r})
    by = {}
    for h in hits:
        by.setdefault(h["pattern"], 0)
        by[h["pattern"]] += 1
    by_live = {}
    for h in live:
        by_live.setdefault(h["pattern"], 0)
        by_live[h["pattern"]] += 1
    return {"n_hits": len(hits), "by_pattern": by, "n_unallowed": len(live), "unallowed_by_pattern": by_live,
            "unallowed": live, "allowlisted": allowed}


## The two audits: `audit/rederive.py` and `audit/headline_rederive.py` (unchanged)

Both use only the standard library and share no code with `eval.py` or `src/`; they have their own `.npz` reader,
ranks, Spearman and permutation enumeration.
- `rederive.py` re-checks every ledger value that has a plain key path against its source file and recomputes item 0e's
  statistics. It also re-applies the edits and runs 4 placebos that must fail.
- `headline_rederive.py` re-derives the headline numbers from the raw per-row files, with its own placebos.

`eval.py` runs both as subprocesses.

In [ ]:
%%writefile eval_demo_run/art_iYh9zVP7myjJ/audit/rederive.py
#!/usr/bin/env python3
"""Independent re-derivation of the Step-0 numbers with the Python standard library only.

Shares no code with eval.py or src/: its own .npz/.npy reader (zipfile + header parsing + array), its own JSON key
navigation, its own ranks, Spearman, exact permutation enumeration, apply check and L1 lint. Run after eval.py:
    python -I audit/rederive.py
Writes audit/rederive_result.json {n_checks, n_mismatches, placebos_failed, checks[]}.
"""
import array
import ast
import hashlib
import itertools
import json
import math
import random
import re
import statistics
import sys
import zipfile
from pathlib import Path

ROOT = Path(__file__).resolve().parent.parent
DEPS = Path(__import__("os").environ.get("S0_DEPS_DIR", ROOT.parents[2] / "iter_8" / "gen_art"))
ARTS = {"art_vnyi0Fm5SyNH": "gen_art_experiment_39", "art_e-_q5mSJGrSb": "gen_art_experiment_40",
        "art_18XvfcJJZi1O": "gen_art_experiment_41", "art_FJMIE9T9_PMv": "gen_art_experiment_42",
        "art_lzuosbrb34wB": "gen_art_experiment_43", "art_R-HSmXq2t97f": "gen_art_experiment_44"}
CHECKS = []
_cache = {}


def apath(art, rel):
    for base in (DEPS, ROOT.parent):
        for name in (art, ARTS[art]):
            if (base / name).is_dir():
                return base / name / rel
    return DEPS / ARTS[art] / rel


def jload(art, rel):
    k = (art, rel)
    if k not in _cache:
        _cache[k] = json.loads(apath(art, rel).read_text())
    return _cache[k]


def check(name, expected, got, kind="independent", tol=1e-9):
    if isinstance(expected, str) or isinstance(got, str):
        ok = str(expected) == str(got)
    elif expected is None or got is None:
        ok = expected is got
    else:
        ok = abs(float(expected) - float(got)) <= tol * max(1.0, abs(float(expected)))
    CHECKS.append({"name": name, "expected": expected, "got": got, "ok": bool(ok), "kind": kind})
    return ok


# ------------------------------------------------------------------------------------------------ npz reader
def npy_member(zpath, member):
    with zipfile.ZipFile(zpath) as z:
        raw = z.read(member + ".npy")
    if raw[:6] != b"\x93NUMPY":
        raise ValueError("not an npy member")
    major = raw[6]
    if major == 1:
        hlen = int.from_bytes(raw[8:10], "little")
        off = 10
    else:
        hlen = int.from_bytes(raw[8:12], "little")
        off = 12
    header = ast.literal_eval(raw[off:off + hlen].decode("latin1"))
    data = raw[off + hlen:]
    descr, shape = header["descr"], header["shape"]
    n = 1
    for s in shape:
        n *= s
    if descr in ("<f8", "<i8"):
        arr = array.array("d" if descr == "<f8" else "q")
        arr.frombytes(data[: n * 8])
        if sys.byteorder != "little":
            arr.byteswap()
        out = list(arr)
    elif descr.startswith("<U"):
        w = int(descr[2:])
        out = [data[i * 4 * w:(i + 1) * 4 * w].decode("utf-32-le").rstrip("\x00") for i in range(n)]
    else:
        raise ValueError(f"unsupported dtype {descr}")
    return out, shape


# ------------------------------------------------------------------------------------------------ key navigation
def nav(obj, key):
    for part in re.findall(r"[^.\[\]]+|\[[^\]]*\]", key):
        if part.startswith("["):
            sel = part[1:-1]
            if "=" in sel:
                f, v = sel.split("=", 1)
                obj = [e for e in obj if str(e.get(f)) == v][0]
            else:
                obj = obj[int(sel)]
        else:
            obj = obj[part]
    return obj


def resolve_simple(src):
    art, rest = src.split(":", 1)
    rel, key = rest.split("#", 1)
    key = re.sub(r"\s*\(=.*\)$", "", key).strip()
    if art not in ARTS or not rel.endswith(".json") or re.search(r"[*{} ]|->|-n_heldout|- correct|1-twin", key):
        return None
    return nav(jload(art, rel), key)


# ------------------------------------------------------------------------------------------------ statistics
def ranks(x):
    order = sorted(range(len(x)), key=lambda i: x[i])
    r = [0.0] * len(x)
    i = 0
    while i < len(x):
        j = i
        while j + 1 < len(x) and x[order[j + 1]] == x[order[i]]:
            j += 1
        for k in range(i, j + 1):
            r[order[k]] = (i + j) / 2.0 + 1.0
        i = j + 1
    return r


def spearman(x, y):
    rx, ry = ranks(x), ranks(y)
    mx, my = sum(rx) / len(rx), sum(ry) / len(ry)
    sxy = sum((a - mx) * (b - my) for a, b in zip(rx, ry))
    sx = math.sqrt(sum((a - mx) ** 2 for a in rx))
    sy = math.sqrt(sum((b - my) ** 2 for b in ry))
    return sxy / (sx * sy) if sx > 0 and sy > 0 else float("nan")


def arrangements(values):
    vals = sorted(set(values))
    counts = [values.count(v) for v in vals]
    n = len(values)

    def rec(free, k, cur):
        if k == len(vals):
            yield list(cur)
            return
        for pos in itertools.combinations(free, counts[k]):
            for p in pos:
                cur[p] = vals[k]
            yield from rec([f for f in free if f not in pos], k + 1, cur)

    yield from rec(list(range(n)), 0, [None] * n)


def exact_null(values):
    """rho of g-ranks 1..n against every distinct arrangement (rank vector of the arrangement precomputed per value)."""
    rv = ranks(values)
    rmap = {}
    for v, r in zip(values, rv):
        rmap[v] = r
    n = len(values)
    gr = [i + 1 - (n + 1) / 2 for i in range(n)]
    mean_r = sum(rv) / n
    sr = math.sqrt(sum((r - mean_r) ** 2 for r in rv))
    sg = math.sqrt(sum(g * g for g in gr))
    out = []
    for arr in arrangements(list(values)):
        out.append(sum((rmap[a] - mean_r) * g for a, g in zip(arr, gr)) / (sr * sg))
    return out


def p_lower(rho, null):
    return sum(1 for v in null if v <= rho + 1e-12) / len(null)


# ------------------------------------------------------------------------------------------------ main checks
def main():
    corr = json.loads((ROOT / "corrections.json").read_text())
    ledger = {v["name"]: v for v in corr["values"]}
    s0e = json.loads((ROOT / "results/s0e.json").read_text())

    # (1) generic ledger values with plain JSON key paths (every MATCH, NEW and CORRECTED value that has one)
    n_generic = 0
    for name, v in ledger.items():
        if v["source"].startswith("S0:"):
            continue
        try:
            got = resolve_simple(v["source"])
        except (KeyError, IndexError, ValueError):
            got = None
        if got is None:
            continue
        n_generic += 1
        check(f"ledger:{name}", v["recomputed"], got)

    # (2) composite ledger values, recomputed by hand
    circ = "art_18XvfcJJZi1O"
    S = jload(circ, "results/summary.json")
    V = S["validation"]
    check("ledger:0a.C1forced.n_pair", ledger["0a.C1forced.n_pair"]["recomputed"],
          V["C1forced"]["n_prompts"] - V["C1forced"]["n_heldout"])
    vrows = [json.loads(l) for l in apath(circ, "data/validation_rows_C1forced.jsonl").read_text().splitlines() if l]
    pair_t = {r["template"] for r in vrows if not r["heldout"]}
    held = [r for r in vrows if r["heldout"]]
    check("ledger:0a.cond_matches", ledger["0a.cond_matches"]["recomputed"],
          sum(1 for r in held if r["template"] in pair_t))
    zero = set()
    ident_ok = True
    for c in ("C1forced", "C2a", "C1nat"):
        nh = V[c]["n_heldout"]
        for mode in ("ablate", "steer"):
            for r in V[c]["per_node"][mode]:
                lhs = r["mean_d"] * r["n"]
                rhs = r["mean_d_pair_prompts"] * (r["n"] - nh) + r["mean_d_heldout"] * nh
                ident_ok &= abs(lhs - rhs) <= 1e-6 * max(1, abs(lhs))
                if r["mean_d_heldout"] == 0.0:
                    zero.add((c, r["node"]))
    check("0a.identity_all_nodes", True, ident_ok)
    check("ledger:0a.n_zero_nodes", ledger["0a.n_zero_nodes"]["recomputed"], len(zero))
    check("0a.zero_node_is_L0_COND", "C1forced|mlp|L0|H-1|COND", "|".join(sorted(zero)[0]) if len(zero) == 1 else "")
    l0 = [r for r in V["C1forced"]["per_node"]["ablate"] if r["node"] == "mlp|L0|H-1|COND"][0]
    check("0a.L0.worked_example", round(8.144148281097412 * 20 / 30, 6), round(l0["mean_d_pair_prompts"] * 20 / 30, 6))
    raw_zero = 0
    for r in held:
        for mode in ("ablate", "steer"):
            vals = [x for k, x in r["res"].items() if k.startswith(f"{mode}|mlp|L0|H-1|COND|")]
            raw_zero += int(len(vals) == 51 and all(x == 0.0 for x in vals))
    check("0a.raw_rows_unapplied_edit", 20, raw_zero)
    depths = []
    for c in ("C1nat", "C1forced", "C1mix", "C1natB"):
        q = S["motif"]["qwen"][c]
        for kk in ("dec_stack", "extended_top60_dec_stack"):
            if kk in q and q[kk] and q[kk].get("depth") is not None:
                depths.append(q[kk]["depth"])
    check("ledger:0a.motif.mst_depth_min", ledger["0a.motif.mst_depth_min"]["recomputed"], min(depths))
    check("ledger:0a.motif.mst_depth_max", ledger["0a.motif.mst_depth_max"]["recomputed"], max(depths))
    src_line = [l for l in apath(circ, "src/k3motif.py").read_text().splitlines() if l.startswith("BANDS =")][0]
    bands = ast.literal_eval(src_line.split("=", 1)[1].strip())
    check("ledger:0a.band.cond.lo", ledger["0a.band.cond.lo"]["recomputed"], bands["COND"][0])
    check("ledger:0a.band.cond.hi", ledger["0a.band.cond.hi"]["recomputed"], bands["COND"][1])
    check("ledger:0a.band.depth.lo", ledger["0a.band.depth.lo"]["recomputed"], bands["depth"][0])
    check("ledger:0a.band.depth.hi", ledger["0a.band.depth.hi"]["recomputed"], bands["depth"][1])
    das = jload("art_e-_q5mSJGrSb", "results/analysis.json")
    bfs = das["gen_T10_dec"]["classes"]["bfs"]["arms"]["DAS"]
    check("ledger:0b.bfs.fail", ledger["0b.bfs.fail"]["recomputed"], bfs["attempted_target"] - bfs["correct_target"])
    check("0b.mcnemar_exact_5_0", ledger["0b.bfs.mc_p"]["recomputed"], 2 * 0.5 ** 5)
    k1 = jload("art_vnyi0Fm5SyNH", "results/k1_verdict.json")["K1c"]
    dose = jload("art_vnyi0Fm5SyNH", "predictions/commit1_dose_table.json")["units"]
    med = {}
    with apath("art_vnyi0Fm5SyNH", "data/samples.jsonl").open() as f:
        for line in f:
            r = json.loads(line)
            if r.get("arm") == "A0" and r.get("stage") == "S0" and r.get("reached_decision") and \
                    r.get("decision_H1") is not None:
                med.setdefault(r["unit"], []).append(r["decision_H1"])
    tot = 0
    for u in ("G", "B3", "B6", "T10", "A3"):
        check(f"ledger:0c.{u}.above", ledger[f"0c.{u}.above"]["recomputed"], 1 - k1[u]["twin_dec_H_below_q90_frac"])
        check(f"ledger:0c.{u}.median_H", ledger[f"0c.{u}.median_H"]["recomputed"], statistics.median(med[u]))
        check(f"ledger:0c.{u}.q90_direct", ledger[f"0c.{u}.q90"]["recomputed"], dose[u]["q90"])
        tot += k1[u]["n"]
    check("ledger:0c.n_total", ledger["0c.n_total"]["recomputed"], tot)
    q = [dose[u]["q90"] for u in ("G", "B3", "B6", "T10", "A3")]
    check("ledger:0c.q90.min", ledger["0c.q90.min"]["recomputed"], min(q))
    check("ledger:0c.q90.max", ledger["0c.q90.max"]["recomputed"], max(q))
    mixed = [statistics.median(med[u]) for u in ("B3", "A3", "B6", "G")]
    check("ledger:0c.mixed_median.min", ledger["0c.mixed_median.min"]["recomputed"], min(mixed))
    check("ledger:0c.mixed_median.max", ledger["0c.mixed_median.max"]["recomputed"], max(mixed))
    law = jload("art_lzuosbrb34wB", "results/analysis_L.json")["verdict"]["per_unit"]
    check("ledger:0c.law.Hmax5", ledger["0c.law.Hmax5"]["recomputed"],
          max(law[u]["H"] for u in ("A3", "B6", "B3", "G-T04", "G-R6")))
    cells = jload("art_vnyi0Fm5SyNH", "results/k1_cells.json")
    check("0f.G.arms_missing", "A2,A4,A6", ",".join(a for a in ("A1", "A2", "A3g", "A4", "A5", "A6") if a not in cells["G"]))
    check("0f.A3.arms_missing", "A4,A6", ",".join(a for a in ("A1", "A2", "A3g", "A4", "A5", "A6") if a not in cells["A3"]))
    check("0f.B6.arms_missing", "A4,A6", ",".join(a for a in ("A1", "A2", "A3g", "A4", "A5", "A6") if a not in cells["B6"]))

    # (3) item 0e from the raw npz files
    tp = "art_R-HSmXq2t97f"
    gz = apath(tp, "cache/states/geo_C6_nf4.npz")
    sz = apath(tp, "cache/states/smean_C6_nf4.npz")
    labels, _ = npy_member(gz, "labels")
    if sz.exists():
        kind, _ = npy_member(sz, "kind")
        alg, _ = npy_member(sz, "algorithm")
        grp, _ = npy_member(sz, "group")
    else:   # published copy without the 181 MB states file: the vendored index of the same rows
        ix = json.loads((ROOT / "inputs/forced_index_C6_nf4.json").read_text())
        kind, alg, grp = ix["kind"], ix["algorithm"], ix["group"]
    F = [i for i, k in enumerate(kind) if k == "forced"]
    alg_f = [alg[i] for i in F]
    grp_f = [int(grp[i]) for i in F]
    check("0e.E1_labels_equal", True, alg_f == labels)
    common = (apath(tp, "src/common.py")).read_text()
    tree = ast.parse(common)
    lit = {}
    for node in tree.body:
        if isinstance(node, (ast.Assign, ast.AnnAssign)):
            tgt = node.targets[0] if isinstance(node, ast.Assign) else node.target
            if isinstance(tgt, ast.Name) and tgt.id in ("ALIAS_TABLE", "CANON_ORDER"):
                lit[tgt.id] = ast.literal_eval(node.value)
    canon = lit["CANON_ORDER"]
    expect = sorted((a, t, j) for a in canon for t in range(10) for j in range(3))
    check("0e.E2_multiset", True, sorted(zip(alg_f, [g // 3 for g in grp_f], [g % 3 for g in grp_f])) == expect)
    banned = ["bfs", "dfs", "iddfs", "astar", "gbfs"]
    nonb = ["bibfs", "dijkstra_unit", "idastar", "jps", "biastar"]
    m1 = jload(tp, "results/geometry_M1.json")["M1"]
    det = [r"\bbfs\b", r"breadth[-_ ]?first", r"\bdfs\b", r"depth[-_ ]?first", r"\biddfs\b", r"iterative[-_ ]?deepening",
           r"a\*|a[-_ ]?star|astar", r"greedy", r"best[-_ ]?first", r"manhattan", r"heuristic"]
    prim = [sum(1 for d in det if re.search(d, s, re.I)) for a in nonb for s in lit["ALIAS_TABLE"][a]]
    check("0e.primary_vector", "1,1,1,0,0,0,1,2,3,0,0,0,1,1,1", ",".join(map(str, prim)))
    null1 = exact_null([float(x) for x in prim])
    check("0e.null_size", 360360, len(null1))
    for k in (15, 10):
        geo, shape = npy_member(gz, f"geo_k{k}")
        N = shape[0]

        def mean_block(I, J):
            return sum(geo[i * N + j] for i in I for j in J) / (len(I) * len(J))

        bidx = {b: [i for i in range(N) if labels[i] == b] for b in banned}
        galg = {a: min(mean_block([i for i in range(N) if labels[i] == a], bidx[b]) for b in banned) for a in nonb}
        for a in nonb:
            check(f"0e.k{k}.g_alg.{a}", m1[f"k{k}"]["g"][a], galg[a], tol=1e-9)
        gal = []
        for a in nonb:
            for j in range(3):
                I = [i for i in range(N) if labels[i] == a and grp_f[i] % 3 == j]
                gal.append(min(mean_block(I, bidx[b]) for b in banned))
        for (a, j), gv in zip([(a, j) for a in nonb for j in range(3)], gal):
            check(f"0e.k{k}.g_alias.{a}|{j}", s0e["per_k"][f"k{k}"]["g_alias"][f"{a}|{j}"], gv)
        rho = spearman(gal, [float(x) for x in prim])
        check(f"ledger:0e.k{k}.rho_primary", ledger[f"0e.k{k}.rho_primary"]["recomputed"], rho)
        check(f"ledger:0e.k{k}.p_primary", ledger[f"0e.k{k}.p_primary"]["recomputed"], p_lower(rho, null1))
        xa = [sum(prim[3 * i:3 * i + 3]) / 3 for i in range(5)]
        ga = [galg[a] for a in nonb]
        r5 = spearman(ga, xa)
        perms = [spearman(ga, [xa[p] for p in perm]) for perm in itertools.permutations(range(5))]
        if k == 15:
            check("ledger:0e.k15.rho_5c", ledger["0e.k15.rho_5c"]["recomputed"], r5)
            check("ledger:0e.k15.p_5c", ledger["0e.k15.p_5c"]["recomputed"],
                  sum(1 for v in perms if v <= r5 + 1e-12) / 120)
            check("ledger:0e.k15.floor_5c", ledger["0e.k15.floor_5c"]["recomputed"],
                  sum(1 for v in perms if v <= min(perms) + 1e-12) / 120)
        # within-algorithm stratified test on content-token overlap
        stop = {"a", "an", "the", "with", "from", "both", "at", "once", "on", "run", "and", "of", "to", "s", "search",
                "algorithm"}
        tok = lambda s: {t for t in re.split(r"[^a-z0-9*]+", s.lower()) if t and t not in stop}  # noqa: E731
        bal = [tok(s) for b in banned for s in lit["ALIAS_TABLE"][b]]
        sec = [max(len(tok(s) & bt) / len(tok(s) | bt) for bt in bal) for a in nonb for s in lit["ALIAS_TABLE"][a]]
        G = [gal[3 * i:3 * i + 3] for i in range(5)]
        X = [sec[3 * i:3 * i + 3] for i in range(5)]

        def strat(Xs):
            vals = [spearman(G[i], Xs[i]) for i in range(5) if max(Xs[i]) - min(Xs[i]) > 1e-12]
            return sum(vals) / len(vals)

        def beta(Xs):
            num = den = 0.0
            for i in range(5):
                mg, mx = sum(G[i]) / 3, sum(Xs[i]) / 3
                num += sum((g - mg) * (x - mx) for g, x in zip(G[i], Xs[i]))
                den += sum((x - mx) ** 2 for x in Xs[i])
            return num / den

        so, bo = strat(X), beta(X)
        sts, bes = [], []
        for combo in itertools.product(list(itertools.permutations(range(3))), repeat=5):
            Xp = [[X[i][c] for c in combo[i]] for i in range(5)]
            sts.append(strat(Xp))
            bes.append(beta(Xp))
        ps = sum(1 for v in sts if v <= so + 1e-12) / len(sts)
        pb = sum(1 for v in bes if abs(v) >= abs(bo) - 1e-12) / len(bes)
        check(f"ledger:0e.k{k}.strat_p", ledger[f"0e.k{k}.strat_p"]["recomputed"], ps)
        if k == 15:
            check("ledger:0e.k15.strat", ledger["0e.k15.strat"]["recomputed"], so)
            check("ledger:0e.k15.beta", ledger["0e.k15.beta"]["recomputed"], bo)
            check("ledger:0e.k15.beta_p", ledger["0e.k15.beta_p"]["recomputed"], pb)
            check("ledger:0e.k15.rho_secondary", ledger["0e.k15.rho_secondary"]["recomputed"], spearman(gal, sec))
            g15, prim15 = gal, prim
    tert_p = ROOT / "results/s0e_tertiary.json"
    if tert_p.exists() and "0e.k15.bpe_strat" in ledger:
        T = json.loads(tert_p.read_text())
        x3 = [r["tertiary"] for r in T["rows"]]
        geo, shape = npy_member(gz, "geo_k15")
        N = shape[0]
        bidx = {b: [i for i in range(N) if labels[i] == b] for b in banned}
        gal = []
        for a in nonb:
            for j in range(3):
                I = [i for i in range(N) if labels[i] == a and grp_f[i] % 3 == j]
                gal.append(min(sum(geo[i * N + jj] for i in I for jj in bidx[b]) / (len(I) * len(bidx[b])) for b in banned))
        G3 = [gal[3 * i:3 * i + 3] for i in range(5)]
        X3 = [x3[3 * i:3 * i + 3] for i in range(5)]

        def strat3(Xs):
            vals = [spearman(G3[i], Xs[i]) for i in range(5) if max(Xs[i]) - min(Xs[i]) > 1e-12]
            return sum(vals) / len(vals)

        so3 = strat3(X3)
        null3 = [strat3([[X3[i][c] for c in combo[i]] for i in range(5)])
                 for combo in itertools.product(list(itertools.permutations(range(3))), repeat=5)]
        check("ledger:0e.k15.bpe_strat", ledger["0e.k15.bpe_strat"]["recomputed"], so3)
        check("ledger:0e.k15.bpe_strat_p", ledger["0e.k15.bpe_strat_p"]["recomputed"],
              sum(1 for v in null3 if v <= so3 + 1e-12) / len(null3))
        check("ledger:0e.k15.bpe_rho", ledger["0e.k15.bpe_rho"]["recomputed"], spearman(gal, x3))
    # consistency-only checks for randomised 0e quantities (bootstrap, placebos, E4 need numpy-scale work)
    stored = {"0e.k15.ci_lo": s0e["per_k"]["k15"]["ci95_primary"][0], "0e.k15.ci_hi": s0e["per_k"]["k15"]["ci95_primary"][1],
              "0e.P1": s0e["placebos"]["P1_shuffled_alias_labels"]["share_p_lt_0.05"],
              "0e.P2": s0e["placebos"]["P2_random_overlap_vector"]["share_p_lt_0.05"],
              "0e.C1.rho": s0e["placebos"]["C1_planted"]["seed0_rho"], "0e.C1.p": s0e["placebos"]["C1_planted"]["seed0_p_lower"],
              "0e.E4": s0e["gates"].get("E4_spearman_vs_stored_k15")}
    for nm, val in stored.items():
        if nm not in ledger:
            continue
        check(f"ledger:{nm}", ledger[nm]["recomputed"], val, kind="consistency")

    # (4) the apply test, independently
    orig = (ROOT / "inputs/report/current_report.md").read_text()
    edits = corr["edits"]
    spans = []
    for e in edits:
        spans.append((orig.index(e["old_text"]), e))
        check(f"apply.unique.{e['id']}", 1, orig.count(e["old_text"]))
    spans.sort(key=lambda t: t[0])
    out, last = [], 0
    for s, e in spans:
        assert s >= last, f"overlap at {e['id']}"
        out.append(orig[last:s])
        out.append(e["new_text"])
        last = s + len(e["old_text"])
    out.append(orig[last:])
    rebuilt = "".join(out)
    check("apply.rebuilt_equals_report_corrected", hashlib.sha256((ROOT / "report_corrected.md").read_bytes()).hexdigest(),
          hashlib.sha256(rebuilt.encode()).hexdigest())

    # (5) placebos that must FAIL
    plac = {}
    rnd = random.Random(20261023)
    geo, shape = npy_member(gz, "geo_k15")
    N = shape[0]
    pts = list(zip(labels, [g % 3 for g in grp_f]))
    rej = 0
    rhos = []
    for _ in range(200):
        sh = pts[:]
        rnd.shuffle(sh)
        bidx = {b: [i for i in range(N) if sh[i][0] == b] for b in banned}
        gal = []
        for a in nonb:
            for j in range(3):
                I = [i for i in range(N) if sh[i] == (a, j)]
                gal.append(min(sum(geo[i * N + jj] for i in I for jj in bidx[b]) / (len(I) * len(bidx[b]))
                               for b in banned))
        r = spearman(gal, [float(x) for x in prim15])
        rhos.append(r)
        rej += p_lower(r, null1) < 0.05
    plac["P1_shuffled_alias_labels"] = {"reject_share": rej / 200, "mean_rho": sum(rhos) / 200,
                                        "failed_as_required": rej / 200 <= 0.10}
    wrong = [r for r in V["C1forced"]["per_node"]["ablate"] if r["node"] == "head|L33|H14|DEC"][0]["mean_d_heldout"]
    plac["P2_wrong_node"] = {"ledger": ledger["0a.L0.abl.heldout"]["recomputed"], "wrong_node_value": wrong,
                             "failed_as_required": abs(wrong - ledger["0a.L0.abl.heldout"]["recomputed"]) > 1e-6}
    dup = orig + "\n" + edits[0]["old_text"]
    plac["P3_duplicated_old_text"] = {"count": dup.count(edits[0]["old_text"]),
                                      "failed_as_required": dup.count(edits[0]["old_text"]) != 1}
    corrected = (ROOT / "report_corrected.md").read_text()
    start = corrected.index("# Iteration 8: temperature at the fork")

    def l1_hits(text):
        n = 0
        body = text[start:]
        for m in re.finditer(r"same shape", body):
            w = body[max(0, m.start() - 300): m.end() + 300]
            n += ("circuit" in w or "Gemma" in w)
        return n

    clean = l1_hits(corrected)
    inj = corrected.replace("## 8.11 What the next iteration", "The circuit has the same shape as Gemma's circuit.\n\n## 8.11 "
                            "What the next iteration", 1)
    plac["P4_injected_same_shape"] = {"hits_clean": clean, "hits_injected": l1_hits(inj),
                                      "failed_as_required": clean == 0 and l1_hits(inj) >= 1}
    n_fail = sum(1 for p in plac.values() if p["failed_as_required"])
    res = {"n_checks": len(CHECKS), "n_mismatches": sum(1 for c in CHECKS if not c["ok"]),
           "n_independent": sum(1 for c in CHECKS if c["kind"] == "independent"),
           "n_consistency": sum(1 for c in CHECKS if c["kind"] == "consistency"),
           "n_ledger_generic": n_generic, "n_match_values_checked": sum(
               1 for c in CHECKS if c["name"].startswith("ledger:") and ledger.get(c["name"][7:], {}).get("status") == "MATCH"),
           "placebos_failed": n_fail, "placebos_total": len(plac), "placebos": plac,
           "mismatches": [c for c in CHECKS if not c["ok"]], "checks": CHECKS}
    (ROOT / "audit/rederive_result.json").write_text(json.dumps(res, indent=1, default=str))
    print(json.dumps({k: res[k] for k in ("n_checks", "n_mismatches", "n_independent", "n_consistency",
                                          "n_match_values_checked", "placebos_failed", "placebos_total")}))
    print(json.dumps(res["mismatches"][:10], default=str))
    return 0 if res["n_mismatches"] == 0 and n_fail == len(plac) else 1


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile eval_demo_run/art_iYh9zVP7myjJ/audit/headline_rederive.py
#!/usr/bin/env python3
"""Short headline re-derivation from RAW per-row files (not the aggregated result fields), stdlib only.

Separate code path from eval.py/src and from audit/rederive.py. Headlines checked:
  0a  L0 condition-MLP pair-prompt means (8.144 / 7.408) and held-out means (0.0) from raw validation margins
  0b  Qwen3-8B fork toward BFS: 29 written, 5 correct, pass 0.20 vs base 0.83, DAS:random discordance 5:0 (p 0.0625)
  0c  highest-entropy arm hits the fork on 44/44, 32/39, 21/39, 2/50, 0/37 slots
  0e  15-alias Spearman -0.61 and exact p 0.0093 at k 15
Placebos that must FAIL: shuffled point labels (0e), a non-condition node (0a), random dose positions (0c).
    python -I audit/headline_rederive.py   ->   audit/headline_rederive.json
"""
import array
import ast
import itertools
import json
import math
import os
import random
import re
import zipfile
from pathlib import Path

ROOT = Path(__file__).resolve().parent.parent
DEPS = Path(os.environ.get("S0_DEPS_DIR", ROOT.parents[2] / "iter_8" / "gen_art"))
NAMES = {"circ": ("art_18XvfcJJZi1O", "gen_art_experiment_41"), "das": ("art_e-_q5mSJGrSb", "gen_art_experiment_40"),
         "fork": ("art_vnyi0Fm5SyNH", "gen_art_experiment_39"), "tp": ("art_R-HSmXq2t97f", "gen_art_experiment_44")}


def art(key):
    for base in (DEPS, ROOT.parent):
        for n in NAMES[key]:
            if (base / n).is_dir():
                return base / n
    return DEPS / NAMES[key][1]


def jl(p):
    return [json.loads(l) for l in p.read_text().splitlines() if l.strip()]


out, ok = {}, []

# ---- 0a: raw validation margins, d = node margin - mean of its 50 matched random components
rows = jl(art("circ") / "data/validation_rows_C1forced.jsonl")
for mode, exp_pair in (("ablate", 8.144148281097412), ("steer", 7.407574213027954)):
    def d_of(r, node):
        rnd = [v for k, v in r["res"].items() if k.startswith(f"{mode}|{node}|r")]
        return r["res"][f"{mode}|{node}|node"] - sum(rnd) / len(rnd)
    node = "mlp|L0|H-1|COND"
    pair = [d_of(r, node) for r in rows if not r["heldout"]]
    held = [d_of(r, node) for r in rows if r["heldout"]]
    out[f"0a_{mode}"] = {"pair_mean": sum(pair) / len(pair), "heldout_mean": sum(held) / len(held), "n": [len(pair), len(held)]}
    ok.append(abs(out[f"0a_{mode}"]["pair_mean"] - exp_pair) < 1e-6 and out[f"0a_{mode}"]["heldout_mean"] == 0.0)
    plac = [d_of(r, "head|L33|H14|DEC") for r in rows if r["heldout"]]
    out[f"0a_placebo_{mode}"] = {"node": "head|L33|H14|DEC", "heldout_mean": sum(plac) / len(plac),
                                 "failed_as_required": sum(plac) / len(plac) != 0.0}

# ---- 0b: raw generation labels at the Qwen3-8B fork (unit T10, site dec, target BFS)
lab = [r for r in jl(art("das") / "data/labels_T10.jsonl") if r.get("site") == "dec" and r.get("target_class") == "bfs"]
cell = lambda arm: {r["slot"]: r for r in lab if r["arm"] == arm}  # noqa: E731
das, base, rnd = cell("DAS"), cell("BASE"), cell("RAND")
corr = lambda r: r["label"] == "bfs" and r["passed"]  # noqa: E731
b = sum(1 for s in das if corr(das[s]) and not corr(rnd[s]))
c = sum(1 for s in das if corr(rnd[s]) and not corr(das[s]))
p_mc = min(1.0, 2 * sum(math.comb(b + c, i) for i in range(min(b, c) + 1)) / 2 ** (b + c)) if b + c else 1.0
out["0b"] = {"n": len(das), "written_bfs": sum(1 for r in das.values() if r["label"] == "bfs"),
             "correct_bfs": sum(1 for r in das.values() if corr(r)), "pass_das": sum(r["passed"] for r in das.values()) / len(das),
             "pass_base": sum(r["passed"] for r in base.values()) / len(base), "das_vs_random": [b, c], "mcnemar_p": p_mc}
ok.append([out["0b"][k] for k in ("written_bfs", "correct_bfs")] == [29, 5] and abs(out["0b"]["mcnemar_p"] - 0.0625) < 1e-12
          and round(out["0b"]["pass_das"], 2) == 0.20 and round(out["0b"]["pass_base"], 2) == 0.83)

# ---- 0c: raw rows. For every twin slot that reaches the fork, is the position the A7 arm doses (the highest-entropy
# position, fixed from the twin's own entropies) the twin's fork position?
smp = jl(art("fork") / "data/samples.jsonl")
twin = {(r["unit"], r["slot"]): r for r in smp if r.get("arm") == "A0" and r.get("stage") == "S0" and r.get("reached_decision")}
a7pos = {(r["unit"], r["slot"]): set(r["schedule"]["fixed_pos"]["pos"]) for r in smp if r.get("arm") == "A7"}
rng = random.Random(7)
hits, plac = {}, {}
for u in ("G", "B3", "B6", "T10", "A3"):
    keys = [k for k in twin if k[0] == u]
    fork = lambda k: {twin[k]["decision_pos"]} | ({10, 11} if u == "T10" else set())  # noqa: E731
    hits[u] = [sum(1 for k in keys if a7pos.get(k, set()) & fork(k)), len(keys)]
    plac[u] = sum(1 for k in keys if {rng.randrange(64)} & fork(k))
out["0c_hits"] = hits
out["0c_placebo_random_position"] = {"hits": plac, "total": sum(plac.values()),
                                     "real_total": sum(h[0] for h in hits.values()),
                                     "failed_as_required": sum(plac.values()) <= 0.1 * sum(h[0] for h in hits.values())}
ok.append(hits == {"G": [44, 44], "B3": [32, 39], "B6": [21, 39], "T10": [2, 50], "A3": [0, 37]})

# ---- 0e: raw npz -> alias-level g -> Spearman and its exact permutation p
def npy(zp, m):
    raw = zipfile.ZipFile(zp).read(m + ".npy")
    hl = int.from_bytes(raw[8:10], "little")
    hd = ast.literal_eval(raw[10:10 + hl].decode("latin1"))
    data = raw[10 + hl:]
    if hd["descr"].startswith("<U"):
        w = int(hd["descr"][2:]) * 4
        return [data[i:i + w].decode("utf-32-le").rstrip("\x00") for i in range(0, len(data), w)]
    a = array.array("d" if hd["descr"] == "<f8" else "q")
    a.frombytes(data)
    return list(a)


def rk(x):
    s = sorted(range(len(x)), key=x.__getitem__)
    r, i = [0.0] * len(x), 0
    while i < len(s):
        j = i
        while j + 1 < len(s) and x[s[j + 1]] == x[s[i]]:
            j += 1
        for t in s[i:j + 1]:
            r[t] = (i + j) / 2 + 1
        i = j + 1
    return r


def rho(x, y):
    a, b_ = rk(x), rk(y)
    ma, mb = sum(a) / len(a), sum(b_) / len(b_)
    return sum((u - ma) * (v - mb) for u, v in zip(a, b_)) / math.sqrt(
        sum((u - ma) ** 2 for u in a) * sum((v - mb) ** 2 for v in b_))


tp = art("tp") / "cache/states"
geo, labels = npy(tp / "geo_C6_nf4.npz", "geo_k15"), npy(tp / "geo_C6_nf4.npz", "labels")
if (tp / "smean_C6_nf4.npz").exists():
    kind, grp = npy(tp / "smean_C6_nf4.npz", "kind"), npy(tp / "smean_C6_nf4.npz", "group")
else:   # published copy without the 181 MB states file: the vendored index of the same rows
    _ix = json.loads((ROOT / "inputs/forced_index_C6_nf4.json").read_text())
    kind, grp = _ix["kind"], _ix["group"]
alias = [int(g) % 3 for g, k in zip(grp, kind) if k == "forced"]
banned, nonb = ["bfs", "dfs", "iddfs", "astar", "gbfs"], ["bibfs", "dijkstra_unit", "idastar", "jps", "biastar"]
overlap = [1, 1, 1, 0, 0, 0, 1, 2, 3, 0, 0, 0, 1, 1, 1]   # committed primary vectors (predictions/commit_s0e.json)


def g_alias(lab, ali):
    bi = {b_: [i for i in range(300) if lab[i] == b_] for b_ in banned}
    g = []
    for a_ in nonb:
        for j in range(3):
            ii = [i for i in range(300) if lab[i] == a_ and ali[i] == j]
            g.append(min(sum(geo[i * 300 + t] for i in ii for t in bi[b_]) / (len(ii) * len(bi[b_])) for b_ in banned))
    return g


null = []
for p2 in range(15):
    for p3 in range(15):
        if p3 == p2:
            continue
        rest = [i for i in range(15) if i not in (p2, p3)]
        for zeros in itertools.combinations(rest, 6):
            v = [1] * 15
            v[p2], v[p3] = 2, 3
            for z in zeros:
                v[z] = 0
            null.append(v)
gr = list(range(15))
null_r = sorted(rho(gr, v) for v in null)
pval = lambda r: sum(1 for v in null_r if v <= r + 1e-12) / len(null_r)  # noqa: E731
r15 = rho(g_alias(labels, alias), overlap)
out["0e"] = {"rho_k15": r15, "p_exact": pval(r15), "n_assignments": len(null)}
ok.append(abs(r15 - (-0.6068480408008414)) < 1e-9 and abs(out["0e"]["p_exact"] - 0.009324009324009324) < 1e-12)
pts = list(zip(labels, alias))
rej = 0
for _ in range(100):
    sh = pts[:]
    rng.shuffle(sh)
    rej += pval(rho(g_alias([s[0] for s in sh], [s[1] for s in sh]), overlap)) < 0.05
out["0e_placebo_shuffled_labels"] = {"reject_share": rej / 100, "failed_as_required": rej / 100 <= 0.10}
out["all_headlines_match"] = all(ok)
out["placebos_failed_as_required"] = sum(1 for k, v in out.items() if "placebo" in k and isinstance(v, dict)
                                         and v.get("failed_as_required"))
(ROOT / "audit/headline_rederive.json").write_text(json.dumps(out, indent=1))
print(json.dumps({k: out[k] for k in ("0a_ablate", "0a_steer", "0b", "0c_hits", "0e", "0e_placebo_shuffled_labels",
                                      "0c_placebo_random_position", "all_headlines_match", "placebos_failed_as_required")}, indent=1))


## `eval.py`: header

This is the original header, which imports the modules written above. Three changes are marked `# demo:`:
- `ROOT` is `eval_demo_run/art_iYh9zVP7myjJ`, where the inputs were written; in the original it is the folder that
  holds `eval.py`.
- Re-running the cell re-imports the modules.
- The config cell's resampling sizes overwrite the constants in `alias_geometry`.

In [ ]:
ROOT = Path("eval_demo_run/art_iYh9zVP7myjJ").resolve()  # demo: where the inputs were written (original: Path(__file__).resolve().parent)
sys.path.insert(0, str(ROOT / "src"))
for _m in ("alias_geometry", "apply_lint", "edits_spec", "recompute", "s0_extra", "stage_0e", "tertiary_bpe"):
    sys.modules.pop(_m, None)  # demo: a re-run of this cell re-imports the modules the %%writefile cells wrote
import alias_geometry as AG  # noqa: E402
import apply_lint as AL  # noqa: E402
import edits_spec  # noqa: E402
import recompute as RC  # noqa: E402
import s0_extra as SX  # noqa: E402
import stage_0e as S0E  # noqa: E402
import tertiary_bpe  # noqa: E402

AG.N_BOOT, AG.N_PLACEBO, AG.N_MC_SECONDARY = N_BOOT, N_PLACEBO, N_MC_SECONDARY  # demo: sizes from the config cell

REPORT_SRC = Path(os.environ.get("S0_REPORT_SRC", ROOT.parents[1] / "gen_strat" / "current_report.md"))
REPORT = ROOT / "inputs/report/current_report.md"
RESULTS, TABLES = ROOT / "results", ROOT / "tables"

logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")
(ROOT / "logs").mkdir(exist_ok=True)
logger.add(ROOT / "logs/eval.log", rotation="30 MB", level="DEBUG")


## Stage `setup`: input hashes

`stage_setup` refuses any report except the one this repair was built for (sha256 `8b3804f3…`). It hashes every input
into `inputs_manifest.json`, and `main()` checks at the end that no input changed during the run.

One `# demo:` change: only the inputs that exist are hashed. The iteration-8 source files and images that the code
never reads are not shipped. The shipped JSON files are pruned, so their hashes differ from the full run's manifest; the
report's hash does not.

In [ ]:
def sha(p: Path) -> str:
    h = hashlib.sha256()
    with p.open("rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def input_files() -> list[tuple[str, Path]]:
    A = RC.ART
    rel = {
        RC.CIRC: ["results/summary.json", "data/validation_rows_C1forced.jsonl", "data/validation_rows_C2a.jsonl",
                  "data/validation_rows_C1nat.jsonl", "src/k3circuit.py", "src/k3motif.py", "results/positioning.json"],
        RC.DAS: ["results/analysis.json", "results/geometry.json", "results/positioning.json", "src/figures8.py"],
        RC.FORK: ["results/k1_verdict.json", "results/k1_cells.json", "results/amendment.json",
                  "predictions/commit1_dose_table.json", "data/samples.jsonl", "src/analysis_k1.py",
                  "results/geometry_f8.json", "results/positioning.json", "src/figures8.py"],
        RC.LAW: ["results/analysis_L.json", "results/positioning.json"],
        RC.GEM: ["results/positioning.json", "src/outputs8.py"],
        RC.TP: ["results/geometry_M1.json", "results/geometry_M2.json", "src/common.py", "src/prompts_tp.py",
                "src/geometry.py", "src/stages.py", "src/figures.py", "cache/states/geo_C6_nf4.npz",
                "cache/states/smean_C6_nf4.npz", "results/positioning.json"],
    }
    figs = {RC.FORK: ["F8_activation_geometry"], RC.DAS: ["F3_geometry_T10", "F3_geometry_GRID", "F3b_handle_axis_T10",
                                                          "F3b_handle_axis_GRID"],
            RC.GEM: ["F4_geometry"], RC.TP: ["F3_mds_C6_nf4", "F4_isomap_C6_nf4", "F5_M2_C6"]}
    out = [("report:inputs/report/current_report.md", REPORT)]
    for art, files in rel.items():
        out += [(f"{art}:{f}", A[art] / f) for f in files]
    for art, fs in figs.items():
        out += [(f"{art}:figures/{f}.{e}", A[art] / f"figures/{f}.{e}") for f in fs for e in ("png", "pdf")]
    return out


def stage_setup() -> dict[str, Any]:
    if REPORT_SRC.exists():
        if sha(REPORT) != sha(REPORT_SRC):
            raise RuntimeError("vendored report differs from the strategy's input")
    else:
        logger.info("strategy input not present (published copy): using the vendored report and its recorded sha256")
    if sha(REPORT) != "8b3804f3ecbfedd4c1d54586edd56fe02aaa5c81e4570f4a3aa171ad0cbd7438":
        raise RuntimeError("vendored report is not the report this repair was built for")
    man = {k: {"sha256": sha(p), "bytes": p.stat().st_size} for k, p in input_files()
           if p.exists()}  # demo: hash only the inputs that are shipped
    man["_report_source"] = str(REPORT_SRC.name)
    (ROOT / "inputs_manifest.json").write_text(json.dumps(man, indent=1))
    logger.info(f"inputs manifest: {len(man) - 1} files hashed; report sha256 {man['report:inputs/report/current_report.md']['sha256'][:16]}")
    return man


## Stage `0e`

Before running item 0e, `eval.py` checks the pre-commitment. It hashes the committed code files and, if they changed
after the commitment, appends an AMENDMENT line to `commitments.log`. The original run logged two amendments, each with
a reason. `src/stage_0e.py` here has the same hash as in the last logged amendment, so no new line is added. The
warning still prints, as it does in the original.

The log also shows `ERROR | E4 failed to run: states file not available`. That is the original code's message for a
copy without the 181 MB states file, and the run continues. Gates E1–E3 run on the vendored prompt index, which is
also what the published repository does.

In [ ]:
def stage_0e() -> dict[str, Any]:
    commit = ROOT / "predictions/commit_s0e.json"
    if not commit.exists():
        raise RuntimeError("predictions/commit_s0e.json missing: run src/commit_s0e.py before any alias-level number")
    committed = json.loads(commit.read_text())["code_sha256"]
    now = {p: sha(ROOT / p) for p in committed}
    if now != committed:
        line = (f"{dt.datetime.now(dt.timezone.utc).isoformat()} AMENDMENT S0-0e code changed after commit: "
                + " ".join(f"{k}={v}" for k, v in now.items()) + "\n")
        log = ROOT / "commitments.log"
        if line.split(" ", 1)[1] not in log.read_text():
            with log.open("a") as f:
                f.write(line)
        logger.warning("0e code differs from the committed hash; amendment logged")
    t0 = time.time()
    res = S0E.run(RC.ART[RC.TP], ROOT)
    res["seconds"] = round(time.time() - t0, 1)
    res["code_matches_commit"] = now == committed
    (RESULTS / "s0e.json").write_text(json.dumps(res, indent=1, default=float))
    logger.info(f"0e verdict {res.get('verdict', {}).get('code')} in {res['seconds']} s")
    return res


## Stage `ledger`

This stage recomputes items 0a, 0b, 0c and 0f into one shared `Ledger`. Stage `edits` adds the 0e and 0d values, and
`tables/ledger.csv` lists them all.

In [ ]:
def stage_ledger(led: RC.Ledger) -> dict[str, Any]:
    out = {"0a": RC.recompute_0a(led, ROOT), "0b": RC.recompute_0b(led), "0c": RC.recompute_0c(led, ROOT),
           "0f": RC.recompute_0f(led, ROOT)}
    logger.info(f"ledger: {Counter(r['status'] for r in led.rows)}")
    return out


## Stage `edits`

This stage builds:
- the 0d positioning section;
- the 0g figure markers;
- the 0e paragraph;
- the held-out sentence for Table 396.

Then it does the following, in order:
1. builds the 49 edits, checking that every value they cite is in the ledger;
2. applies them, running the apply test;
3. lints the whole report before and after;
4. writes `report_corrected.md` and `report.diff`;
5. classifies every "29 of 30" before and after;
6. writes the ledger and lint tables and `figure_markers.json`.

In [ ]:
def stage_edits(led: RC.Ledger, rec: dict[str, Any], z: dict[str, Any], tz: dict[str, Any] | None) -> dict[str, Any]:
    report = REPORT.read_text()
    pos = SX.positioning(ROOT)
    markers = SX.figure_markers(ROOT, z)
    alias_md = next(m for m in markers if m["id"] == "fig_alias_overlap")["block_md"]
    at = SX.alias_text(z, led, alias_md, tz)
    led.add("0e.M2.g16", "0e", RC.get(RC.TP, "results/geometry_M2.json", "label_decodability_balanced_acc.g16"),
            RC.src(RC.TP, "results/geometry_M2.json", "label_decodability_balanced_acc.g16"), ".2f", recorded="0.53")
    weak = []
    c2 = [r for r in rec["0a"]["table_rows"] if r["contrast"] == "C2a"]
    both = {n for n in {r["node"] for r in c2} if all(r["p_holm_20"] < 0.05 for r in c2 if r["node"] == n)}
    for r in c2:
        if r["node"] in both and (r["p_sign_flip_heldout"] or 1) > 0.05:
            weak.append(f"the {'ablation' if r['mode'] == 'ablate' else 'steering'} of the node '{RC.node_label(r['node'])}' "
                        f"({r['mean_d_heldout']:.2f} nats, p {r['p_sign_flip_heldout']:.2f})")
    weak396 = ("On the 10 held-out prompts every node that passes both tests keeps a held-out effect of the same sign "
               "with sign-flip p at most 0.003, except " + "; ".join(weak) + ".") if weak else \
        "On the 10 held-out prompts every node that passes both tests keeps a held-out effect of the same sign."
    ctx = {"weak396": weak396, "rec395": rec["0a"]["rec395"], "rec396": rec["0a"]["rec396"], "code_quotes": rec["0a"]["code_quotes"],
           "bands_line": rec["0a"]["bands"]["line"], "positioning_section": pos["section"], "new_refs": pos["new_refs"],
           "markers": SX.marker_edits(markers), "alias_paragraph": at["paragraph"], "alias_short": at["short"],
           "alias_value_names": at["value_names"], "z_printed": at["printed"]}
    edits = edits_spec.build(report, led.p, z, ctx)
    for e in edits:
        for v in e["values"]:
            if v not in led.by:
                raise KeyError(f"{e['id']} cites unknown value {v}")
    corrected, app = AL.apply_edits(report, edits)
    lint_before = AL.lint_report(report)
    lint_after = AL.lint_report(corrected) if app["apply_test_pass"] else {"n_unallowed": None}
    logger.info(f"apply_test_pass={app['apply_test_pass']} n_edits={app['n_edits']} problems={app['problems'][:5]}")
    logger.info(f"lint before {lint_before['by_pattern']} unallowed {lint_before['n_unallowed']}; after unallowed "
                f"{lint_after.get('n_unallowed')} {lint_after.get('unallowed_by_pattern')}")
    if app["apply_test_pass"]:
        (ROOT / "report_corrected.md").write_text(corrected)
        (ROOT / "report.diff").write_text(AL.unified_diff(report, corrected))
    occ = RC.classify_29of30(report)
    occ_after = RC.classify_29of30(corrected)
    with (TABLES / "occurrences_29of30.csv").open("w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["when", "line", "class", "context"])
        w.writeheader()
        w.writerows([{"when": "before", **o} for o in occ] + [{"when": "after", **o} for o in occ_after])
    with (TABLES / "lint_hits.csv").open("w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["when", "pattern", "line", "section", "iteration8_text", "allow_reason",
                                          "snippet"])
        w.writeheader()
        for when, L in (("before", lint_before), ("after", lint_after)):
            for h in L.get("unallowed", []) + L.get("allowlisted", []):
                w.writerow({"when": when, **{k: h.get(k) for k in w.fieldnames if k != "when"}})
    (ROOT / "lint_allowlist.json").write_text(json.dumps(
        {"rule": "hits in pre-iteration-8 text or in the iteration-8 prediction statements of 8.1 are allowlisted with a "
                 "reason; every other hit must be gone", "before": lint_before["allowlisted"],
         "after": lint_after.get("allowlisted", [])}, indent=1))
    with (TABLES / "ledger.csv").open("w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(led.rows[0].keys()))
        w.writeheader()
        w.writerows(led.rows)
    fm = [{k: m[k] for k in ("id", "art", "files", "section", "fields", "caption", "sha256", "n_fields", "code")}
          for m in markers]
    (ROOT / "figure_markers.json").write_text(json.dumps(fm, indent=1))
    return {"edits": edits, "apply": app, "lint_before": lint_before, "lint_after": lint_after, "positioning": pos,
            "markers": markers, "alias_text": at, "occ": occ, "occ_after": occ_after, "corrected": corrected}


## `corrections.json`

The item checklist names the evidence for each review item. `write_corrections` writes `corrections.json`, which holds:
- the edits, each with its line hint and the full ledger rows of the values it cites;
- every value;
- the counts by status;
- the apply and lint summary.

In [ ]:
CHECKLIST = {
    "0a": ("MUST-FIX", "Circuit shape and the unapplied held-out arm", "tables/circuit_heldout.csv; results/s0_core.json#0a"),
    "0b": ("MUST-FIX", "Qwen3-8B fork NO HANDLE under the committed rule", "art_e-_q5mSJGrSb:results/analysis.json#verdicts"),
    "0c": ("MUST-FIX", "Entropy superlative and temperature-law scope", "tables/entropy_scope.csv; tables/law_scope.csv"),
    "0d": ("MUST-FIX", "Iteration-8 positioning table", "tables/positioning_rows.csv; audit/web/"),
    "0e": ("MUST-FIX", "Alias-named manifold", "results/s0e.json; tables/alias_stats.csv; figures/alias_overlap.png"),
    "0f": ("minor", "Gemma comparators", "tables/gemma_arms.csv"),
    "0g": ("minor", "Missing figure markers", "figure_markers.json"),
}


def write_corrections(led: RC.Ledger, ed: dict[str, Any], man: dict[str, Any]) -> dict[str, Any]:
    edits = ed["edits"]
    vb = Counter(r["status"] for r in led.rows)
    for s in ("MATCH", "CORRECTED", "NEW", "CARRIED"):
        vb.setdefault(s, 0)
    items = {}
    for it in CHECKLIST:
        ids = [e["id"] for e in edits if e["item"] == it]
        items[it] = {"severity": CHECKLIST[it][0], "item": CHECKLIST[it][1], "edit_ids": ids,
                     "status": "ANSWERED" if ids and ed["apply"]["apply_test_pass"] else "OPEN",
                     "evidence": CHECKLIST[it][2]}
    corr = {"read_date": dt.date.today().isoformat(), "report_sha256": man["report:inputs/report/current_report.md"]["sha256"],
            "schema": "S0 corrections v1 (rebuilt from the plan: edits[] id/item/section/line_hint/old_text/new_text/"
                      "rationale/values; values[] name/recorded/recomputed/status/tolerance/source)",
            "n_edits": len(edits), "n_values": len(led.rows), "values_by_status": dict(vb),
            "n_edits_by_item": dict(Counter(e["item"] for e in edits)),
            "apply": {"apply_test_pass": ed["apply"]["apply_test_pass"], "n_edits": ed["apply"]["n_edits"],
                      "problems": ed["apply"]["problems"], "lint_before": ed["lint_before"]["n_unallowed"],
                      "lint_before_total_hits": ed["lint_before"]["n_hits"],
                      "lint_after": ed["lint_after"].get("n_unallowed"),
                      "lint_before_by_pattern": ed["lint_before"]["by_pattern"],
                      "lint_after_by_pattern": ed["lint_after"].get("by_pattern")},
            "item_checklist": items,
            "edits": [{**{k: e[k] for k in ("id", "item", "section")},
                       "line_hint": ed["apply"].get("line_of_edit", {}).get(e["id"]),
                       **{k: e[k] for k in ("old_text", "new_text", "rationale")},
                       "values": [led.by[v] for v in e["values"]]} for e in edits],
            "values": led.rows}
    (ROOT / "corrections.json").write_text(json.dumps(corr, indent=1, default=str))
    (ROOT / "apply_test.json").write_text(json.dumps(corr["apply"], indent=1))
    return corr


## `eval_out.json` (exp_eval_sol_out)

`ex()` turns one row into an example. Keys starting `eval_` are numeric scores and are never NaN; keys starting
`predict_` are strings; any other key becomes `metadata_*`.

`write_eval_out` builds:
- `metrics_agg`, the headline metrics;
- 10 datasets: corrections, report_edits, circuit_heldout, occurrences_29of30, entropy_scope, positioning_rows,
  alias_overlap, gemma_arms, figure_markers and lint_hits;
- per-item metadata.

It writes the full, mini and preview variants.

In [ ]:
def ex(inp: str, out: str, **kw: Any) -> dict[str, Any]:
    d = {"input": inp, "output": out}
    if not any(k.startswith("predict_") for k in kw):
        d["predict_s0"] = out
    for k, v in kw.items():
        if k.startswith("eval_"):
            if v is None or (isinstance(v, float) and v != v):
                continue                      # not available (e.g. an arm that was not run): omit, never NaN
            d[k] = float(v)
        elif k.startswith("predict_"):
            d[k] = str(v)
        else:
            d[f"metadata_{k}"] = v
    return d


def write_eval_out(led: RC.Ledger, rec: dict[str, Any], z: dict[str, Any], ed: dict[str, Any], corr: dict[str, Any],
                   audit: dict[str, Any] | None) -> dict[str, Any]:
    k15, k10 = z["per_k"]["k15"], z["per_k"]["k10"]
    st = k15["stratified_secondary"]
    c5 = k15["five_centroid_primary"]
    a0 = rec["0a"]
    vb = corr["values_by_status"]
    metrics = {
        "items_answered": sum(1 for v in corr["item_checklist"].values() if v["status"] == "ANSWERED"),
        "items_total": len(CHECKLIST), "n_edits": corr["n_edits"], "apply_test_pass": int(corr["apply"]["apply_test_pass"]),
        "lint_before_unallowed": ed["lint_before"]["n_unallowed"], "lint_before_total": ed["lint_before"]["n_hits"],
        "lint_after_unallowed": ed["lint_after"].get("n_unallowed") or 0,
        "values_total": corr["n_values"], "values_match": vb["MATCH"], "values_corrected": vb["CORRECTED"],
        "values_new": vb["NEW"], "values_carried": vb["CARRIED"],
        "circuit_identity_check_pass": int(a0["identity_check_pass"]),
        "circuit_n_exact_zero_heldout_nodes": a0["n_exact_zero_heldout_nodes"],
        "circuit_cond_span_matches_heldout": a0["cond_span_matches_heldout"]["n_heldout_with_any_match"],
        "circuit_L0_pair_mean_ablate": led.by["0a.L0.abl.pair"]["recomputed"],
        "circuit_L0_pair_mean_steer": led.by["0a.L0.ste.pair"]["recomputed"],
        "circuit_natural_heldout_ablate": led.by["0a.nat.abl.heldout"]["recomputed"],
        "circuit_natural_heldout_p": led.by["0a.nat.abl.p_heldout"]["recomputed"],
        "das_T10_bfs_written": led.by["0b.bfs.written"]["recomputed"],
        "das_T10_bfs_correct": led.by["0b.bfs.correct"]["recomputed"],
        "das_T10_bfs_mcnemar_p": led.by["0b.bfs.mc_p"]["recomputed"],
        "das_n_29of30_occurrences": len(ed["occ"]),
        "entropy_n_twin_samples_above_q90": led.by["0c.n_total"]["recomputed"],
        "law_units_pass": led.by["0c.law.n_pass"]["recomputed"], "law_sign_test_p_vs_flat": led.by["0c.law.sign_p"]["recomputed"],
        "positioning_rows": len(ed["positioning"]["rows"]),
        "positioning_live_verified": ed["positioning"]["status_counts"].get("VERIFIED", 0),
        "positioning_live_verified_abstract": ed["positioning"]["status_counts"].get("VERIFIED-ABSTRACT", 0),
        "positioning_live_not_found": ed["positioning"]["status_counts"].get("NOT-FOUND", 0),
        "n_venue_or_version_corrections": ed["positioning"]["n_venue_or_version_corrections"],
        "alias_gate_E1": int(z["gates"]["E1_labels_equal"]), "alias_gate_E2": int(z["gates"]["E2_multiset_equal_forced_set"]),
        "alias_gate_E3": int(z["gates"]["E3_g_matches_M1"]),
        **({"alias_gate_E4_spearman": z["gates"]["E4_spearman_vs_stored_k15"]}
           if z["gates"].get("E4_spearman_vs_stored_k15") is not None else {}),
        "alias_rho_k15": k15["rho_primary"], "alias_p_k15": k15["primary_p_lower"],
        "alias_rho_k15_ci_low": k15["ci95_primary"][0], "alias_rho_k15_ci_high": k15["ci95_primary"][1],
        "alias_rho_k10": k10["rho_primary"], "alias_p_k10": k10["primary_p_lower"],
        "alias_rho_k15_secondary": k15["rho_secondary"], "alias_p_k15_secondary": k15["secondary_p_lower"],
        "alias_rho_5centroid": c5["rho"], "alias_p_5centroid": c5["p_lower"], "alias_5centroid_floor": c5["floor_lower"],
        "alias_stratified_stat": st["stat"], "alias_stratified_p": st["p_lower"], "alias_fe_beta": st["beta"],
        "alias_fe_beta_p_two_sided": st["beta_p_two_sided"],
        "alias_placebo_P1_reject_rate": z["placebos"]["P1_shuffled_alias_labels"]["share_p_lt_0.05"],
        "alias_placebo_P2_reject_rate": z["placebos"]["P2_random_overlap_vector"]["share_p_lt_0.05"],
        "alias_placebo_P3_mean": z["placebos"]["P3_within_algorithm_shuffles"]["stat_mean"],
        "alias_planted_control_pass": int(z["placebos"]["C1_planted"]["seed0_succeeds"]),
        **({"alias_bpe_rho_k15": led.by["0e.k15.bpe_rho"]["recomputed"],
            "alias_bpe_stratified_k15": led.by["0e.k15.bpe_strat"]["recomputed"],
            "alias_bpe_stratified_p_k15": led.by["0e.k15.bpe_strat_p"]["recomputed"],
            "alias_bpe_stratified_p_k10": led.by["0e.k10.bpe_strat_p"]["recomputed"]} if "0e.k15.bpe_rho" in led.by else {}),
        "alias_verdict_confounded": int(z["verdict"]["code"] == "CONFOUNDED"),
        "gemma_families_twin_fallback": sum(1 for v in rec["0f"]["family_status_G"].values() if v["status"] != "run arm"),
        "figure_markers_added": sum(1 for m in ed["markers"] if m["id"] != "fig_alias_overlap"),
        "figure_caption_fields_complete": sum(1 for m in ed["markers"] if m["n_fields"] == 6),
    }
    if audit:
        metrics.update({"audit_n_checks": audit["n_checks"], "audit_n_mismatches": audit["n_mismatches"],
                        "audit_placebos_failed": audit["placebos_failed"]})
        if "headline_all_match" in audit:
            metrics.update({"audit_raw_headlines_all_match": int(audit["headline_all_match"]),
                            "audit_raw_headline_placebos_failed": audit["headline_placebos_failed"]})
    metrics = {k: (float(v) if isinstance(v, float) else int(v)) for k, v in metrics.items()}
    ds = []
    ds.append({"dataset": "corrections", "examples": [
        ex(f"{r['name']} <- {r['source']}", str(r["printed"]), item=r["item"], status=r["status"], recorded=r["recorded"],
           recomputed=r["recomputed"] if not isinstance(r["recomputed"], float) else round(r["recomputed"], 12),
           tolerance=r["tolerance"], predict_recorded=str(r["recorded"]), predict_recomputed=str(r["printed"]),
           eval_match=1.0 if r["status"] == "MATCH" else 0.0) for r in led.rows]})
    ds.append({"dataset": "report_edits", "examples": [
        ex(e["old_text"], e["new_text"], id=e["id"], item=e["item"], section=e["section"], rationale=e["rationale"],
           line_hint=ed["apply"].get("line_of_edit", {}).get(e["id"]), values=e["values"],
           eval_n_values=len(e["values"]), eval_old_text_unique=float(ed["corrected"].count(e["old_text"]) == 0 or
                                                                       e["old_text"] in e["new_text"]))
        for e in ed["edits"]]})
    ds.append({"dataset": "circuit_heldout", "examples": [
        ex(f"{r['contrast']} {r['mode']} {r['node']}", f"held-out mean {r['mean_d_heldout']:.4f}",
           label=r["label"], n=r["n"], n_heldout=r["n_heldout"], eval_mean_d=r["mean_d"],
           eval_mean_d_pair_prompts=r["mean_d_pair_prompts"], eval_mean_d_heldout=r["mean_d_heldout"],
           eval_p_sign_flip_heldout=r["p_sign_flip_heldout"], eval_p_holm_20=r["p_holm_20"],
           eval_identity_pass=float(r["identity_pass"]), eval_n_exact_zero_heldout=r["n_exact_zero_heldout_entries"])
        for r in a0["table_rows"]]})
    ds.append({"dataset": "occurrences_29of30", "examples": [
        ex(o["context"], o["class"], line=o["line"], when="before", eval_das_fork=float(o["class"].startswith("DAS-fork")))
        for o in ed["occ"]] + [
        ex(o["context"], o["class"], line=o["line"], when="after", eval_das_fork=float(o["class"].startswith("DAS-fork")))
        for o in ed["occ_after"]]})
    ds.append({"dataset": "entropy_scope", "examples": [
        ex(f"{r['unit']} ({r['task']})", f"{r['argmax_entropy_hits_first64']} of {r['n_twin_reaching_fork']}",
           eval_share_above_q90=r["share_above_q90"], eval_q90_nats=r["q90_nats"],
           eval_argmax_hits=r["argmax_entropy_hits_first64"], eval_n=r["n_twin_reaching_fork"],
           eval_median_fork_entropy=r["median_fork_entropy_nats"]) for r in rec["0c"]["entropy_rows"]] + [
        ex(f"law {r['unit']}", f"law {r['law']:.3f}", eval_law_mae=r["law"], eval_flat_mae=r["flat_B0"],
           eval_tempering_mae=r["tempering_B1"], eval_H_bits=r["H_bits"], eval_pass=float(r["pass"]))
        for r in rec["0c"]["law_rows"]]})
    ds.append({"dataset": "positioning_rows", "examples": [
        ex(f"{r['claim']} | {r['neighbour']}", r["verdict"] or "n/a", status=r["status"], venue=r["venue_track"],
           url=r["url"], quote=r["quote"], adds=r["adds"], source=r["source"],
           eval_live_verified=float(r["status"] == "VERIFIED")) for r in ed["positioning"]["rows"]]})
    order = [(a, j) for a in AG.NONBANNED for j in range(3)]
    rows = z["overlap_rows"]
    ds.append({"dataset": "alias_overlap", "examples": [
        ex(f"{a} alias {j}: {r['alias']}", f"g_k15 {k15['g_alias'][f'{a}|{j}']:.3f}", detectors=r["detectors"],
           eval_primary_overlap=r["primary"], eval_secondary_overlap=r["secondary"],
           eval_g_k15=k15["g_alias"][f"{a}|{j}"], eval_g_k10=k10["g_alias"][f"{a}|{j}"]) for (a, j), r in zip(order, rows)]
        + [ex(f"{a} (algorithm centroid)", f"g_k15 {k15['g_algorithm'][a]:.3f}", eval_g_k15=k15["g_algorithm"][a],
              eval_g_k10=k10["g_algorithm"][a]) for a in AG.NONBANNED]})
    gam = []
    with (TABLES / "gemma_arms.csv").open() as f:
        for r in csv.DictReader(f):
            gam.append(ex(f"{r['unit']} {r['arm']} ({r['arm_name']})", "present" if r["present"] == "True" else "not run",
                          eval_present=float(r["present"] == "True"),
                          eval_pass_cost=float(r["pass_cost"]) if r["pass_cost"] not in ("", "None") else float("nan")))
    ds.append({"dataset": "gemma_arms", "examples": gam})
    ds.append({"dataset": "figure_markers", "examples": [
        ex(m["id"], m["caption"], art=m["art"], files=m["files"], sha256=m["sha256"], eval_n_fields=m["n_fields"])
        for m in ed["markers"]]})
    lh = []
    for when, L in (("before", ed["lint_before"]), ("after", ed["lint_after"])):
        for h in L.get("unallowed", []) + L.get("allowlisted", []):
            lh.append(ex(h["snippet"], h["pattern"], when=when, line=h["line"], section=h["section"],
                         allow_reason=h.get("allow_reason"), eval_unallowed=float(h.get("allow_reason") is None)))
    if not lh:
        lh.append(ex("no hits", "none", when="after"))
    ds.append({"dataset": "lint_hits", "examples": lh})
    sections = {}
    for it in CHECKLIST:
        sections[it] = {"edits": corr["item_checklist"][it]["edit_ids"],
                        "values": [r["name"] for r in led.rows if r["item"] == it],
                        "verdict": corr["item_checklist"][it]["status"]}
    sections["0a"]["metrics"] = {k: a0[k] for k in ("identity_check_pass", "n_exact_zero_heldout_nodes",
                                                     "cond_span_matches_heldout", "replication", "cause_evidence",
                                                     "code_quotes", "C2a_top10_has_COND", "bands", "zero_heldout")}
    sections["0b"]["metrics"] = rec["0b"] | {"occurrences_by_class_before": dict(Counter(o["class"] for o in ed["occ"]))}
    sections["0c"]["metrics"] = {"entropy_rows": rec["0c"]["entropy_rows"], "law_rows": rec["0c"]["law_rows"]}
    sections["0d"]["metrics"] = {"status_counts": ed["positioning"]["status_counts"],
                                 "n_venue_or_version_corrections": ed["positioning"]["n_venue_or_version_corrections"],
                                 "live": ed["positioning"]["live"], "n_carried_rows": ed["positioning"]["carried_n"]}
    sections["0e"]["metrics"] = {k: z[k] for k in ("gates", "verdict", "placebos", "null_sizes", "flags")} | {
        "per_k": {k: {kk: vv for kk, vv in v.items() if kk not in ("g_alias",)} for k, v in z["per_k"].items()}}
    sections["0e"]["verdict"] = z["verdict"]["sentence"]
    sections["0f"]["metrics"] = rec["0f"]
    sections["0g"]["metrics"] = {"markers": [{k: m[k] for k in ("id", "art", "files", "n_fields", "sha256")}
                                             for m in ed["markers"]]}
    flags = []
    if vb["CORRECTED"]:
        flags.append("direction_number_not_reproduced")
    meta = {"evaluation_name": "Iteration-9 Step 0: repair of the iteration-8 research record",
            "plan_id": "gen_plan_evaluation_1_idx1", "utc": dt.datetime.now(dt.timezone.utc).isoformat(),
            "report_sha256": corr["report_sha256"], "sections": sections, "flags": flags,
            "outputs": ["corrections.json", "report_corrected.md", "report.diff", "apply_test.json", "lint_allowlist.json",
                        "tables/*.csv", "figures/alias_overlap.(pdf|png)", "figure_markers.json",
                        "predictions/commit_s0e.json", "commitments.log", "audit/rederive.py"]}
    out = {"metadata": meta, "metrics_agg": metrics, "datasets": ds}
    json.dumps(out, allow_nan=False, default=str)        # raises if any NaN or infinity slipped in
    (ROOT / "eval_out.json").write_text(json.dumps(out, indent=1, default=str))
    (ROOT / "full_eval_out.json").write_text(json.dumps(out, indent=1, default=str))

    def trunc(x: Any) -> Any:
        if isinstance(x, str):
            return x[:200]
        if isinstance(x, list):
            return [trunc(v) for v in x]
        if isinstance(x, dict):
            return {k: trunc(v) for k, v in x.items()}
        return x

    mini = {"metadata": {k: v for k, v in meta.items() if k != "sections"} | {"sections_note": "see full_eval_out.json"},
            "metrics_agg": metrics, "datasets": [{"dataset": d["dataset"], "examples": d["examples"][:3]} for d in ds]}
    (ROOT / "mini_eval_out.json").write_text(json.dumps(mini, indent=1, default=str))
    (ROOT / "preview_eval_out.json").write_text(json.dumps(trunc(mini), indent=1, default=str))
    return out


## `main()`

The stages run in this order:
1. setup;
2. item 0e;
3. the BPE sensitivity;
4. the ledger (0a, 0b, 0c, 0f);
5. the edits, apply test and lint;
6. `corrections.json`;
7. the two audits, as subprocesses;
8. `results/s0_core.json`;
9. `eval_out.json`;
10. the check that no input changed.

Two changes are marked `# demo:`: `--stages` comes from `STAGES`, and the final check hashes only the shipped inputs.

In [ ]:
@logger.catch(reraise=True)
def main() -> None:
    args = argparse.Namespace(stages=STAGES)  # demo: the --stages argument comes from the config cell
    resource.setrlimit(resource.RLIMIT_AS, (24 * 1024 ** 3, 24 * 1024 ** 3))
    for d in (RESULTS, TABLES, ROOT / "figures"):
        d.mkdir(exist_ok=True)
    t0 = time.time()
    man = stage_setup()
    if args.stages in ("all", "0e") or not (RESULTS / "s0e.json").exists():
        z = stage_0e()
    else:
        z = json.loads((RESULTS / "s0e.json").read_text())
    if args.stages == "0e":
        return
    import os
    os.environ.setdefault("HF_HUB_OFFLINE", "1")
    tz = tertiary_bpe.main()
    led = RC.Ledger()
    rec = stage_ledger(led)
    ed = stage_edits(led, rec, z, tz)
    corr = write_corrections(led, ed, man)
    import subprocess
    pr = subprocess.run([sys.executable, "-I", str(ROOT / "audit/rederive.py")], capture_output=True, text=True,
                        timeout=1800)
    logger.info(f"audit/rederive.py exit {pr.returncode}: {pr.stdout.strip()[:300]}")
    if pr.returncode != 0:
        logger.error(f"audit failed: {pr.stderr[-2000:]}")
    audit = json.loads((ROOT / "audit/rederive_result.json").read_text()) if (ROOT / "audit/rederive_result.json").exists() else None
    ph = subprocess.run([sys.executable, "-I", str(ROOT / "audit/headline_rederive.py")], capture_output=True, text=True,
                        timeout=1800)
    logger.info(f"audit/headline_rederive.py exit {ph.returncode}")
    hl = json.loads((ROOT / "audit/headline_rederive.json").read_text()) if ph.returncode == 0 else None
    if audit is not None and hl is not None:
        audit = audit | {"headline_all_match": hl["all_headlines_match"], "headline_placebos_failed": hl["placebos_failed_as_required"]}
    (RESULTS / "s0_core.json").write_text(json.dumps({"0a": {k: v for k, v in rec["0a"].items() if k != "table_rows"},
                                                      "0b": rec["0b"], "0c": rec["0c"], "0f": rec["0f"]}, indent=1,
                                                     default=str))
    write_eval_out(led, rec, z, ed, corr, audit)
    after = {k: sha(p) for k, p in input_files() if p.exists()}  # demo: only the shipped inputs
    changed = [k for k, v in after.items() if v != man[k]["sha256"]]
    assert not changed, f"inputs changed during the run: {changed}"
    logger.info(f"done in {time.time() - t0:.0f} s; edits {corr['n_edits']}, values {corr['values_by_status']}, "
                f"lint {corr['apply']['lint_before']} -> {corr['apply']['lint_after']}; inputs unchanged")


In [ ]:
t_run = time.time()
main()
print(f"eval.py main() finished in {time.time() - t_run:.0f} s")


## Results: this run against the full run

The run wrote the same files as the original, into `eval_demo_run/art_iYh9zVP7myjJ/`:
- `eval_out.json` and its full, mini and preview variants;
- `corrections.json`, `report_corrected.md`, `report.diff`, `apply_test.json`, `lint_allowlist.json` and
  `figure_markers.json`;
- `results/s0e.json`, `results/s0e_tertiary.json` and `results/s0_core.json`;
- `tables/*.csv`;
- `figures/alias_overlap.(pdf|png)`;
- the two audit results, `audit/*.json`;
- `logs/eval.log`.

The next cells compare them with the full run's numbers, which `mini_demo_data.json` carries in `metadata.full_run`.
Every difference should have a stated reason: gate E4, or resampling sizes below the original.

In [ ]:
REF = data["metadata"]["full_run"]
out = json.loads((ROOT / "eval_out.json").read_text())
corr = json.loads((ROOT / "corrections.json").read_text())
M = out["metrics_agg"]
RESAMPLED = ("ci_low", "ci_high", "placebo", "secondary")      # metrics that depend on N_BOOT / N_PLACEBO / N_MC_SECONDARY
WHY = {"alias_gate_E4_spearman": "gate E4 needs the 181 MB states file (not shipped)",
       "values_total": "ledger value 0e.E4 is not created without gate E4",
       "values_new": "ledger value 0e.E4 is not created without gate E4",
       "audit_n_checks": "the audit's 0e.E4 consistency check is skipped with it"}
n_same = 0
print(f"{'metric':40s} {'full run':>12s} {'this run':>12s}  status")
for k, v in REF["metrics_agg"].items():
    g = M.get(k)
    same = g is not None and abs(float(g) - float(v)) <= 1e-9 * max(1.0, abs(float(v)))
    n_same += same
    why = WHY.get(k) or ("resampling sizes below the original" if any(s in k for s in RESAMPLED) else "UNEXPECTED")
    print(f"{k:40s} {v:>12.6g} {'-' if g is None else format(g, '.6g'):>12s}  {'same' if same else 'differs: ' + why}")
print(f"\n{n_same} of {len(REF['metrics_agg'])} metrics equal the full run's")


In [ ]:
# the value ledger and the 49 edits, against the full run
led = {v["name"]: v for v in corr["values"]}
same = [n for n, v in REF["ledger"].items() if n in led and led[n]["printed"] == v["printed"] and led[n]["status"] == v["status"]]
print(f"value ledger: {len(led)} values here, {len(REF['ledger'])} in the full run; {len(same)} identical (printed value and status)")
print("  missing here:", [n for n in REF["ledger"] if n not in led])
print("  different:   ", [(n, REF["ledger"][n]["printed"], led[n]["printed"]) for n in REF["ledger"] if n in led and n not in same])
print("  status counts here:", dict(Counter(v["status"] for v in led.values())), "| full run:", REF["values_by_status"])

mask = lambda t: re.sub(r"sha256 (?:[0-9a-f]{12}|missing)", "sha256 <hash>", t)  # figure-file hashes masked
eds = {e["id"]: e for e in corr["edits"]}
e_same = [i for i, r in REF["edits"].items() if i in eds and hashlib.sha256(mask(eds[i]["new_text"]).encode()).hexdigest()
          == r["new_text_sha256_hashes_masked"]]
print(f"\nedits: {len(eds)} here, {len(REF['edits'])} in the full run; {len(e_same)} identical once figure-file hashes are masked")
print("  different:", [i for i in REF["edits"] if i not in e_same])
for i in [i for i in REF["edits"] if i not in e_same and i in REF["edits_new_text_citing_E4"]]:
    a, b = mask(REF["edits_new_text_citing_E4"][i]).split(" "), mask(eds[i]["new_text"]).split(" ")
    for tag, i1, i2, j1, j2 in difflib.SequenceMatcher(a=a, b=b, autojunk=False).get_opcodes():
        if tag != "equal":   # word-level differences of an edit that cites gate E4
            print(f"    {i} full run: {' '.join(a[i1:i2])!r}\n    {i} this run: {' '.join(b[j1:j2])!r}")
ap = corr["apply"]
print(f"apply test pass: {ap['apply_test_pass']} ({ap['n_edits']} edits, problems {ap['problems']}); "
      f"lint: {ap['lint_before']} live hits before ({ap['lint_before_total_hits']} with the allowlisted), {ap['lint_after']} after")
for nm in ("rederive_result.json", "headline_rederive.json"):
    a = json.loads((ROOT / "audit" / nm).read_text())
    keys = ("n_checks", "n_mismatches", "placebos_failed", "placebos_total") if "n_checks" in a else (
        "all_headlines_match", "placebos_failed_as_required")
    print(f"audit/{nm}:", {k: a[k] for k in keys})
print("full-run audits:", REF["audit_rederive"], REF["audit_headline"])


### One edit, as the run wrote it

This is the first edit of item 0b, exactly as `edits_spec.build` wrote it, with the ledger values it cites. In section
8.4, iteration 8 called the rank-4 DAS edit at the fork "a strong steering handle" in both model families. Under the
committed generation rule, it is a sufficient handle in Gemma but NO HANDLE at Qwen3-8B's fork. There, the edit toward
BFS writes BFS in 29 of 30 programs, but only 5 of them pass the hidden tests.

In [ ]:
e = next(e for e in corr["edits"] if e["item"] == "0b")
print(f"{e['id']} (section {e['section']}): {e['rationale']}\n")
print("OLD:", e["old_text"][:900], "\n")
print("NEW:", e["new_text"][:1400])
print("\ncites:", [f"{v['name']} = {v['printed']} ({v['status']})" for v in e["values"]][:10])


### Item 0e: the figure written by `stage_0e._figure`

This is the original figure, rebuilt from the stored activation geometry.
- **Panels A and B** plot each alias set's g, its distance to the banned five, against its primary lexical overlap with
  the banned names, at k = 15 and k = 10. Circle, square and triangle mark aliases 0, 1 and 2. Stars are the five
  algorithm centroids, labelled with iteration 8's edge or opposite class.
- **Panel C** is a classical MDS of the stored k = 15 geodesics between all 300 forced prompts at block 31 of
  Qwen3-32B: the actual activation-space manifold. The banned five are grey, and each outline encloses one alias set
  of 10 prompts.

In [ ]:
display(Image(filename=str(ROOT / "figures/alias_overlap.png")))
z = json.loads((ROOT / "results/s0e.json").read_text())
print("gates:", {k: v for k, v in z["gates"].items() if k != "E3_detail"})
print("flags:", z["flags"])
print(f"\nverdict {z['verdict']['code']}: {z['verdict']['sentence']}\n")
print(f"{'algorithm':14s} {'alias':52s} {'primary':>7s} {'secondary':>9s} {'g k15':>7s} {'g k10':>7s}")
for r in z["overlap_rows"]:
    key = f"{r['algorithm']}|{r['alias_index']}"
    print(f"{r['algorithm']:14s} {r['alias'][:52]:52s} {r['primary']:7d} {r['secondary']:9.3f} "
          f"{z['per_k']['k15']['g_alias'][key]:7.2f} {z['per_k']['k10']['g_alias'][key]:7.2f}")


### Summary figure

- **(a)** The exact null of the 15-alias Spearman: every one of the 360,360 distinct arrangements of the primary overlap
  vector, with the observed correlation at k = 15 and k = 10.
- **(b)** The within-algorithm view. Each line joins the three aliases of one algorithm, plotting g at k = 15 against
  content-word overlap. Across algorithms, g falls as overlap rises. Within an algorithm, where the mechanism is held
  fixed, there is no consistent direction. The Spearman correlation is −0.87 for bidirectional A*, +0.5 for IDA* and
  0 for bidirectional BFS, with mean −0.12 and exact p 0.46. All three aliases of unit-weight Dijkstra and of jump point
  search have zero overlap. Their vertical spread shows how much g varies between aliases when no name overlaps at all.
- **(c)** The whole-report lint by rule, before and after the 49 edits, live and allowlisted hits.
- **(d)** The value ledger by review item and status.

In [ ]:
%matplotlib inline
import stage_0e as S0E  # the module eval.py imported (already loaded)

k15, k10 = z["per_k"]["k15"], z["per_k"]["k10"]
x1 = np.array([r["primary"] for r in z["overlap_rows"]], dtype=float)
null1 = S0E.exact_null(x1)
fig, ax = plt.subplots(2, 2, figsize=(14, 10))

a = ax[0, 0]
vals, cnt = np.unique(np.round(null1, 6), return_counts=True)
a.bar(vals, cnt / cnt.sum(), width=0.012, color="#999999")
for kk, d, c in (("k15", k15, "#D55E00"), ("k10", k10, "#0072B2")):
    a.axvline(d["rho_primary"], color=c, lw=2, label=f"{kk}: rho {d['rho_primary']:.2f}, exact p {d['primary_p_lower']:.4f}")
a.set_xlabel("Spearman(g, primary overlap) over the 15 alias sets")
a.set_ylabel("share of the 360,360 arrangements")
a.set_title("(a) exact permutation null of the committed statistic")
a.legend(fontsize=8)

a = ax[0, 1]
col = {"bibfs": "#0072B2", "biastar": "#56B4E9", "idastar": "#009E73", "jps": "#E69F00", "dijkstra_unit": "#D55E00"}
for alg in AG.NONBANNED:
    rows_a = [r for r in z["overlap_rows"] if r["algorithm"] == alg]
    xs = [r["secondary"] for r in rows_a]
    gs = [k15["g_alias"][f"{alg}|{r['alias_index']}"] for r in rows_a]
    o = np.argsort(xs)
    a.plot(np.array(xs)[o], np.array(gs)[o], "-o", color=col[alg], label=AG.DISPLAY[alg])
st = k15["stratified_secondary"]
a.set_xlabel("content-word overlap with the banned aliases (Jaccard)")
a.set_ylabel("g at k = 15 (min over banned of mean geodesic)")
a.set_title(f"(b) within-algorithm: mean Spearman {st['stat']:.2f}, exact p {st['p_lower']:.2f} "
            f"({st['n_arrangements_effective']} relabelings)")
a.legend(fontsize=8)

a = ax[1, 0]
lint = list(csv.DictReader((ROOT / "tables/lint_hits.csv").open()))
pats = sorted({h["pattern"] for h in lint}, key=lambda p: int(p[1:]))
xs = np.arange(len(pats))
for i, (when, kind, c) in enumerate((("before", "live", "#D55E00"), ("before", "allowlisted", "#E69F00"),
                                     ("after", "live", "#0072B2"), ("after", "allowlisted", "#56B4E9"))):
    n = [sum(1 for h in lint if h["pattern"] == p and h["when"] == when and bool(h["allow_reason"]) == (kind == "allowlisted"))
         for p in pats]
    a.bar(xs + (i - 1.5) * 0.2, n, width=0.2, color=c, label=f"{when} the edits, {kind}")
a.set_xticks(xs, pats)
a.set_ylabel("hits in the whole report")
a.set_title(f"(c) residual-claim lint: {ap['lint_before']} live hits before, {ap['lint_after']} after")
a.legend(fontsize=8)

a = ax[1, 1]
items = list(CHECKLIST)
bottom = np.zeros(len(items))
for s, c in (("MATCH", "#009E73"), ("NEW", "#56B4E9"), ("CORRECTED", "#D55E00"), ("CARRIED", "#999999")):
    n = np.array([sum(1 for v in corr["values"] if v["item"] == it and v["status"] == s) for it in items])
    a.bar(items, n, bottom=bottom, color=c, label=s)
    bottom += n
a.set_ylabel("ledger values")
a.set_title(f"(d) value ledger: {len(corr['values'])} values by review item and status")
a.legend(fontsize=8)
fig.tight_layout()
plt.show()
